## Conclusion — Entraînement adversarial et robustesse finale

L'évaluation finale met en évidence une amélioration globale de la robustesse face à l'attaque HopSkipJump après entraînement adversarial. Sur les 53 fraudes communes de l'ensemble de test, le taux de succès local de HSJ diminue pour les cinq classificateurs. La réduction la plus importante est observée pour Random Forest, dont l'ASR passe de 30,19 % à 5,66 %, suivie de LightGBM, qui passe de 33,96 % à 15,09 %. XGBoost passe de 35,85 % à 20,75 %, tandis que le SVM linéaire passe de 28,30 % à 16,98 %. L'amélioration est plus limitée pour la régression logistique, avec une diminution de 22,64 % à 20,75 %. La détection propre des 53 fraudes est néanmoins conservée à 100 % pour l'ensemble des modèles étudiés.

L'analyse face à ZOO révèle cependant une robustesse plus dépendante du modèle et du critère considéré. Pour la régression logistique V2, l'ASR brut diminue de 62,26 % à 49,06 %, ce qui traduit une amélioration de la robustesse. Pour le SVM V2, l'ASR brut atteint en revanche 100 %, contre 81,13 % avant défense. L'entraînement adversarial ne confère donc pas une robustesse uniforme face à toutes les stratégies d'attaque. Toutefois, dans l'analyse locale a posteriori, aucune attaque ZOO réussie après défense ne présente une norme L∞ inférieure ou égale à 0,05 pour les deux modèles linéaires. Cette dernière observation doit être interprétée avec prudence, car ZOO n'a pas été exécuté sous une contrainte explicite ε = 0,05.

Ces résultats montrent que l'entraînement adversarial améliore substantiellement la robustesse locale de plusieurs modèles, particulièrement Random Forest et LightGBM, tout en mettant en évidence les limites d'une défense universelle. La sélection du modèle final ne doit donc pas dépendre d'une seule métrique, mais d'un compromis entre performance sur données propres, capacité de détection des fraudes et résistance aux différentes attaques adversariales. Cette sélection multicritère constitue l'objectif du Notebook 12.

In [26]:
# ============================================================
# ÉTAPE 16G — CLÔTURE OFFICIELLE DU NOTEBOOK 11
# EXPORTS + AUDIT FINAL
# AUCUNE ATTAQUE / AUCUN ENTRAÎNEMENT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

FINAL_SELECTION_DIR = Path(FINAL_SELECTION_DIR)

# ------------------------------------------------------------
# 1. Vérifications fondamentales
# ------------------------------------------------------------

EXPECTED_MODELS = {
    "Régression logistique",
    "SVM linéaire",
    "Random Forest",
    "XGBoost",
    "LightGBM"
}

assert len(final_notebook11_summary) == 5

assert set(
    final_notebook11_summary["Model"]
) == EXPECTED_MODELS

assert final_notebook11_summary[
    "HSJ_Clean_Retention_%"
].eq(100).all()

print("✅ 5 modèles présents")
print("✅ Clean retention HSJ = 100 % pour les 5 modèles")


# ------------------------------------------------------------
# 2. Tableau officiel des versions retenues
# ------------------------------------------------------------

final_versions = pd.DataFrame({

    "Model": [
        "Régression logistique",
        "SVM linéaire",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],

    "Selected_Version": [
        "V2",
        "V2",
        "V1",
        "V1",
        "V1"
    ],

    "Reason": [
        "V2 retenue après validation HSJ + ZOO",
        "V2 retenue après validation HSJ + ZOO",
        "V1 retenue après amélioration HSJ",
        "V1 retenue après amélioration HSJ",
        "V1 retenue après amélioration HSJ"
    ]
})

final_versions.to_csv(
    FINAL_SELECTION_DIR /
    "final_defense_versions.csv",
    index=False
)


# ------------------------------------------------------------
# 3. Sauvegarde définitive du grand tableau
# ------------------------------------------------------------

FINAL_SUMMARY_FILE = (
    FINAL_SELECTION_DIR /
    "final_notebook11_summary.csv"
)

final_notebook11_summary.to_csv(
    FINAL_SUMMARY_FILE,
    index=False
)


# ------------------------------------------------------------
# 4. Tableau synthétique pour Notebook 12
# ------------------------------------------------------------

notebook12_input = final_notebook11_summary[
    [
        "Model",
        "Selected_Version",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "HSJ_Local_ASR_%",
        "HSJ_Gain_points",
        "After_ZOO_Raw_ASR_%",
        "ZOO_Raw_Gain_points",
        "After_ZOO_Local_005_ASR_%"
    ]
].copy()

notebook12_input.to_csv(
    FINAL_SELECTION_DIR /
    "notebook12_input_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Note méthodologique importante
# ------------------------------------------------------------

methodological_note = """
NOTEBOOK 11 — NOTE MÉTHODOLOGIQUE FINALE

1. Les modèles sélectionnés avant l'évaluation finale sont :
   - Régression logistique : V2
   - SVM linéaire : V2
   - Random Forest : V1
   - XGBoost : V1
   - LightGBM : V1

2. Le TEST n'a pas été utilisé pour entraîner ou sélectionner
   les versions défendues.

3. HSJ :
   - évaluation locale principale sur 53 fraudes communes ;
   - contrainte Linf <= 0.05 ;
   - Time et Amount conservées ;
   - les cinq modèles sont évalués.

4. ZOO :
   - évaluation finale ciblée sur LR V2 et SVM V2 ;
   - l'ASR brut correspond au succès sans contrainte epsilon ;
   - ZOO_Local_005 correspond uniquement à une analyse
     post-hoc des attaques générées ;
   - il ne faut PAS présenter cette mesure comme une attaque
     ZOO explicitement contrainte par epsilon = 0.05.

5. Aucun modèle final unique n'est choisi dans le Notebook 11.
   Cette décision est réservée au Notebook 12.
"""

(
    FINAL_SELECTION_DIR /
    "notebook11_methodological_note.txt"
).write_text(
    methodological_note,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. Audit des fichiers essentiels
# ------------------------------------------------------------

expected_files = [

    FINAL_SELECTION_DIR /
    "final_notebook11_summary.csv",

    FINAL_SELECTION_DIR /
    "final_defense_versions.csv",

    FINAL_SELECTION_DIR /
    "notebook12_input_summary.csv",

    FINAL_SELECTION_DIR /
    "selected_models_hsj_test_summary.csv",

    FINAL_SELECTION_DIR /
    "selected_models_zoo_test_summary.csv",

    FINAL_SELECTION_DIR /
    "notebook11_methodological_note.txt"
]

print("\n" + "=" * 85)
print("AUDIT FINAL — NOTEBOOK 11")
print("=" * 85)

all_ok = True

for path in expected_files:

    exists = path.exists()

    print(
        "✅" if exists else "❌",
        path.name
    )

    if not exists:
        all_ok = False


if not all_ok:

    raise RuntimeError(
        "Un export final manque."
    )


print("\n" + "=" * 85)
print("✅ NOTEBOOK 11 TERMINÉ")
print("✅ Modèles défendus figés")
print("✅ Test clean terminé")
print("✅ HSJ final terminé : 265 / 265")
print("✅ ZOO final terminé : 106 / 106")
print("✅ Exports Notebook 12 prêts")
print("=" * 85)

✅ 5 modèles présents
✅ Clean retention HSJ = 100 % pour les 5 modèles

AUDIT FINAL — NOTEBOOK 11
✅ final_notebook11_summary.csv
✅ final_defense_versions.csv
✅ notebook12_input_summary.csv
✅ selected_models_hsj_test_summary.csv
✅ selected_models_zoo_test_summary.csv
✅ notebook11_methodological_note.txt

✅ NOTEBOOK 11 TERMINÉ
✅ Modèles défendus figés
✅ Test clean terminé
✅ HSJ final terminé : 265 / 265
✅ ZOO final terminé : 106 / 106
✅ Exports Notebook 12 prêts


In [25]:
# ============================================================
# ÉTAPE 16F — GRAND TABLEAU FINAL NOTEBOOK 11
# CLEAN TEST + HSJ + ZOO
# AUCUNE NOUVELLE ATTAQUE
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Performances CLEAN finales sur TEST
# ------------------------------------------------------------

clean_final = pd.DataFrame({

    "Model": [
        "Régression logistique",
        "SVM linéaire",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],

    "Selected_Version": [
        "V2",
        "V2",
        "V1",
        "V1",
        "V1"
    ],

    "Accuracy": [
        0.9980,
        0.9989,
        0.9995,
        0.9993,
        0.9995
    ],

    "Precision": [
        0.4436,
        0.6477,
        0.8889,
        0.8116,
        0.8871
    ],

    "Recall": [
        0.8310,
        0.8028,
        0.7887,
        0.7887,
        0.7746
    ],

    "F1": [
        0.5784,
        0.7170,
        0.8358,
        0.8000,
        0.8271
    ],

    "ROC_AUC": [
        0.9457,
        0.9534,
        0.9685,
        0.9758,
        0.9632
    ]
})


# ------------------------------------------------------------
# 2. Résultats HSJ
# ------------------------------------------------------------

hsj_final = hsj_before_after[
    [
        "Model",
        "Before_HSJ_Local_ASR_%",
        "HSJ_Local_ASR_%",
        "Robustness_Gain_points",
        "Relative_ASR_Reduction_%",
        "Clean_Retention_%"
    ]
].copy()

hsj_final = hsj_final.rename(
    columns={
        "Robustness_Gain_points":
            "HSJ_Gain_points",

        "Relative_ASR_Reduction_%":
            "HSJ_Relative_Reduction_%",

        "Clean_Retention_%":
            "HSJ_Clean_Retention_%"
    }
)


# ------------------------------------------------------------
# 3. Résultats ZOO
# ------------------------------------------------------------

zoo_final = pd.DataFrame({

    "Model": [
        "Régression logistique",
        "SVM linéaire"
    ],

    "Before_ZOO_Raw_ASR_%": [
        62.26,
        81.13
    ],

    "After_ZOO_Raw_ASR_%": [
        49.06,
        100.00
    ],

    "ZOO_Raw_Gain_points": [
        13.20,
        -18.87
    ],

    "After_ZOO_Local_005_ASR_%": [
        0.00,
        0.00
    ],

    "ZOO_Clean_Retention_%": [
        100.00,
        100.00
    ]
})


# ------------------------------------------------------------
# 4. Fusion générale
# ------------------------------------------------------------

final_notebook11_summary = (
    clean_final
    .merge(
        hsj_final,
        on="Model",
        how="left"
    )
    .merge(
        zoo_final,
        on="Model",
        how="left"
    )
)


# ------------------------------------------------------------
# 5. Ordre final
# ------------------------------------------------------------

final_notebook11_summary = final_notebook11_summary[
    [
        "Model",
        "Selected_Version",

        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",

        "Before_HSJ_Local_ASR_%",
        "HSJ_Local_ASR_%",
        "HSJ_Gain_points",
        "HSJ_Relative_Reduction_%",

        "Before_ZOO_Raw_ASR_%",
        "After_ZOO_Raw_ASR_%",
        "ZOO_Raw_Gain_points",
        "After_ZOO_Local_005_ASR_%",

        "HSJ_Clean_Retention_%",
        "ZOO_Clean_Retention_%"
    ]
]


# ------------------------------------------------------------
# 6. Affichage
# ------------------------------------------------------------

print("=" * 140)
print("BILAN FINAL — NOTEBOOK 11 — ADVERSARIAL TRAINING")
print("=" * 140)

display(
    final_notebook11_summary.round(4)
)


# ------------------------------------------------------------
# 7. Sauvegarde
# ------------------------------------------------------------

FINAL_NB11_SUMMARY_FILE = (
    FINAL_SELECTION_DIR
    / "final_notebook11_summary.csv"
)

final_notebook11_summary.to_csv(
    FINAL_NB11_SUMMARY_FILE,
    index=False
)

print(
    "\n✅ Tableau bilan final Notebook 11 sauvegardé."
)

print(
    "✅ Aucune attaque supplémentaire exécutée."
)

BILAN FINAL — NOTEBOOK 11 — ADVERSARIAL TRAINING


,Model,Selected_Version,Accuracy,Precision,Recall,F1,ROC_AUC,Before_HSJ_Local_ASR_%,HSJ_Local_ASR_%,HSJ_Gain_points,HSJ_Relative_Reduction_%,Before_ZOO_Raw_ASR_%,After_ZOO_Raw_ASR_%,ZOO_Raw_Gain_points,After_ZOO_Local_005_ASR_%,HSJ_Clean_Retention_%,ZOO_Clean_Retention_%
0,Régression logistique,V2,0.9980,0.4436,0.8310,0.5784,0.9457,22.64,20.7547,1.8853,8.3272,62.26,49.06,13.20,0.0,100.0,100.0
1,SVM linéaire,V2,0.9989,0.6477,0.8028,0.7170,0.9534,28.30,16.9811,11.3189,39.9960,81.13,100.00,-18.87,0.0,100.0,100.0
2,Random Forest,V1,0.9995,0.8889,0.7887,0.8358,0.9685,30.19,5.6604,24.5296,81.2508,NaN,NaN,NaN,NaN,100.0,NaN
3,XGBoost,V1,0.9993,0.8116,0.7887,0.8000,0.9758,35.85,20.7547,15.0953,42.1068,NaN,NaN,NaN,NaN,100.0,NaN
4,LightGBM,V1,0.9995,0.8871,0.7746,0.8271,0.9632,33.96,15.0943,18.8657,55.5526,NaN,NaN,NaN,NaN,100.0,NaN



✅ Tableau bilan final Notebook 11 sauvegardé.
✅ Aucune attaque supplémentaire exécutée.


In [24]:
# ============================================================
# ÉTAPE 16E — HSJ AVANT / APRÈS DÉFENSE
# AUCUNE NOUVELLE ATTAQUE
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Résultats HSJ AVANT défense — Notebook 10
#    53 fraudes communes du TEST
# ------------------------------------------------------------

hsj_before = pd.DataFrame({
    "Model": [
        "Régression logistique",
        "SVM linéaire",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],

    "Before_Success": [
        12,
        15,
        16,
        19,
        18
    ],

    "Before_Attacks": [
        53,
        53,
        53,
        53,
        53
    ],

    "Before_HSJ_Local_ASR_%": [
        22.64,
        28.30,
        30.19,
        35.85,
        33.96
    ]
})

# ------------------------------------------------------------
# 2. Recharger le résumé HSJ APRÈS défense
# ------------------------------------------------------------

HSJ_SUMMARY_FILE = (
    FINAL_SELECTION_DIR
    / "selected_models_hsj_test_summary.csv"
)

hsj_after = pd.read_csv(
    HSJ_SUMMARY_FILE
)

# ------------------------------------------------------------
# 3. Garder les colonnes nécessaires
# ------------------------------------------------------------

hsj_after = hsj_after[
    [
        "Model",
        "Selected_Version",
        "Local_Success",
        "HSJ_Local_ASR_%",
        "Clean_Retention_%"
    ]
].copy()

# ------------------------------------------------------------
# 4. Fusion avant / après
# ------------------------------------------------------------

hsj_before_after = (
    hsj_before
    .merge(
        hsj_after,
        on="Model",
        how="inner"
    )
)

# ------------------------------------------------------------
# 5. Gain de robustesse
#
# Positif  = amélioration
# Zéro     = aucune différence
# Négatif  = dégradation
# ------------------------------------------------------------

hsj_before_after[
    "Robustness_Gain_points"
] = (
    hsj_before_after[
        "Before_HSJ_Local_ASR_%"
    ]
    -
    hsj_before_after[
        "HSJ_Local_ASR_%"
    ]
)

# ------------------------------------------------------------
# 6. Réduction relative de l'ASR
# ------------------------------------------------------------

hsj_before_after[
    "Relative_ASR_Reduction_%"
] = (
    hsj_before_after[
        "Robustness_Gain_points"
    ]
    /
    hsj_before_after[
        "Before_HSJ_Local_ASR_%"
    ]
    *
    100
)

# ------------------------------------------------------------
# 7. Trier par gain de robustesse
# ------------------------------------------------------------

hsj_before_after = (
    hsj_before_after
    .sort_values(
        "Robustness_Gain_points",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 8. Affichage
# ------------------------------------------------------------

print("=" * 115)
print("HSJ — TEST AVANT vs APRÈS DÉFENSE")
print("=" * 115)

display(
    hsj_before_after.round(2)
)

# ------------------------------------------------------------
# 9. Sauvegarde
# ------------------------------------------------------------

HSJ_COMPARISON_FILE = (
    FINAL_SELECTION_DIR
    / "hsj_test_before_after_defense.csv"
)

hsj_before_after.to_csv(
    HSJ_COMPARISON_FILE,
    index=False
)

print("\n✅ Comparaison HSJ avant/après sauvegardée.")
print("✅ Aucune attaque supplémentaire exécutée.")

HSJ — TEST AVANT vs APRÈS DÉFENSE


,Model,Before_Success,Before_Attacks,Before_HSJ_Local_ASR_%,Selected_Version,Local_Success,HSJ_Local_ASR_%,Clean_Retention_%,Robustness_Gain_points,Relative_ASR_Reduction_%
0,Random Forest,16,53,30.19,V1,3,5.66,100.0,24.53,81.25
1,LightGBM,18,53,33.96,V1,8,15.09,100.0,18.87,55.55
2,XGBoost,19,53,35.85,V1,11,20.75,100.0,15.10,42.11
3,SVM linéaire,15,53,28.30,V2,9,16.98,100.0,11.32,40.00
4,Régression logistique,12,53,22.64,V2,11,20.75,100.0,1.89,8.33



✅ Comparaison HSJ avant/après sauvegardée.
✅ Aucune attaque supplémentaire exécutée.


In [23]:
# ============================================================
# ÉTAPE 16D — ZOO AVANT / APRÈS DÉFENSE
# AUCUNE NOUVELLE ATTAQUE
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Baseline officielle Notebook 10
# ------------------------------------------------------------

zoo_before = pd.DataFrame({
    "Model": [
        "Régression logistique",
        "SVM linéaire"
    ],
    "Before_Success": [
        33,
        43
    ],
    "Before_Attacks": [
        53,
        53
    ],
    "Before_Raw_ASR_%": [
        62.26,
        81.13
    ]
})

# ------------------------------------------------------------
# 2. Résultats finaux après défense
# ------------------------------------------------------------

zoo_after = zoo_final_summary[
    [
        "Model",
        "Selected_Version",
        "Raw_Success",
        "ZOO_Raw_ASR_%",
        "Local_Success_005",
        "ZOO_Local_005_ASR_%",
        "Clean_Retention_%"
    ]
].copy()

# ------------------------------------------------------------
# 3. Fusion
# ------------------------------------------------------------

zoo_before_after = (
    zoo_before
    .merge(
        zoo_after,
        on="Model",
        how="inner"
    )
)

# ------------------------------------------------------------
# 4. Gain de robustesse
# positif = amélioration
# négatif = dégradation
# ------------------------------------------------------------

zoo_before_after[
    "Robustness_Gain_points"
] = (
    zoo_before_after["Before_Raw_ASR_%"]
    -
    zoo_before_after["ZOO_Raw_ASR_%"]
)

# ------------------------------------------------------------
# 5. Affichage
# ------------------------------------------------------------

print("=" * 110)
print("ZOO — TEST AVANT vs APRÈS DÉFENSE")
print("=" * 110)

display(
    zoo_before_after.round(2)
)

# ------------------------------------------------------------
# 6. Sauvegarde
# ------------------------------------------------------------

ZOO_COMPARISON_FILE = (
    FINAL_SELECTION_DIR
    / "zoo_test_before_after_defense.csv"
)

zoo_before_after.to_csv(
    ZOO_COMPARISON_FILE,
    index=False
)

print("\n✅ Comparaison ZOO avant/après sauvegardée.")
print("✅ Aucune attaque supplémentaire exécutée.")

ZOO — TEST AVANT vs APRÈS DÉFENSE


,Model,Before_Success,Before_Attacks,Before_Raw_ASR_%,Selected_Version,Raw_Success,ZOO_Raw_ASR_%,Local_Success_005,ZOO_Local_005_ASR_%,Clean_Retention_%,Robustness_Gain_points
0,Régression logistique,33,53,62.26,V2,26,49.06,0,0.0,100.0,13.20
1,SVM linéaire,43,53,81.13,V2,53,100.00,0,0.0,100.0,-18.87



✅ Comparaison ZOO avant/après sauvegardée.
✅ Aucune attaque supplémentaire exécutée.


In [22]:
# ============================================================
# ÉTAPE 16C — RÉSUMÉ FINAL ZOO SUR LE TEST
# AUCUNE ATTAQUE N'EST RELANCÉE
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Recharger les résultats
# ------------------------------------------------------------

zoo_final_test_df = pd.read_csv(
    FINAL_ZOO_FILE
)

assert len(zoo_final_test_df) == 106, (
    f"Erreur : {len(zoo_final_test_df)} résultats "
    "au lieu de 106."
)

# ------------------------------------------------------------
# 2. Définition de l'analyse locale POST-HOC
# ------------------------------------------------------------
# IMPORTANT :
# ZOO n'a PAS été généré avec epsilon = 0.05.
# On filtre seulement après coup les attaques réussies
# dont Linf <= 0.05.
# ------------------------------------------------------------

ZOO_LOCAL_EPS = 0.05

zoo_final_test_df[
    "Local_Success_005"
] = (
    (zoo_final_test_df["Eligible"] == 1)
    &
    (zoo_final_test_df["Attack_Success"] == 1)
    &
    (zoo_final_test_df["Linf"] <= ZOO_LOCAL_EPS)
).astype(int)

# ------------------------------------------------------------
# 3. Résumé par modèle
# ------------------------------------------------------------

zoo_final_summary = (
    zoo_final_test_df
    .groupby(
        ["Model", "Selected_Version"],
        as_index=False
    )
    .agg(
        Attacks=("Source_Position", "count"),
        Eligible=("Eligible", "sum"),
        Raw_Success=("Attack_Success", "sum"),
        Local_Success_005=("Local_Success_005", "sum"),
        Mean_L2=("L2", "mean"),
        Mean_Linf=("Linf", "mean"),
        Mean_Queries=("Queries", "mean"),
        Mean_Time_s=("Time_s", "mean")
    )
)

# ------------------------------------------------------------
# 4. ASR
# ------------------------------------------------------------

zoo_final_summary[
    "ZOO_Raw_ASR_%"
] = np.where(
    zoo_final_summary["Eligible"] > 0,
    zoo_final_summary["Raw_Success"]
    / zoo_final_summary["Eligible"]
    * 100,
    np.nan
)

zoo_final_summary[
    "ZOO_Local_005_ASR_%"
] = np.where(
    zoo_final_summary["Eligible"] > 0,
    zoo_final_summary["Local_Success_005"]
    / zoo_final_summary["Eligible"]
    * 100,
    np.nan
)

# ------------------------------------------------------------
# 5. Maintien de la détection propre
# ------------------------------------------------------------

zoo_final_summary[
    "Clean_Retention_%"
] = (
    zoo_final_summary["Eligible"]
    / 53
    * 100
)

# ------------------------------------------------------------
# 6. Ordre des colonnes
# ------------------------------------------------------------

zoo_final_summary = zoo_final_summary[
    [
        "Model",
        "Selected_Version",
        "Attacks",
        "Eligible",
        "Raw_Success",
        "ZOO_Raw_ASR_%",
        "Local_Success_005",
        "ZOO_Local_005_ASR_%",
        "Clean_Retention_%",
        "Mean_L2",
        "Mean_Linf",
        "Mean_Queries",
        "Mean_Time_s"
    ]
]

print("=" * 110)
print("TEST FINAL — ZOO APRÈS DÉFENSE")
print("=" * 110)

display(
    zoo_final_summary.round(2)
)

# ------------------------------------------------------------
# 7. Sauvegarde
# ------------------------------------------------------------

ZOO_SUMMARY_FILE = (
    FINAL_SELECTION_DIR
    / "selected_models_zoo_test_summary.csv"
)

zoo_final_summary.to_csv(
    ZOO_SUMMARY_FILE,
    index=False
)

print(
    "\n✅ Résumé ZOO final sauvegardé."
)

print(
    "✅ Aucune attaque supplémentaire exécutée."
)

print(
    "⚠️ ZOO_Local_005 = analyse post-hoc Linf <= 0.05,"
)

print(
    "   PAS une attaque ZOO générée sous contrainte epsilon."
)

TEST FINAL — ZOO APRÈS DÉFENSE


,Model,Selected_Version,Attacks,Eligible,Raw_Success,ZOO_Raw_ASR_%,Local_Success_005,ZOO_Local_005_ASR_%,Clean_Retention_%,Mean_L2,Mean_Linf,Mean_Queries,Mean_Time_s
0,Régression logistique,V2,53,53,26,49.06,0,0.0,100.0,0.11,0.05,16355.91,24.04
1,SVM linéaire,V2,53,53,53,100.00,0,0.0,100.0,0.38,0.17,14744.58,55.42



✅ Résumé ZOO final sauvegardé.
✅ Aucune attaque supplémentaire exécutée.
⚠️ ZOO_Local_005 = analyse post-hoc Linf <= 0.05,
   PAS une attaque ZOO générée sous contrainte epsilon.


In [21]:
# ============================================================
# ÉTAPE 16C-0 — AUDIT DU FICHIER ZOO FINAL
# AUCUNE ATTAQUE N'EST RELANCÉE
# ============================================================

import pandas as pd

zoo_final_test_df = pd.read_csv(
    FINAL_ZOO_FILE
)

print("=" * 80)
print("AUDIT ZOO FINAL")
print("=" * 80)

print("\nNombre de lignes :", len(zoo_final_test_df))

print("\nColonnes disponibles :")
print(zoo_final_test_df.columns.tolist())

print("\nNombre d'attaques par modèle :")
display(
    zoo_final_test_df
    .groupby("Model")
    .size()
    .rename("Nombre")
    .to_frame()
)

print("\nPremières lignes :")
display(
    zoo_final_test_df.head()
)

print("\n✅ Audit terminé — aucune attaque exécutée.")

AUDIT ZOO FINAL

Nombre de lignes : 106

Colonnes disponibles :
['Model', 'Source_Position', 'Eligible', 'Clean_Pred', 'Adv_Pred', 'Clean_Score', 'Adv_Score', 'Attack_Success', 'L2', 'Linf', 'Queries', 'Time_s', 'Selected_Version']

Nombre d'attaques par modèle :


,Nombre
Model,
Régression logistique,53
SVM linéaire,53



Premières lignes :


,Model,Source_Position,Eligible,Clean_Pred,Adv_Pred,Clean_Score,Adv_Score,Attack_Success,L2,Linf,Queries,Time_s,Selected_Version
0,Régression logistique,304,1,1,0,0.999943,0.465036,1,0.284205,0.122965,29891,22.868614,V2
1,Régression logistique,1252,1,1,0,0.999880,0.436209,1,0.263316,0.114148,28671,20.885059,V2
2,Régression logistique,2799,1,1,0,0.999661,0.498773,1,0.223292,0.101364,33551,24.326972,V2
3,Régression logistique,3603,1,1,0,0.977448,0.490810,1,0.100757,0.061161,12811,9.421587,V2
4,Régression logistique,4772,1,1,0,0.999943,0.386426,1,0.295444,0.116690,28671,20.645431,V2



✅ Audit terminé — aucune attaque exécutée.


In [20]:
# ============================================================
# ÉTAPE 16B — ZOO FINAL SUR TEST
# Régression logistique V2 + SVM V2
# 53 fraudes × 2 modèles = 106 attaques
# Sauvegarde après CHAQUE attaque
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Données TEST
# ------------------------------------------------------------

X_test_zoo = X_test.reset_index(drop=True)

# ------------------------------------------------------------
# 2. Modèles concernés
# ------------------------------------------------------------

ZOO_FINAL_MODELS = [
    "Régression logistique",
    "SVM linéaire"
]

# ------------------------------------------------------------
# 3. Fichier de sauvegarde / reprise
# ------------------------------------------------------------

FINAL_ZOO_FILE = (
    FINAL_SELECTION_DIR
    / "selected_models_zoo_test.csv"
)

if FINAL_ZOO_FILE.exists():

    zoo_rows = (
        pd.read_csv(FINAL_ZOO_FILE)
        .to_dict("records")
    )

    print(
        "✅ Reprise automatique :",
        len(zoo_rows),
        "/106 déjà calculées"
    )

else:

    zoo_rows = []

    print("▶️ Nouvelle exécution ZOO")

# ------------------------------------------------------------
# 4. Attaques déjà terminées
# ------------------------------------------------------------

completed_zoo = {
    (
        str(row["Model"]),
        int(row["Source_Position"])
    )
    for row in zoo_rows
}

print(
    "▶️ Restantes :",
    106 - len(completed_zoo)
)

# ------------------------------------------------------------
# 5. Exécution
# ------------------------------------------------------------

for model_name in ZOO_FINAL_MODELS:

    model = models_selected[model_name]

    print("\n" + "=" * 75)
    print("ZOO TEST FINAL →", model_name)
    print("=" * 75)

    for i, position in enumerate(
        FINAL_TEST_POSITIONS,
        start=1
    ):

        key = (
            model_name,
            int(position)
        )

        # Ne jamais refaire une attaque déjà sauvegardée
        if key in completed_zoo:
            continue

        output = run_zoo_eval_attack(
            model_name=model_name,
            model=model,
            source_position=int(position),
            X_source=X_test_zoo
        )

        # La fonction peut renvoyer un dict
        # ou un tuple contenant le dict résultat
        if isinstance(output, tuple):

            result_candidates = [
                item
                for item in output
                if isinstance(item, dict)
            ]

            if not result_candidates:
                raise RuntimeError(
                    "Aucun dictionnaire résultat retourné par ZOO."
                )

            result = result_candidates[0]

        elif isinstance(output, dict):

            result = output

        else:

            raise TypeError(
                f"Sortie ZOO inattendue : {type(output)}"
            )

        result = result.copy()

        # Uniformisation
        result["Model"] = model_name
        result["Source_Position"] = int(position)

        result["Selected_Version"] = (
            selection_df.loc[
                selection_df["Model"] == model_name,
                "Selected_Version"
            ].iloc[0]
        )

        zoo_rows.append(result)
        completed_zoo.add(key)

        # ----------------------------------------------------
        # Sauvegarde immédiate
        # ----------------------------------------------------

        pd.DataFrame(
            zoo_rows
        ).to_csv(
            FINAL_ZOO_FILE,
            index=False
        )

        print(
            f"{model_name} | "
            f"{i}/53 | "
            f"total {len(completed_zoo)}/106"
        )

# ------------------------------------------------------------
# 6. Fin
# ------------------------------------------------------------

print("\n" + "=" * 75)

print(
    "✅ ZOO FINAL TERMINÉ :",
    len(completed_zoo),
    "/106"
)

print("=" * 75)

▶️ Nouvelle exécution ZOO
▶️ Restantes : 106

ZOO TEST FINAL → Régression logistique
Régression logistique | 1/53 | total 1/106
Régression logistique | 2/53 | total 2/106
Régression logistique | 3/53 | total 3/106
Régression logistique | 4/53 | total 4/106
Régression logistique | 5/53 | total 5/106
Régression logistique | 6/53 | total 6/106
Régression logistique | 7/53 | total 7/106
Régression logistique | 8/53 | total 8/106
Régression logistique | 9/53 | total 9/106
Régression logistique | 10/53 | total 10/106
Régression logistique | 11/53 | total 11/106
Régression logistique | 12/53 | total 12/106
Régression logistique | 13/53 | total 13/106
Régression logistique | 14/53 | total 14/106
Régression logistique | 15/53 | total 15/106
Régression logistique | 16/53 | total 16/106
Régression logistique | 17/53 | total 17/106
Régression logistique | 18/53 | total 18/106
Régression logistique | 19/53 | total 19/106
Régression logistique | 20/53 | total 20/106
Régression logistique | 21/53 | t

In [19]:
# ============================================================
# RESTAURATION EXACTE DE LA CONFIGURATION ZOO
# ============================================================

import numpy as np
import pandas as pd
import time

FEATURE_COLUMNS = list(X_test.columns)

ZOO_CONFIG_TRAIN = {
    "confidence": 0.5,
    "targeted": True,
    "learning_rate": 1e-3,
    "max_iter": 200,
    "binary_search_steps": 10,
    "initial_const": 0.001,
    "abort_early": True,
    "nb_parallel": len(FEATURE_COLUMNS),
    "batch_size": 1,
    "verbose": False
}

print("✅ ZOO_CONFIG_TRAIN restauré")
print("nb_parallel =", ZOO_CONFIG_TRAIN["nb_parallel"])

# Vérification des vraies dépendances globales seulement
import dis

missing_globals = []

for ins in dis.get_instructions(run_zoo_eval_attack):
    if ins.opname == "LOAD_GLOBAL":
        name = ins.argval

        if (
            name not in globals()
            and name not in dir(__builtins__)
        ):
            missing_globals.append(name)

missing_globals = sorted(set(missing_globals))

print("\nDépendances globales réellement manquantes :")
print(missing_globals)

if not missing_globals:
    print("\n✅ ZOO prêt à être lancé.")
else:
    print("\n❌ Il reste des dépendances à restaurer.")

✅ ZOO_CONFIG_TRAIN restauré
nb_parallel = 30

Dépendances globales réellement manquantes :
[]

✅ ZOO prêt à être lancé.


In [18]:
# ============================================================
# RÉCUPÉRER UNIQUEMENT run_zoo_eval_attack DEPUIS NOTEBOOK 11
# AUCUNE ATTAQUE N'EST LANCÉE
# ============================================================

from pathlib import Path
import json
import ast
import inspect

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

NB11 = next(
    ROOT.rglob("11_adversarial_training_defense.ipynb")
)

with open(NB11, "r", encoding="utf-8") as f:
    nb = json.load(f)

candidates = []

for cell_index, cell in enumerate(nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "run_zoo_eval_attack"
        ):

            args = [
                arg.arg
                for arg in node.args.args
            ]

            candidates.append(
                (
                    cell_index,
                    args,
                    node
                )
            )


print(
    "Définitions run_zoo_eval_attack trouvées :",
    len(candidates)
)

for i, (cell_index, args, node) in enumerate(candidates):

    print(
        f"{i} → cellule {cell_index} | args = {args}"
    )


# On cherche précisément la signature utilisée dans notre protocole
valid = [
    item
    for item in candidates
    if item[1] == [
        "model_name",
        "model",
        "source_position",
        "X_source"
    ]
]

if len(valid) != 1:

    raise RuntimeError(
        f"Je trouve {len(valid)} fonction(s) avec la signature attendue. "
        "ARRÊT pour ne pas charger une mauvaise version."
    )


cell_index, args, node = valid[0]

module = ast.Module(
    body=[node],
    type_ignores=[]
)

ast.fix_missing_locations(module)

exec(
    compile(
        module,
        str(NB11),
        "exec"
    ),
    globals()
)

print("\n✅ run_zoo_eval_attack restaurée.")
print(
    "Signature :",
    inspect.signature(run_zoo_eval_attack)
)

print("\nDépendances manquantes éventuelles :")

missing = []

for name in run_zoo_eval_attack.__code__.co_names:

    if (
        name not in globals()
        and
        name not in dir(__builtins__)
    ):
        missing.append(name)

print(missing)

Définitions run_zoo_eval_attack trouvées : 1
0 → cellule 42 | args = ['model_name', 'model', 'source_position', 'X_source']

✅ run_zoo_eval_attack restaurée.
Signature : (model_name, model, source_position, X_source)

Dépendances manquantes éventuelles :
['iloc', 'to_numpy', 'float32', 'DataFrame', 'predict', 'ZOO_CONFIG_TRAIN', 'reset', 'perf_counter', 'generate', 'reshape', 'array', 'int64', 'astype', 'float64', 'linalg', 'norm', 'queries']


In [16]:
# ============================================================
# RÉSUMÉ FINAL HSJ — TEST APRÈS DÉFENSE
# AUCUNE ATTAQUE N'EST RELANCÉE
# ============================================================

import pandas as pd
import numpy as np

# Recharger les 265 résultats sauvegardés
final_hsj_test_df = pd.read_csv(
    FINAL_HSJ_FILE
)

assert len(final_hsj_test_df) == 265, (
    f"Erreur : {len(final_hsj_test_df)} résultats au lieu de 265."
)

# ------------------------------------------------------------
# Résumé par modèle
# ------------------------------------------------------------

final_hsj_summary = (
    final_hsj_test_df
    .groupby(
        ["Model", "Selected_Version"],
        as_index=False
    )
    .agg(
        Attacks=("Model", "size"),
        Eligible=("Eligible", "sum"),
        Raw_Success=("Raw_Success", "sum"),
        Local_Success=("Local_Success", "sum")
    )
)

# ASR brut
final_hsj_summary["Raw_ASR_%"] = np.where(
    final_hsj_summary["Eligible"] > 0,
    final_hsj_summary["Raw_Success"]
    / final_hsj_summary["Eligible"] * 100,
    np.nan
)

# ASR local principal : L∞ <= 0.05 + variables immuables respectées
final_hsj_summary["HSJ_Local_ASR_%"] = np.where(
    final_hsj_summary["Eligible"] > 0,
    final_hsj_summary["Local_Success"]
    / final_hsj_summary["Eligible"] * 100,
    np.nan
)

# Fraudes du panel encore correctement détectées avant attaque
final_hsj_summary["Clean_Retention_%"] = (
    final_hsj_summary["Eligible"]
    / 53
    * 100
)

print("=" * 95)
print("TEST FINAL — HSJ APRÈS DÉFENSE")
print("=" * 95)

display(
    final_hsj_summary.round(2)
)

# Sauvegarde
final_hsj_summary.to_csv(
    FINAL_SELECTION_DIR /
    "selected_models_hsj_test_summary.csv",
    index=False
)

print("\n✅ Résumé HSJ final sauvegardé.")
print("✅ Aucune nouvelle attaque exécutée.")

TEST FINAL — HSJ APRÈS DÉFENSE


,Model,Selected_Version,Attacks,Eligible,Raw_Success,Local_Success,Raw_ASR_%,HSJ_Local_ASR_%,Clean_Retention_%
0,LightGBM,V1,53,53,53,8,100.0,15.09,100.0
1,Random Forest,V1,53,53,53,3,100.0,5.66,100.0
2,Régression logistique,V2,53,53,53,11,100.0,20.75,100.0
3,SVM linéaire,V2,53,53,53,9,100.0,16.98,100.0
4,XGBoost,V1,53,53,53,11,100.0,20.75,100.0



✅ Résumé HSJ final sauvegardé.
✅ Aucune nouvelle attaque exécutée.


In [15]:
# ============================================================
# CORRECTION RÉELLE + REPRISE HSJ
# ============================================================

import numpy as np
import pandas as pd

# 1) Restaurer l'index des 5 modèles
MODEL_INDEX = {
    name: i
    for i, name in enumerate(models_selected.keys())
}

print("MODEL_INDEX =", MODEL_INDEX)

# 2) Restaurer exactement la config HSJ prévue
HSJ_CONFIG_TRAIN = {
    "targeted": True,
    "norm": np.inf,
    "max_iter": 50,
    "max_eval": 10000,
    "init_eval": 100,
    "init_size": 100,
    "batch_size": 64,
    "verbose": False
}

# 3) Recharger les résultats déjà sauvegardés
saved_df = pd.read_csv(FINAL_HSJ_FILE)

position_col = (
    "Source_Position"
    if "Source_Position" in saved_df.columns
    else "Transaction"
)

completed = {
    (
        str(row["Model"]),
        int(row[position_col])
    )
    for _, row in saved_df.iterrows()
}

rows = saved_df.to_dict("records")

print("✅ Déjà sauvegardées :", len(completed), "/265")
print("▶️ Restantes :", 265 - len(completed))

# Sécurité
if len(completed) < 140:
    raise RuntimeError(
        "Moins de 140 résultats retrouvés : arrêt de sécurité."
    )

# 4) Reprendre uniquement les attaques manquantes
for model_name, model in models_selected.items():

    for position in FINAL_TEST_POSITIONS:

        key = (
            model_name,
            int(position)
        )

        if key in completed:
            continue

        output = run_hsj_eval_attack(
            model_name=model_name,
            model=model,
            source_position=int(position),
            X_source=X_test_hsj,
            seed_X=X_test_hsj,
            seed_y=y_test_hsj
        )

        if isinstance(output, tuple):
            result = next(
                item
                for item in output
                if isinstance(item, dict)
            )
        else:
            result = output

        result = result.copy()

        result["Model"] = model_name
        result["Source_Position"] = int(position)

        result["Selected_Version"] = (
            selection_df.loc[
                selection_df["Model"] == model_name,
                "Selected_Version"
            ].iloc[0]
        )

        rows.append(result)
        completed.add(key)

        # Sauvegarde après chaque attaque
        pd.DataFrame(rows).to_csv(
            FINAL_HSJ_FILE,
            index=False
        )

        print(
            f"{model_name} → {len(completed)}/265"
        )

print("\n✅ HSJ terminé :", len(completed), "/265")

MODEL_INDEX = {'Régression logistique': 0, 'SVM linéaire': 1, 'Random Forest': 2, 'XGBoost': 3, 'LightGBM': 4}
✅ Déjà sauvegardées : 140 /265
▶️ Restantes : 125
Random Forest → 141/265
Random Forest → 142/265
Random Forest → 143/265
Random Forest → 144/265
Random Forest → 145/265
Random Forest → 146/265
Random Forest → 147/265
Random Forest → 148/265
Random Forest → 149/265
Random Forest → 150/265
Random Forest → 151/265
Random Forest → 152/265
Random Forest → 153/265
Random Forest → 154/265
Random Forest → 155/265
Random Forest → 156/265
Random Forest → 157/265
Random Forest → 158/265
Random Forest → 159/265
XGBoost → 160/265
XGBoost → 161/265
XGBoost → 162/265
XGBoost → 163/265
XGBoost → 164/265
XGBoost → 165/265
XGBoost → 166/265
XGBoost → 167/265
XGBoost → 168/265
XGBoost → 169/265
XGBoost → 170/265
XGBoost → 171/265
XGBoost → 172/265
XGBoost → 173/265
XGBoost → 174/265
XGBoost → 175/265
XGBoost → 176/265
XGBoost → 177/265
XGBoost → 178/265
XGBoost → 179/265
XGBoost → 180/265
XGBoo

In [14]:
# ============================================================
# TROUVER LE DICTIONNAIRE FAUTIF DANS find_hsj_eval_seed
# AUCUNE ATTAQUE N'EST LANCÉE
# ============================================================

print("Fonction :", find_hsj_eval_seed.__name__)
print("\nDictionnaires utilisés par find_hsj_eval_seed :\n")

for name in find_hsj_eval_seed.__code__.co_names:

    obj = find_hsj_eval_seed.__globals__.get(name)

    if isinstance(obj, dict):

        print(
            f"{name} :",
            list(obj.keys())
        )

Fonction : find_hsj_eval_seed

Dictionnaires utilisés par find_hsj_eval_seed :

MODEL_INDEX : []


In [13]:
import dis

instructions = list(
    dis.get_instructions(run_hsj_eval_attack)
)

print("=== ACCÈS AVEC model_name ===\n")

for i, ins in enumerate(instructions):

    if ins.opname == "LOAD_FAST" and ins.argval == "model_name":

        print("=" * 70)

        for x in instructions[
            max(0, i - 8):
            min(len(instructions), i + 12)
        ]:
            print(
                f"{x.offset:4} | "
                f"{x.opname:25} | "
                f"{x.argrepr}"
            )

=== ACCÈS AVEC model_name ===

 248 | LOAD_CONST                | 0
 250 | BINARY_SUBSCR             | 
 254 | CALL                      | 
 262 | STORE_FAST                | clean_score
 264 | LOAD_FAST                 | clean_pred
 266 | LOAD_CONST                | 1
 268 | COMPARE_OP                | bool(!=)
 272 | POP_JUMP_IF_FALSE         | to L1
 276 | LOAD_FAST                 | model_name
 278 | LOAD_FAST                 | source_position
 280 | LOAD_CONST                | 0
 282 | LOAD_FAST                 | clean_pred
 284 | LOAD_FAST                 | clean_score
 286 | LOAD_CONST                | 0
 288 | LOAD_CONST                | 0
 290 | LOAD_CONST                | ('Model', 'Source_Position', 'Eligible', 'Clean_Pred', 'Clean_Score', 'Raw_Success', 'Local_Success')
 292 | BUILD_CONST_KEY_MAP       | 
 294 | LOAD_CONST                | None
 296 | BUILD_TUPLE               | 
 298 | RETURN_VALUE              | 
 286 | LOAD_CONST                | 0
 288 | LOAD_CONST     

In [12]:
import dis

instructions = list(
    dis.get_instructions(run_hsj_eval_attack)
)

for i, ins in enumerate(instructions):

    if (
        ins.argval == "HSJ_CONFIG_TRAIN"
        or ins.argrepr == "HSJ_CONFIG_TRAIN"
    ):

        print("\n" + "=" * 70)
        print("UTILISATION DE HSJ_CONFIG_TRAIN")
        print("=" * 70)

        for x in instructions[
            max(0, i - 5):
            min(len(instructions), i + 15)
        ]:

            print(
                f"{x.offset:4} | "
                f"{x.opname:25} | "
                f"{x.argrepr}"
            )


UTILISATION DE HSJ_CONFIG_TRAIN
 388 | LOAD_GLOBAL               | HopSkipJump + NULL
 398 | LOAD_CONST                | ()
 400 | LOAD_CONST                | 'classifier'
 402 | LOAD_FAST                 | classifier
 404 | BUILD_MAP                 | 
 406 | LOAD_GLOBAL               | HSJ_CONFIG_TRAIN
 416 | DICT_MERGE                | 
 418 | CALL_FUNCTION_EX          | 
 420 | STORE_FAST                | attack
 422 | LOAD_FAST                 | counter
 424 | LOAD_ATTR                 | reset + NULL|self
 444 | CALL                      | 
 452 | POP_TOP                   | 
 454 | LOAD_GLOBAL               | time
 464 | LOAD_ATTR                 | perf_counter + NULL|self
 484 | CALL                      | 
 492 | STORE_FAST                | start
 494 | LOAD_FAST                 | attack
 496 | LOAD_ATTR                 | generate + NULL|self
 516 | LOAD_FAST                 | x_clean


In [11]:
import dis

dis.dis(run_hsj_eval_attack)

  1           RESUME                   0

 11           LOAD_GLOBAL              1 (int + NULL)

 12           LOAD_FAST                2 (source_position)

 11           CALL                     1
              STORE_FAST               2 (source_position)

 16           LOAD_FAST                3 (X_source)

 17           LOAD_ATTR                2 (iloc)
              LOAD_FAST                2 (source_position)

 16           BINARY_SUBSCR

 18           LOAD_ATTR                5 (to_numpy + NULL|self)

 19           LOAD_GLOBAL              6 (np)
              LOAD_ATTR                8 (float32)

 18           LOAD_CONST               1 (('dtype',))
              CALL_KW                  1

 15           STORE_FAST               7 (x_clean)

 23           LOAD_GLOBAL             10 (pd)
              LOAD_ATTR               13 (DataFrame + NULL|self)

 24           LOAD_FAST                7 (x_clean)
              BUILD_LIST               1

 25           LOAD_GLOBAL           

In [10]:
# ============================================================
# CORRECTION DÉFINITIVE HSJ_CONFIG_TRAIN + REPRISE
# ============================================================

import pandas as pd
import shutil

# ------------------------------------------------------------
# 1. SÉCURISER IMMÉDIATEMENT LES RÉSULTATS EXISTANTS
# ------------------------------------------------------------

saved_df = pd.read_csv(FINAL_HSJ_FILE)

print("✅ Résultats retrouvés :", len(saved_df), "/265")

backup_file = FINAL_HSJ_FILE.with_name(
    "selected_models_hsj_test_BACKUP.csv"
)

shutil.copy2(
    FINAL_HSJ_FILE,
    backup_file
)

print("✅ Copie de sécurité créée :", backup_file.name)


# ------------------------------------------------------------
# 2. RÉCUPÉRER LA CONFIGURATION HSJ RÉELLE
# ------------------------------------------------------------

PARAMS = [
    "targeted",
    "norm",
    "max_iter",
    "max_eval",
    "init_eval",
    "init_size",
    "batch_size",
    "verbose"
]

MODEL_NAMES = list(models_selected.keys())

current_cfg = HSJ_CONFIG_TRAIN

base_cfg = None

# Cas 1 : configuration simple actuelle
if all(k in current_cfg for k in PARAMS):

    base_cfg = {
        k: current_cfg[k]
        for k in PARAMS
    }

# Cas 2 : configuration déjà transformée par modèle
else:

    for model_name in MODEL_NAMES:

        value = current_cfg.get(model_name)

        if isinstance(value, dict):

            if all(k in value for k in PARAMS):

                base_cfg = {
                    k: value[k]
                    for k in PARAMS
                }

                break


if base_cfg is None:

    raise RuntimeError(
        "Impossible de retrouver la configuration HSJ."
    )


print("\nConfiguration HSJ conservée :")
print(base_cfg)


# ------------------------------------------------------------
# 3. CONFIGURATION HYBRIDE
#
# Fonctionne avec :
# HSJ_CONFIG_TRAIN["targeted"]
# ET
# HSJ_CONFIG_TRAIN["Random Forest"]
# ------------------------------------------------------------

class HybridHSJConfig(dict):

    def __init__(self, base, model_names):

        super().__init__(base)

        self.base = dict(base)
        self.model_names = set(model_names)

    def __getitem__(self, key):

        if key in self.model_names:
            return dict(self.base)

        return super().__getitem__(key)

    def get(self, key, default=None):

        if key in self.model_names:
            return dict(self.base)

        return super().get(key, default)


HSJ_CONFIG_TRAIN = HybridHSJConfig(
    base_cfg,
    MODEL_NAMES
)


# ------------------------------------------------------------
# 4. CONTRÔLE AVANT DE LANCER QUOI QUE CE SOIT
# ------------------------------------------------------------

print("\n✅ Test paramètres :")
print(
    "max_iter =",
    HSJ_CONFIG_TRAIN["max_iter"]
)

print(
    "Random Forest max_iter =",
    HSJ_CONFIG_TRAIN["Random Forest"]["max_iter"]
)

print(
    "XGBoost max_iter =",
    HSJ_CONFIG_TRAIN["XGBoost"]["max_iter"]
)


# ------------------------------------------------------------
# 5. RECONSTRUIRE LA LISTE DES ATTAQUES DÉJÀ FAITES
# ------------------------------------------------------------

position_col = (
    "Source_Position"
    if "Source_Position" in saved_df.columns
    else "Transaction"
)

completed = {
    (
        str(row["Model"]),
        int(row[position_col])
    )
    for _, row in saved_df.iterrows()
}

rows = saved_df.to_dict("records")

print(
    "\n✅ Déjà terminées :",
    len(completed),
    "/265"
)

print(
    "▶️ Restantes :",
    265 - len(completed)
)


# ------------------------------------------------------------
# 6. REPRENDRE UNIQUEMENT LES MANQUANTES
# ------------------------------------------------------------

for model_name, model in models_selected.items():

    for position in FINAL_TEST_POSITIONS:

        key = (
            model_name,
            int(position)
        )

        if key in completed:
            continue

        output = run_hsj_eval_attack(
            model_name=model_name,
            model=model,
            source_position=int(position),
            X_source=X_test_hsj,
            seed_X=X_test_hsj,
            seed_y=y_test_hsj
        )

        if isinstance(output, tuple):

            result = next(
                item
                for item in output
                if isinstance(item, dict)
            )

        else:

            result = output

        result = result.copy()

        result["Model"] = model_name
        result["Source_Position"] = int(position)

        result["Selected_Version"] = (
            selection_df.loc[
                selection_df["Model"] == model_name,
                "Selected_Version"
            ].iloc[0]
        )

        rows.append(result)
        completed.add(key)

        # sauvegarde après CHAQUE attaque
        pd.DataFrame(rows).to_csv(
            FINAL_HSJ_FILE,
            index=False
        )

        print(
            f"{model_name} → {len(completed)}/265"
        )


print("\n" + "=" * 70)
print("✅ HSJ TERMINÉ :", len(completed), "/265")
print("=" * 70)

✅ Résultats retrouvés : 140 /265
✅ Copie de sécurité créée : selected_models_hsj_test_BACKUP.csv

Configuration HSJ conservée :
{'targeted': True, 'norm': inf, 'max_iter': 50, 'max_eval': 10000, 'init_eval': 100, 'init_size': 100, 'batch_size': 64, 'verbose': False}

✅ Test paramètres :
max_iter = 50
Random Forest max_iter = 50
XGBoost max_iter = 50

✅ Déjà terminées : 140 /265
▶️ Restantes : 125


KeyError: 'Random Forest'

In [7]:
# ============================================================
# DIAGNOSTIC EXACT DU KEYERROR RANDOM FOREST
# ============================================================

import types

print("Fonction :", run_hsj_eval_attack.__name__)
print("\nDictionnaires utilisés par cette fonction :\n")

for name in run_hsj_eval_attack.__code__.co_names:

    obj = run_hsj_eval_attack.__globals__.get(name)

    if isinstance(obj, dict):

        print(
            f"{name} :",
            list(obj.keys())
        )

Fonction : run_hsj_eval_attack

Dictionnaires utilisés par cette fonction :

HSJ_CONFIG_TRAIN : ['targeted', 'norm', 'max_iter', 'max_eval', 'init_eval', 'init_size', 'batch_size', 'verbose']


In [2]:
# ============================================================
# RÉCUPÉRATION APRÈS REDÉMARRAGE DU KERNEL
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# ------------------------------------------------------------
# 1. Racine du projet
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("PROJECT_ROOT :", PROJECT_ROOT)


# ------------------------------------------------------------
# 2. Recharger TEST
# ------------------------------------------------------------

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test.parquet"
)

y_test = pd.read_parquet(
    PROCESSED_DIR / "y_test.parquet"
).squeeze()

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# ------------------------------------------------------------
# 3. Retrouver automatiquement la sélection finale
# ------------------------------------------------------------

selection_files = list(
    PROJECT_ROOT.rglob("final_model_selection.csv")
)

if not selection_files:
    raise FileNotFoundError(
        "final_model_selection.csv introuvable."
    )

selection_file = selection_files[0]

FINAL_SELECTION_DIR = selection_file.parent

selection_df = pd.read_csv(
    selection_file
)

print("\nDossier sélection finale :")
print(FINAL_SELECTION_DIR)

display(selection_df)


# ------------------------------------------------------------
# 4. Recharger les 5 modèles sélectionnés
# ------------------------------------------------------------

model_files = {
    "Régression logistique":
        "logistic_regression_selected.joblib",

    "SVM linéaire":
        "linear_svm_selected.joblib",

    "Random Forest":
        "random_forest_selected.joblib",

    "XGBoost":
        "xgboost_selected.joblib",

    "LightGBM":
        "lightgbm_selected.joblib"
}

models_selected = {}

for model_name, filename in model_files.items():

    path = FINAL_SELECTION_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Modèle introuvable : {path}"
        )

    models_selected[model_name] = joblib.load(path)

    print("✅", model_name, "rechargé")


# ------------------------------------------------------------
# 5. Fraudes communes TEST
# ------------------------------------------------------------

common_files = list(
    PROJECT_ROOT.rglob(
        "common_test_fraud_positions.npy"
    )
)

if not common_files:
    raise FileNotFoundError(
        "common_test_fraud_positions.npy introuvable."
    )

common_test_fraud_positions = np.load(
    common_files[0]
).astype(int)

print(
    "\nFraudes TEST communes :",
    len(common_test_fraud_positions)
)


# ------------------------------------------------------------
# 6. Vérifier la progression HSJ sauvegardée
# ------------------------------------------------------------

FINAL_HSJ_FILE = (
    FINAL_SELECTION_DIR /
    "selected_models_hsj_test.csv"
)

if FINAL_HSJ_FILE.exists():

    recovery_hsj = pd.read_csv(
        FINAL_HSJ_FILE
    )

    print(
        "\n✅ Attaques HSJ déjà sauvegardées :",
        len(recovery_hsj),
        "/ 265"
    )

    print("\nPar modèle :")

    display(
        recovery_hsj
        .groupby("Model")
        .size()
        .rename("Nombre")
        .to_frame()
    )

else:

    print(
        "\n⚠️ Fichier de progression HSJ introuvable."
    )

PROJECT_ROOT : c:\Users\MSI\Desktop\fraud_detection_memoire
X_test : (42559, 30)
y_test : (42559,)

Dossier sélection finale :
c:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training\defence_v2\final_selected_models


,Model,Selected_Version
0,Régression logistique,V2
1,SVM linéaire,V2
2,Random Forest,V1
3,XGBoost,V1
4,LightGBM,V1


✅ Régression logistique rechargé
✅ SVM linéaire rechargé
✅ Random Forest rechargé
✅ XGBoost rechargé
✅ LightGBM rechargé

Fraudes TEST communes : 53

✅ Attaques HSJ déjà sauvegardées : 140 / 265

Par modèle :


,Nombre
Model,
Random Forest,34
Régression logistique,53
SVM linéaire,53


In [95]:
# ============================================================
# ÉTAPE 14 — OUVERTURE DU TEST FINAL
# ÉVALUATION CLEAN DES 5 MODÈLES SÉLECTIONNÉS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("=" * 80)
print("TEST FINAL — PERFORMANCE CLEAN")
print("=" * 80)

test_clean_rows = []

for model_name, model in models_selected.items():

    print("\n→", model_name)

    # --------------------------------------------------------
    # Prédiction sur TEST
    # --------------------------------------------------------

    y_pred = np.asarray(
        model.predict(X_test)
    ).ravel()

    # --------------------------------------------------------
    # Score pour ROC-AUC
    # --------------------------------------------------------

    if hasattr(model, "predict_proba"):

        y_score = (
            model.predict_proba(X_test)[:, 1]
        )

    elif hasattr(model, "decision_function"):

        y_score = np.asarray(
            model.decision_function(X_test)
        ).ravel()

    else:

        raise TypeError(
            f"Aucun score disponible pour {model_name}"
        )

    # --------------------------------------------------------
    # Matrice de confusion
    # --------------------------------------------------------

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    # --------------------------------------------------------
    # Métriques
    # --------------------------------------------------------

    test_clean_rows.append({

        "Model":
            model_name,

        "Selected_Version":
            selection_df.loc[
                selection_df["Model"] == model_name,
                "Selected_Version"
            ].iloc[0],

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Precision":
            precision_score(
                y_test,
                y_pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_test,
                y_pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_test,
                y_pred,
                zero_division=0
            ),

        "ROC_AUC":
            roc_auc_score(
                y_test,
                y_score
            ),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp)
    })


# ============================================================
# TABLEAU FINAL CLEAN TEST
# ============================================================

test_selected_clean_df = pd.DataFrame(
    test_clean_rows
)

display(
    test_selected_clean_df.round(4)
)


# ============================================================
# SAUVEGARDE
# ============================================================

test_selected_clean_df.to_csv(
    FINAL_SELECTION_DIR /
    "selected_models_clean_test.csv",
    index=False
)

print("\n✅ Évaluation CLEAN sur TEST terminée.")
print("✅ Aucun modèle réentraîné.")
print("✅ Sélection toujours figée.")

TEST FINAL — PERFORMANCE CLEAN

→ Régression logistique

→ SVM linéaire

→ Random Forest

→ XGBoost

→ LightGBM


,Model,Selected_Version,Accuracy,Precision,Recall,F1,ROC_AUC,TN,FP,FN,TP
0,Régression logistique,V2,0.9980,0.4436,0.8310,0.5784,0.9457,42414,74,12,59
1,SVM linéaire,V2,0.9989,0.6477,0.8028,0.7170,0.9534,42457,31,14,57
2,Random Forest,V1,0.9995,0.8889,0.7887,0.8358,0.9685,42481,7,15,56
3,XGBoost,V1,0.9993,0.8116,0.7887,0.8000,0.9758,42475,13,15,56
4,LightGBM,V1,0.9995,0.8871,0.7746,0.8271,0.9632,42481,7,16,55



✅ Évaluation CLEAN sur TEST terminée.
✅ Aucun modèle réentraîné.
✅ Sélection toujours figée.


In [94]:
# ============================================================
# ÉTAPE 13 — SÉLECTION FINALE DES MODÈLES AVANT TEST
# ============================================================

from pathlib import Path
import pandas as pd
import joblib

FINAL_SELECTION_DIR = (
    V2_DIR /
    "final_selected_models"
)

FINAL_SELECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Modèles retenus
# ------------------------------------------------------------

models_selected = {
    "Régression logistique":
        models_v2["Régression logistique"],

    "SVM linéaire":
        models_v2["SVM linéaire"],

    "Random Forest":
        models_defended["Random Forest"],

    "XGBoost":
        models_defended["XGBoost"],

    "LightGBM":
        models_defended["LightGBM"]
}


# ------------------------------------------------------------
# Versions retenues
# ------------------------------------------------------------

selection_df = pd.DataFrame([
    {
        "Model": "Régression logistique",
        "Selected_Version": "V2"
    },
    {
        "Model": "SVM linéaire",
        "Selected_Version": "V2"
    },
    {
        "Model": "Random Forest",
        "Selected_Version": "V1"
    },
    {
        "Model": "XGBoost",
        "Selected_Version": "V1"
    },
    {
        "Model": "LightGBM",
        "Selected_Version": "V1"
    }
])


# ------------------------------------------------------------
# Sauvegarde des modèles
# ------------------------------------------------------------

model_files = {
    "Régression logistique":
        "logistic_regression_selected.joblib",

    "SVM linéaire":
        "linear_svm_selected.joblib",

    "Random Forest":
        "random_forest_selected.joblib",

    "XGBoost":
        "xgboost_selected.joblib",

    "LightGBM":
        "lightgbm_selected.joblib"
}


for model_name, model in models_selected.items():

    joblib.dump(
        model,
        FINAL_SELECTION_DIR /
        model_files[model_name]
    )


# ------------------------------------------------------------
# Sauvegarde de la décision
# ------------------------------------------------------------

selection_df.to_csv(
    FINAL_SELECTION_DIR /
    "final_model_selection.csv",
    index=False
)


print("=" * 75)
print("SÉLECTION FINALE AVANT TEST")
print("=" * 75)

display(selection_df)

print()
print("✅ Sélection Validation figée.")
print("✅ Modèles sauvegardés.")
print("✅ Aucun modèle ne sera encore modifié.")
print("🔒 Test toujours fermé.")

SÉLECTION FINALE AVANT TEST


,Model,Selected_Version
0,Régression logistique,V2
1,SVM linéaire,V2
2,Random Forest,V1
3,XGBoost,V1
4,LightGBM,V1



✅ Sélection Validation figée.
✅ Modèles sauvegardés.
✅ Aucun modèle ne sera encore modifié.
🔒 Test toujours fermé.


In [93]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 12
# TABLEAU FINAL VALIDATION : ORIGINAL vs V1 vs V2
# ============================================================

import pandas as pd
import numpy as np


TARGET_MODELS = [
    "Régression logistique",
    "SVM linéaire"
]


# ============================================================
# 1. CLEAN : ORIGINAL + V1
# ============================================================

clean_old = (
    validation_clean_df[
        validation_clean_df["Model"].isin(TARGET_MODELS)
    ]
    .copy()
)

clean_old["Version"] = (
    clean_old["Condition"]
    .replace({
        "Original": "Original",
        "Defended": "V1"
    })
)

clean_old = clean_old[
    [
        "Model",
        "Version",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "FN",
        "TP"
    ]
]


# ============================================================
# 2. CLEAN : V2
# ============================================================

clean_v2 = (
    v2_validation_clean[
        v2_validation_clean["Model"].isin(TARGET_MODELS)
    ]
    .copy()
)

clean_v2["Version"] = "V2"

clean_v2 = clean_v2[
    [
        "Model",
        "Version",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "FN",
        "TP"
    ]
]


clean_all = pd.concat(
    [
        clean_old,
        clean_v2
    ],
    ignore_index=True
)


# ============================================================
# 3. HSJ : ORIGINAL + V1
# ============================================================

hsj_old = (
    val_hsj_summary[
        val_hsj_summary["Model"].isin(TARGET_MODELS)
    ]
    .copy()
)

hsj_old["Version"] = (
    hsj_old["Condition"]
    .replace({
        "Original": "Original",
        "Defended": "V1"
    })
)

hsj_old = hsj_old[
    [
        "Model",
        "Version",
        "ASR_%"
    ]
].rename(
    columns={
        "ASR_%": "HSJ_ASR_%"
    }
)


# ============================================================
# 4. HSJ : V2
# ============================================================

hsj_v2 = (
    v2_hsj_summary[
        v2_hsj_summary["Model"].isin(TARGET_MODELS)
    ]
    .copy()
)

hsj_v2["Version"] = "V2"

hsj_v2 = hsj_v2[
    [
        "Model",
        "Version",
        "ASR_%"
    ]
].rename(
    columns={
        "ASR_%": "HSJ_ASR_%"
    }
)


hsj_all = pd.concat(
    [
        hsj_old,
        hsj_v2
    ],
    ignore_index=True
)


# ============================================================
# 5. ZOO BRUT + ZOO LOCAL <= 0.05
# ============================================================

zoo_final = (
    zoo_local_summary[
        zoo_local_summary["Model"].isin(TARGET_MODELS)
    ]
    .copy()
)

zoo_final["Version"] = (
    zoo_final["Condition"]
    .replace({
        "Original": "Original",
        "Defended": "V1",
        "V2": "V2"
    })
)

zoo_final = zoo_final[
    [
        "Model",
        "Version",
        "Raw_ASR_%",
        "Local_ASR_005_%"
    ]
].rename(
    columns={
        "Raw_ASR_%":
            "ZOO_Raw_ASR_%",
        "Local_ASR_005_%":
            "ZOO_Local_005_ASR_%"
    }
)


# ============================================================
# 6. FUSION FINALE
# ============================================================

validation_final_comparison = (
    clean_all
    .merge(
        hsj_all,
        on=[
            "Model",
            "Version"
        ],
        how="left"
    )
    .merge(
        zoo_final,
        on=[
            "Model",
            "Version"
        ],
        how="left"
    )
)


# ============================================================
# 7. ORDRE
# ============================================================

version_order = {
    "Original": 0,
    "V1": 1,
    "V2": 2
}

model_order = {
    "Régression logistique": 0,
    "SVM linéaire": 1
}

validation_final_comparison[
    "_model_order"
] = validation_final_comparison[
    "Model"
].map(model_order)

validation_final_comparison[
    "_version_order"
] = validation_final_comparison[
    "Version"
].map(version_order)

validation_final_comparison = (
    validation_final_comparison
    .sort_values(
        [
            "_model_order",
            "_version_order"
        ]
    )
    .drop(
        columns=[
            "_model_order",
            "_version_order"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 8. AFFICHAGE
# ============================================================

print("=" * 105)
print("COMPARAISON FINALE SUR VALIDATION")
print("ORIGINAL vs V1 vs V2")
print("=" * 105)

display(
    validation_final_comparison[
        [
            "Model",
            "Version",
            "Recall",
            "F1",
            "FN",
            "TP",
            "HSJ_ASR_%",
            "ZOO_Raw_ASR_%",
            "ZOO_Local_005_ASR_%"
        ]
    ].round(4)
)


# ============================================================
# 9. SAUVEGARDE
# ============================================================

validation_final_comparison.to_csv(
    V2_DIR /
    "validation_original_v1_v2_comparison.csv",
    index=False
)

print(
    "\n✅ Tableau final Validation créé."
)

print(
    "🔒 TEST toujours fermé."
)

COMPARAISON FINALE SUR VALIDATION
ORIGINAL vs V1 vs V2


,Model,Version,Recall,F1,FN,TP,HSJ_ASR_%,ZOO_Raw_ASR_%,ZOO_Local_005_ASR_%
0,Régression logistique,Original,0.8310,0.6114,12,59,25.0,55.0,0.0
1,Régression logistique,V1,0.8310,0.5990,12,59,20.0,60.0,0.0
2,Régression logistique,V2,0.8310,0.5930,12,59,15.0,55.0,0.0
3,SVM linéaire,Original,0.8169,0.7342,13,58,30.0,100.0,15.0
4,SVM linéaire,V1,0.8169,0.7436,13,58,25.0,100.0,0.0
5,SVM linéaire,V2,0.8169,0.7484,13,58,15.0,100.0,0.0



✅ Tableau final Validation créé.
🔒 TEST toujours fermé.


In [92]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 11
# HSJ CONTRE LR V2 ET SVM V2 — VALIDATION
# VERSION CORRIGÉE
# ============================================================

import pandas as pd
import numpy as np


V2_HSJ_VAL_FILE = (
    V2_VALIDATION_DIR /
    "v2_hsj_validation.csv"
)


# ============================================================
# 1. MÊME PANEL DE VALIDATION QUE V1
# ============================================================

V2_HSJ_VAL_POSITIONS = sorted(
    validation_hsj_df[
        "Source_Position"
    ]
    .astype(int)
    .unique()
    .tolist()
)

print(
    "Fraudes Validation utilisées :",
    len(V2_HSJ_VAL_POSITIONS)
)

assert (
    len(V2_HSJ_VAL_POSITIONS)
    == 20
), "On attend exactement 20 fraudes."


# ============================================================
# 2. DONNÉES POUR LES SEEDS HSJ
# ============================================================
# HSJ a besoin d'exemples normaux pour commencer son attaque.
# On utilise la Validation, comme ensemble d'évaluation.
# Aucun entraînement ici.
# ============================================================

X_val_hsj = (
    X_val
    .reset_index(drop=True)
)

y_val_hsj = (
    y_val
    .reset_index(drop=True)
)


print(
    "Validation disponible pour les seeds :",
    X_val_hsj.shape
)

print(
    "Normales disponibles :",
    int(
        (y_val_hsj == 0).sum()
    )
)


# ============================================================
# 3. FONCTION D'APPEL HSJ V2
# ============================================================

def call_hsj_v2(
    model_name,
    model,
    position
):

    output = run_hsj_eval_attack(
        model_name=model_name,
        model=model,
        source_position=int(position),

        # transaction frauduleuse attaquée
        X_source=X_val_hsj,

        # pool utilisé pour trouver le seed normal
        seed_X=X_val_hsj,
        seed_y=y_val_hsj
    )

    # --------------------------------------------------------
    # Certaines fonctions renvoient un tuple
    # On récupère le dictionnaire résultat
    # --------------------------------------------------------

    if isinstance(
        output,
        tuple
    ):

        dict_items = [
            item
            for item in output
            if isinstance(
                item,
                dict
            )
        ]

        if len(dict_items) == 0:

            raise TypeError(
                "run_hsj_eval_attack a renvoyé "
                "un tuple sans dictionnaire."
            )

        result = dict_items[0]

    elif isinstance(
        output,
        dict
    ):

        result = output

    else:

        raise TypeError(
            "Type inattendu renvoyé par HSJ : "
            f"{type(output)}"
        )

    result = result.copy()

    result[
        "Condition"
    ] = "V2"

    return result


# ============================================================
# 4. LANCEMENT HSJ
# ============================================================

v2_hsj_rows = []


for model_name in [
    "Régression logistique",
    "SVM linéaire"
]:

    print(
        "\n" + "=" * 70
    )

    print(
        "HSJ VALIDATION V2 →",
        model_name
    )

    print(
        "=" * 70
    )

    model = models_v2[
        model_name
    ]


    for i, position in enumerate(
        V2_HSJ_VAL_POSITIONS,
        start=1
    ):

        result = call_hsj_v2(
            model_name=model_name,
            model=model,
            position=position
        )

        v2_hsj_rows.append(
            result
        )

        print(
            f"{model_name} | {i}/20",
            end="\r"
        )


# ============================================================
# 5. DATAFRAME
# ============================================================

v2_hsj_validation_df = pd.DataFrame(
    v2_hsj_rows
)


print("\n")

print(
    "Nombre total de résultats :",
    len(v2_hsj_validation_df)
)

assert (
    len(v2_hsj_validation_df)
    == 40
), (
    "On attend 40 résultats "
    "(20 LR + 20 SVM)."
)


# ============================================================
# 6. SAUVEGARDE
# ============================================================

v2_hsj_validation_df.to_csv(
    V2_HSJ_VAL_FILE,
    index=False
)


# ============================================================
# 7. RÉSUMÉ HSJ LOCAL
# ============================================================

v2_hsj_summary = (
    v2_hsj_validation_df
    .groupby(
        "Model"
    )
    .agg(

        Eligible=(
            "Eligible",
            "sum"
        ),

        Successful=(
            "Local_Success",
            "sum"
        )

    )
    .reset_index()
)


v2_hsj_summary[
    "ASR_%"
] = (
    v2_hsj_summary[
        "Successful"
    ]
    /
    v2_hsj_summary[
        "Eligible"
    ]
    *
    100
)


print(
    "\n" + "=" * 70
)

print(
    "RÉSUMÉ HSJ — V2"
)

print(
    "=" * 70
)


display(
    v2_hsj_summary.round(2)
)


print(
    "\n✅ HSJ V2 terminé."
)

print(
    "✅ Même contrainte locale ε = 0.05."
)

print(
    "🔒 Test toujours fermé."
)

Fraudes Validation utilisées : 20
Validation disponible pour les seeds : (42559, 30)
Normales disponibles : 42488

HSJ VALIDATION V2 → Régression logistique
Régression logistique | 20/20
HSJ VALIDATION V2 → SVM linéaire
SVM linéaire | 20/20

Nombre total de résultats : 40

RÉSUMÉ HSJ — V2


,Model,Eligible,Successful,ASR_%
0,Régression logistique,20,3,15.0
1,SVM linéaire,20,3,15.0



✅ HSJ V2 terminé.
✅ Même contrainte locale ε = 0.05.
🔒 Test toujours fermé.


In [90]:
# ============================================================
# V2 — ÉTAPE 11A
# VÉRIFIER L'APPEL EXACT DE HSJ
# ============================================================

import inspect

print("=" * 70)
print("HSJ — FONCTION D'ÉVALUATION")
print("=" * 70)

print(
    "run_hsj_eval_attack",
    inspect.signature(
        run_hsj_eval_attack
    )
)

print("\nPanel HSJ V1 :")

print(
    "Nombre de positions =",
    validation_hsj_df[
        "Source_Position"
    ]
    .nunique()
)

print("\n✅ Aucun calcul lancé.")
print("✅ Test toujours fermé.")

HSJ — FONCTION D'ÉVALUATION
run_hsj_eval_attack (model_name, model, source_position, X_source, seed_X, seed_y, salt=0)

Panel HSJ V1 :
Nombre de positions = 20

✅ Aucun calcul lancé.
✅ Test toujours fermé.


In [89]:
# ============================================================
# ANALYSE ZOO LOCALE — EPS = 0.05
# AUCUNE NOUVELLE ATTAQUE
# ============================================================

import pandas as pd
import numpy as np

ZOO_LOCAL_EPS = 0.05

# ------------------------------------------------------------
# Original + V1
# ------------------------------------------------------------

zoo_old = validation_zoo_df.copy()

# ------------------------------------------------------------
# V2
# ------------------------------------------------------------

zoo_v2 = v2_zoo_validation_df.copy()

# Harmonisation
if "Condition" not in zoo_v2.columns:
    zoo_v2["Condition"] = "V2"

# Fusion
zoo_all = pd.concat(
    [
        zoo_old,
        zoo_v2
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Succès local :
# attaque réussie ET Linf <= 0.05
# ------------------------------------------------------------

zoo_all["Local_Success_005"] = (
    (zoo_all["Eligible"] == 1)
    &
    (zoo_all["Attack_Success"] == 1)
    &
    (zoo_all["Linf"] <= ZOO_LOCAL_EPS)
).astype(int)

# ------------------------------------------------------------
# Résumé
# ------------------------------------------------------------

zoo_local_summary = (
    zoo_all
    .groupby(
        ["Condition", "Model"]
    )
    .agg(
        Eligible=("Eligible", "sum"),
        Raw_Success=("Attack_Success", "sum"),
        Local_Success=("Local_Success_005", "sum")
    )
    .reset_index()
)

zoo_local_summary["Raw_ASR_%"] = (
    zoo_local_summary["Raw_Success"]
    /
    zoo_local_summary["Eligible"]
    *
    100
)

zoo_local_summary["Local_ASR_005_%"] = (
    zoo_local_summary["Local_Success"]
    /
    zoo_local_summary["Eligible"]
    *
    100
)

display(
    zoo_local_summary.round(2)
)

print("\n✅ Analyse terminée — aucune attaque relancée.")
print("ε local utilisé :", ZOO_LOCAL_EPS)

,Condition,Model,Eligible,Raw_Success,Local_Success,Raw_ASR_%,Local_ASR_005_%
0,Defended,Régression logistique,20,12,0,60.0,0.0
1,Defended,SVM linéaire,20,20,0,100.0,0.0
2,Original,Régression logistique,20,11,0,55.0,0.0
3,Original,SVM linéaire,20,20,3,100.0,15.0
4,V2,Régression logistique,20,11,0,55.0,0.0
5,V2,SVM linéaire,20,20,0,100.0,0.0



✅ Analyse terminée — aucune attaque relancée.
ε local utilisé : 0.05


In [88]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 10
# ZOO CONTRE LR V2 ET SVM V2 — VALIDATION
# VERSION CORRIGÉE
# ============================================================

import pandas as pd

V2_ZOO_VAL_FILE = (
    V2_VALIDATION_DIR /
    "v2_zoo_validation.csv"
)

# ------------------------------------------------------------
# Même panel Validation que V1
# ------------------------------------------------------------

V2_ZOO_VAL_POSITIONS = sorted(
    validation_zoo_df[
        "Source_Position"
    ]
    .astype(int)
    .unique()
    .tolist()
)

print(
    "Fraudes Validation utilisées :",
    len(V2_ZOO_VAL_POSITIONS)
)

assert (
    len(V2_ZOO_VAL_POSITIONS)
    == 20
), "On attend exactement 20 fraudes de Validation."


# ------------------------------------------------------------
# Lancement ZOO contre V2
# ------------------------------------------------------------

v2_zoo_val_rows = []

for model_name in [
    "Régression logistique",
    "SVM linéaire"
]:

    print("\n" + "=" * 70)
    print(
        "ZOO VALIDATION V2 →",
        model_name
    )
    print("=" * 70)

    model = models_v2[
        model_name
    ]

    for i, position in enumerate(
        V2_ZOO_VAL_POSITIONS,
        start=1
    ):

        # ----------------------------------------------------
        # run_zoo_eval_attack renvoie un tuple
        # On récupère automatiquement le dictionnaire résultat
        # ----------------------------------------------------

        output = run_zoo_eval_attack(
            model_name=model_name,
            model=model,
            source_position=position,
            X_source=X_val
        )

        if isinstance(
            output,
            tuple
        ):

            dict_items = [
                item
                for item in output
                if isinstance(
                    item,
                    dict
                )
            ]

            if len(dict_items) == 0:
                raise TypeError(
                    "run_zoo_eval_attack a renvoyé un tuple "
                    "sans dictionnaire de résultats."
                )

            result = dict_items[0]

        elif isinstance(
            output,
            dict
        ):

            result = output

        else:

            raise TypeError(
                "Type inattendu renvoyé par "
                "run_zoo_eval_attack : "
                f"{type(output)}"
            )

        # Copie pour ne pas modifier
        # l'objet original éventuellement retourné
        result = result.copy()

        result[
            "Condition"
        ] = "V2"

        v2_zoo_val_rows.append(
            result
        )

        print(
            f"{model_name} | "
            f"{i}/20",
            end="\r"
        )


# ============================================================
# DATAFRAME FINAL
# ============================================================

v2_zoo_validation_df = pd.DataFrame(
    v2_zoo_val_rows
)

print("\n")

print(
    "Nombre total de lignes :",
    len(v2_zoo_validation_df)
)

display(
    v2_zoo_validation_df.head()
)


# ------------------------------------------------------------
# Sauvegarde
# ------------------------------------------------------------

v2_zoo_validation_df.to_csv(
    V2_ZOO_VAL_FILE,
    index=False
)


# ============================================================
# RÉSUMÉ ASR
# ============================================================

v2_zoo_summary = (
    v2_zoo_validation_df
    .groupby("Model")
    .agg(
        Eligible=(
            "Eligible",
            "sum"
        ),
        Successful=(
            "Attack_Success",
            "sum"
        )
    )
    .reset_index()
)

v2_zoo_summary[
    "ASR_%"
] = (
    v2_zoo_summary[
        "Successful"
    ]
    /
    v2_zoo_summary[
        "Eligible"
    ]
    *
    100
)

print(
    "\n" + "=" * 70
)

print(
    "RÉSUMÉ ZOO — V2"
)

print(
    "=" * 70
)

display(
    v2_zoo_summary.round(2)
)

print(
    "\n✅ ZOO contre V2 terminé."
)

print(
    "🔒 Test toujours non utilisé."
)

Fraudes Validation utilisées : 20

ZOO VALIDATION V2 → Régression logistique
Régression logistique | 20/20
ZOO VALIDATION V2 → SVM linéaire
SVM linéaire | 20/20

Nombre total de lignes : 40


,Model,Source_Position,Eligible,Clean_Pred,Adv_Pred,Clean_Score,Adv_Score,Attack_Success,L2,Linf,Queries,Time_s,Condition
0,Régression logistique,544,1,1,0,0.999983,0.480930,1,0.324161,0.120856,33551,24.975102,V2
1,Régression logistique,666,1,1,1,1.000000,1.000000,0,0.000000,0.000000,11591,8.720738,V2
2,Régression logistique,1899,1,1,0,0.985469,0.460074,1,0.116219,0.067772,12811,9.926945,V2
3,Régression logistique,2897,1,1,0,0.999998,0.450786,1,0.396355,0.182729,28671,20.054294,V2
4,Régression logistique,4127,1,1,1,1.000000,1.000000,0,0.000000,0.000000,11591,8.092176,V2



RÉSUMÉ ZOO — V2


,Model,Eligible,Successful,ASR_%
0,Régression logistique,20,11,55.0
1,SVM linéaire,20,20,100.0



✅ ZOO contre V2 terminé.
🔒 Test toujours non utilisé.


In [86]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 9
# ÉVALUATION CLEAN SUR VALIDATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

import pandas as pd


rows = []

for model_name, model in models_v2.items():

    y_pred = model.predict(X_val)

    # Score pour AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_val)[:, 1]
    else:
        y_score = model.decision_function(X_val)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()

    rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "ROC_AUC": roc_auc_score(y_val, y_score),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })


v2_validation_clean = pd.DataFrame(rows)

display(
    v2_validation_clean.round(4)
)

print(
    "\n✅ Évaluation CLEAN V2 terminée."
)
print(
    "🔒 Aucun accès au Test."
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,TN,FP,FN,TP
0,Régression logistique,0.9981,0.4609,0.8310,0.5930,0.9681,42419,69,12,59
1,SVM linéaire,0.9991,0.6905,0.8169,0.7484,0.9727,42462,26,13,58



✅ Évaluation CLEAN V2 terminée.
🔒 Aucun accès au Test.


In [85]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 8
# RÉENTRAÎNEMENT LR / SVM DEPUIS ZÉRO
# ============================================================

from sklearn.base import clone
import joblib
import time
import pandas as pd
import numpy as np


models_v2 = {}

v2_training_rows = []


for model_name in V2_TARGET_MODELS:

    print("\n" + "=" * 75)
    print("ENTRAÎNEMENT V2 →", model_name)
    print("=" * 75)

    bank_v2 = (
        v2_bank_by_model[
            model_name
        ]
        .copy()
    )

    # --------------------------------------------------------
    # TRAIN SMOTE + banque adversariale V2
    # --------------------------------------------------------

    X_train_v2 = pd.concat(
        [
            X_train_aug[
                FEATURE_COLUMNS
            ].reset_index(drop=True),

            bank_v2[
                FEATURE_COLUMNS
            ].reset_index(drop=True)
        ],
        ignore_index=True
    )

    y_train_v2 = pd.concat(
        [
            pd.Series(
                y_train_aug
            ).reset_index(drop=True),

            pd.Series(
                np.ones(
                    len(bank_v2),
                    dtype=int
                )
            )
        ],
        ignore_index=True
    )

    # --------------------------------------------------------
    # Nouveau modèle vierge
    # mêmes hyperparamètres que V1
    # --------------------------------------------------------

    model_v2 = clone(
        models_defended[
            model_name
        ]
    )

    start = time.perf_counter()

    model_v2.fit(
        X_train_v2,
        y_train_v2
    )

    elapsed = (
        time.perf_counter()
        -
        start
    )

    models_v2[
        model_name
    ] = model_v2

    # --------------------------------------------------------
    # Sauvegarde
    # --------------------------------------------------------

    slug = (
        "logistic_regression"
        if model_name
        == "Régression logistique"
        else "linear_svm"
    )

    joblib.dump(
        model_v2,
        V2_MODEL_DIR /
        f"{slug}_v2.joblib"
    )

    v2_training_rows.append({
        "Model": model_name,
        "Train_SMOTE": len(
            X_train_aug
        ),
        "Adversarial_V2": len(
            bank_v2
        ),
        "Total_Train_V2": len(
            X_train_v2
        ),
        "Frauds_V2": int(
            y_train_v2.sum()
        ),
        "Training_Time_s": elapsed
    })

    print(
        f"✅ {model_name} V2 entraîné"
    )

    print(
        f"Train total : {len(X_train_v2)}"
    )

    print(
        f"Temps : {elapsed:.2f} s"
    )


v2_training_df = pd.DataFrame(
    v2_training_rows
)

print("\n" + "=" * 75)
print("RÉSUMÉ ENTRAÎNEMENT V2")
print("=" * 75)

display(
    v2_training_df.round(2)
)

print(
    "\n✅ LR V2 et SVM V2 sont prêts."
)

print(
    "🔒 TEST toujours non utilisé."
)


ENTRAÎNEMENT V2 → Régression logistique
✅ Régression logistique V2 entraîné
Train total : 218493
Temps : 0.56 s

ENTRAÎNEMENT V2 → SVM linéaire
✅ SVM linéaire V2 entraîné
Train total : 218762
Temps : 1.24 s

RÉSUMÉ ENTRAÎNEMENT V2


,Model,Train_SMOTE,Adversarial_V2,Total_Train_V2,Frauds_V2,Training_Time_s
0,Régression logistique,218105,388,218493,20216,0.56
1,SVM linéaire,218105,657,218762,20485,1.24



✅ LR V2 et SVM V2 sont prêts.
🔒 TEST toujours non utilisé.


In [84]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 7
# BANQUE V1 + NOUVELLES ATTAQUES ZOO
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

V2_TARGET_MODELS = [
    "Régression logistique",
    "SVM linéaire"
]


def convert_bank_to_df(bank):

    # Déjà un DataFrame
    if isinstance(bank, pd.DataFrame):

        df = bank.copy()

        if all(
            col in df.columns
            for col in FEATURE_COLUMNS
        ):
            return (
                df[FEATURE_COLUMNS]
                .reset_index(drop=True)
            )

        if df.shape[1] == len(FEATURE_COLUMNS):
            df.columns = FEATURE_COLUMNS
            return df.reset_index(drop=True)

    # ndarray
    if isinstance(bank, np.ndarray):

        arr = bank

        if arr.ndim == 1:
            arr = arr.reshape(1, -1)

        if arr.ndim == 3 and arr.shape[1] == 1:
            arr = arr[:, 0, :]

        return pd.DataFrame(
            arr,
            columns=FEATURE_COLUMNS
        )

    # liste d'arrays
    if isinstance(bank, list):

        arrays = []

        for item in bank:

            arr = np.asarray(item)

            arr = arr.reshape(
                -1,
                len(FEATURE_COLUMNS)
            )

            arrays.append(arr)

        arr = np.vstack(arrays)

        return pd.DataFrame(
            arr,
            columns=FEATURE_COLUMNS
        )

    raise TypeError(
        f"Type de banque non supporté : {type(bank)}"
    )


v2_bank_by_model = {}

summary_rows = []


for model_name in V2_TARGET_MODELS:

    print("\n" + "=" * 75)
    print(model_name)
    print("=" * 75)

    # --------------------------------------------------------
    # 1. Banque V1
    # --------------------------------------------------------

    old_bank = convert_bank_to_df(
        adversarial_bank_by_model[
            model_name
        ]
    )

    # --------------------------------------------------------
    # 2. Nouvelles attaques ZOO réussies contre V1
    # --------------------------------------------------------

    rows_success = v2_zoo_train_df[
        (v2_zoo_train_df["Model"] == model_name)
        &
        (v2_zoo_train_df["Attack_Success"] == 1)
        &
        (v2_zoo_train_df["Adv_File"].notna())
    ].copy()

    new_arrays = []

    for file_path in rows_success["Adv_File"]:

        arr = np.load(
            Path(file_path)
        )

        arr = np.asarray(
            arr
        ).reshape(
            -1,
            len(FEATURE_COLUMNS)
        )

        new_arrays.append(arr)

    if len(new_arrays) > 0:

        new_bank = pd.DataFrame(
            np.vstack(new_arrays),
            columns=FEATURE_COLUMNS
        )

    else:

        new_bank = pd.DataFrame(
            columns=FEATURE_COLUMNS
        )

    # --------------------------------------------------------
    # 3. Fusion V1 + nouvelles attaques
    # --------------------------------------------------------

    combined_bank = pd.concat(
        [
            old_bank,
            new_bank
        ],
        ignore_index=True
    )

    before_dedup = len(
        combined_bank
    )

    # Éviter les copies exactement identiques
    combined_bank = (
        combined_bank
        .drop_duplicates()
        .reset_index(drop=True)
    )

    after_dedup = len(
        combined_bank
    )

    v2_bank_by_model[
        model_name
    ] = combined_bank

    # Sauvegarde
    slug = (
        "logistic_regression"
        if model_name
        == "Régression logistique"
        else "linear_svm"
    )

    combined_bank.to_parquet(
        V2_BANK_DIR /
        f"{slug}_v2_bank.parquet",
        index=False
    )

    summary_rows.append({
        "Model": model_name,
        "V1_Bank": len(old_bank),
        "New_ZOO": len(new_bank),
        "Before_Dedup": before_dedup,
        "Final_V2_Bank": after_dedup
    })


v2_bank_summary = pd.DataFrame(
    summary_rows
)

display(
    v2_bank_summary
)

print(
    "\n✅ Banques V2 construites."
)


Régression logistique

SVM linéaire


,Model,V1_Bank,New_ZOO,Before_Dedup,Final_V2_Bank
0,Régression logistique,237,151,388,388
1,SVM linéaire,376,281,657,657



✅ Banques V2 construites.


In [83]:
# ============================================================
# V2 — ÉTAPE 6
# CONTENU DE LA BANQUE V1
# ============================================================

print("=" * 70)
print("BANQUES ADVERSARIALES V1")
print("=" * 70)

for model_name, bank in adversarial_bank_by_model.items():

    print("\n", model_name)

    try:
        print("Shape :", bank.shape)
    except:
        print("Type :", type(bank))
        print("Taille :", len(bank))

BANQUES ADVERSARIALES V1

 Régression logistique
Shape : (237, 30)

 SVM linéaire
Shape : (376, 30)

 Random Forest
Shape : (89, 30)

 XGBoost
Shape : (97, 30)

 LightGBM
Shape : (129, 30)


In [82]:
# ============================================================
# V2 — ÉTAPE 5
# RETROUVER LES BANQUES ADVERSARIALES V1
# ============================================================

print("=" * 70)
print("VARIABLES LIÉES AUX BANQUES / ADVERSARIAUX")
print("=" * 70)

for name in sorted(globals()):
    low = name.lower()

    if any(word in low for word in [
        "bank",
        "adversarial",
        "adv_train",
        "hsj_train",
        "zoo_train"
    ]):
        obj = globals()[name]

        try:
            shape = obj.shape
        except:
            shape = ""

        print(f"{name:40s} {type(obj).__name__:25s} {shape}")

VARIABLES LIÉES AUX BANQUES / ADVERSARIAUX
BANK_DIR                                 WindowsPath               
CLEAN_BANK_DIR                           WindowsPath               
HSJ_TRAIN_ADV_DIR                        WindowsPath               
HSJ_TRAIN_DIR                            WindowsPath               
HSJ_TRAIN_RESULTS_FILE                   WindowsPath               
V2_BANK_DIR                              WindowsPath               
ZOO_TRAIN_ADV_DIR                        WindowsPath               
ZOO_TRAIN_DIR                            WindowsPath               
ZOO_TRAIN_MODELS                         list                      
ZOO_TRAIN_RESULTS_FILE                   WindowsPath               
adversarial_bank_by_model                dict                      
adversarial_metadata                     DataFrame                 (928, 5)
adversarial_metadata_rows                list                      
adversarial_training_df                  DataFrame               

In [81]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 4
# RÉSUMÉ DES NOUVELLES ATTAQUES
# ============================================================

v2_zoo_train_df = pd.read_csv(
    V2_ZOO_RESULTS_FILE
)

v2_zoo_summary = (
    v2_zoo_train_df
    .groupby("Model")
    .agg(
        Sources=(
            "Source_Position",
            "count"
        ),
        Eligible=(
            "Eligible",
            "sum"
        ),
        Successful=(
            "Attack_Success",
            "sum"
        ),
        Errors=(
            "Error",
            lambda x:
                x.notna().sum()
        )
    )
    .reset_index()
)

v2_zoo_summary[
    "ASR_%"
] = (
    v2_zoo_summary[
        "Successful"
    ]
    /
    v2_zoo_summary[
        "Eligible"
    ]
    *
    100
)

display(
    v2_zoo_summary.round(2)
)

print(
    "\n✅ Résumé V2 disponible."
)

,Model,Sources,Eligible,Successful,Errors,ASR_%
0,Régression logistique,281,281,151,0,53.74
1,SVM linéaire,281,281,281,0,100.00



✅ Résumé V2 disponible.


In [80]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 3
# ZOO ADAPTATIF SUR LR/SVM V1 — TRAIN UNIQUEMENT
# ============================================================

import pandas as pd
import time


# ------------------------------------------------------------
# Exactement les mêmes fraudes TRAIN que V1
# ------------------------------------------------------------

V2_TRAIN_POSITIONS = sorted(
    zoo_train_df[
        "Source_Position"
    ]
    .astype(int)
    .unique()
    .tolist()
)

print(
    "Fraudes TRAIN utilisées :",
    len(V2_TRAIN_POSITIONS)
)

assert (
    len(V2_TRAIN_POSITIONS)
    == 281
), (
    "On attend exactement "
    "281 fraudes TRAIN communes."
)


# ------------------------------------------------------------
# Reprise automatique
# ------------------------------------------------------------

if V2_ZOO_RESULTS_FILE.exists():

    v2_zoo_results = (
        pd.read_csv(
            V2_ZOO_RESULTS_FILE
        )
        .to_dict(
            "records"
        )
    )

    print(
        "Reprise :",
        len(v2_zoo_results),
        "résultats déjà présents."
    )

else:

    v2_zoo_results = []


completed = {
    (
        row["Model"],
        int(
            row[
                "Source_Position"
            ]
        )
    )
    for row
    in v2_zoo_results
}


TOTAL = (
    len(V2_MODELS)
    *
    len(V2_TRAIN_POSITIONS)
)

print(
    "Attaques attendues :",
    TOTAL
)

print(
    "Déjà terminées :",
    len(completed)
)


# ------------------------------------------------------------
# ATTAQUES
# ------------------------------------------------------------

for model_name in V2_MODELS:

    print(
        "\n" + "=" * 70
    )

    print(
        "V2 ZOO →",
        model_name
    )

    print(
        "=" * 70
    )

    # IMPORTANT :
    # on attaque ici le modèle DEFENDED V1
    model_v1 = (
        models_defended[
            model_name
        ]
    )

    for i, position in enumerate(
        V2_TRAIN_POSITIONS,
        start=1
    ):

        key = (
            model_name,
            int(position)
        )

        if key in completed:
            continue

        result = (
            run_zoo_v2_train_attack(
                model_name=model_name,
                model=model_v1,
                source_position=position
            )
        )

        v2_zoo_results.append(
            result
        )

        completed.add(key)

        # Sauvegarde après CHAQUE attaque
        pd.DataFrame(
            v2_zoo_results
        ).to_csv(
            V2_ZOO_RESULTS_FILE,
            index=False
        )

        if (
            i == 1
            or
            i % 10 == 0
        ):

            print(
                f"{model_name} | "
                f"{i}/{len(V2_TRAIN_POSITIONS)}"
            )


print(
    "\n✅ ZOO V2 TRAIN terminé."
)

Fraudes TRAIN utilisées : 281
Attaques attendues : 562
Déjà terminées : 0

V2 ZOO → Régression logistique
Régression logistique | 1/281
Régression logistique | 10/281
Régression logistique | 20/281
Régression logistique | 30/281
Régression logistique | 40/281
Régression logistique | 50/281
Régression logistique | 60/281
Régression logistique | 70/281
Régression logistique | 80/281
Régression logistique | 90/281
Régression logistique | 100/281
Régression logistique | 110/281
Régression logistique | 120/281
Régression logistique | 130/281
Régression logistique | 140/281
Régression logistique | 150/281
Régression logistique | 160/281
Régression logistique | 170/281
Régression logistique | 180/281
Régression logistique | 190/281
Régression logistique | 200/281
Régression logistique | 210/281
Régression logistique | 220/281
Régression logistique | 230/281
Régression logistique | 240/281
Régression logistique | 250/281
Régression logistique | 260/281
Régression logistique | 270/281
Régressio

In [79]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 2
# FONCTION ZOO ADAPTATIVE CONTRE LES MODÈLES V1
# ============================================================

import numpy as np
import pandas as pd
import time
from pathlib import Path


V2_ZOO_DIR = V2_ATTACK_DIR / "zoo_against_v1"
V2_ZOO_ADV_DIR = V2_ZOO_DIR / "adversarial_samples"

V2_ZOO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

V2_ZOO_ADV_DIR.mkdir(
    parents=True,
    exist_ok=True
)

V2_ZOO_RESULTS_FILE = (
    V2_ZOO_DIR /
    "zoo_v2_train_results.csv"
)


def _unpack_blackbox(model):

    built = build_blackbox_classifier(
        model
    )

    # Certains wrappers retournent
    # (classifier, query_counter)
    if isinstance(built, tuple):

        classifier = built[0]

        predictor = (
            built[1]
            if len(built) > 1
            else None
        )

    else:

        classifier = built
        predictor = None

    return classifier, predictor


def _get_query_count(predictor):

    if predictor is None:
        return np.nan

    for attr in [
        "query_count",
        "queries",
        "n_queries",
        "total_queries"
    ]:

        if hasattr(predictor, attr):

            try:
                return int(
                    getattr(
                        predictor,
                        attr
                    )
                )
            except Exception:
                pass

    return np.nan


def run_zoo_v2_train_attack(
    model_name,
    model,
    source_position
):

    source_position = int(
        source_position
    )

    # --------------------------------------------------------
    # Transaction réelle TRAIN
    # --------------------------------------------------------

    X_train_positional = (
        X_train
        .reset_index(drop=True)
    )

    x_df = (
        X_train_positional
        .iloc[
            [source_position]
        ]
        .copy()
    )

    x = (
        x_df[
            FEATURE_COLUMNS
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    clean_pred = int(
        np.asarray(
            model.predict(
                x_df
            )
        ).ravel()[0]
    )

    # Une attaque d'évasion 1→0
    # n'est pertinente que si la fraude
    # est encore détectée avant attaque.
    if clean_pred != 1:

        return {
            "Model": model_name,
            "Source_Position": source_position,
            "Eligible": 0,
            "Clean_Pred": clean_pred,
            "Adv_Pred": np.nan,
            "Attack_Success": 0,
            "L2": np.nan,
            "Linf": np.nan,
            "Queries": 0,
            "Time_s": 0.0,
            "Adv_File": None,
            "Error": None
        }

    classifier, predictor = (
        _unpack_blackbox(
            model
        )
    )

    attack = ZooAttack(
        classifier=classifier,
        **ZOO_CONFIG_TRAIN
    )

    # Target = classe normale 0
    y_target = np.array(
        [[1.0, 0.0]],
        dtype=np.float32
    )

    start = time.perf_counter()

    try:

        x_adv = attack.generate(
            x=x,
            y=y_target
        )

        elapsed = (
            time.perf_counter()
            -
            start
        )

        x_adv = np.asarray(
            x_adv,
            dtype=np.float32
        )

        x_adv_df = pd.DataFrame(
            x_adv,
            columns=FEATURE_COLUMNS
        )

        adv_pred = int(
            np.asarray(
                model.predict(
                    x_adv_df
                )
            ).ravel()[0]
        )

        success = int(
            clean_pred == 1
            and
            adv_pred == 0
        )

        delta = (
            x_adv.astype(float)
            -
            x.astype(float)
        )

        l2 = float(
            np.linalg.norm(
                delta.ravel(),
                ord=2
            )
        )

        linf = float(
            np.max(
                np.abs(delta)
            )
        )

        adv_file = None

        # ----------------------------------------------------
        # Sauvegarder uniquement les attaques réussies
        # ----------------------------------------------------

        if success:

            slug = (
                "logistic_regression"
                if model_name
                == "Régression logistique"
                else "linear_svm"
            )

            adv_file = (
                V2_ZOO_ADV_DIR
                /
                f"{slug}_v2_train_{source_position}.npy"
            )

            np.save(
                adv_file,
                x_adv
            )

        return {
            "Model": model_name,
            "Source_Position": source_position,
            "Eligible": 1,
            "Clean_Pred": clean_pred,
            "Adv_Pred": adv_pred,
            "Attack_Success": success,
            "L2": l2,
            "Linf": linf,
            "Queries":
                _get_query_count(
                    predictor
                ),
            "Time_s": elapsed,
            "Adv_File":
                str(adv_file)
                if adv_file
                is not None
                else None,
            "Error": None
        }

    except Exception as e:

        return {
            "Model": model_name,
            "Source_Position": source_position,
            "Eligible": 1,
            "Clean_Pred": clean_pred,
            "Adv_Pred": np.nan,
            "Attack_Success": 0,
            "L2": np.nan,
            "Linf": np.nan,
            "Queries":
                _get_query_count(
                    predictor
                ),
            "Time_s":
                time.perf_counter()
                -
                start,
            "Adv_File": None,
            "Error": repr(e)
        }


print(
    "✅ Fonction ZOO V2 prête."
)

print(
    "Aucun résultat V1 ne sera écrasé."
)

✅ Fonction ZOO V2 prête.
Aucun résultat V1 ne sera écrasé.


In [78]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 1
# VERROUILLAGE DU PROTOCOLE ZOO
# ============================================================

print("=" * 80)
print("DÉFENSE V2 — PROTOCOLE")
print("=" * 80)

V2_MODELS = [
    "Régression logistique",
    "SVM linéaire"
]

# ------------------------------------------------------------
# La configuration doit déjà exister dans Notebook 11
# ------------------------------------------------------------

if "ZOO_CONFIG_TRAIN" not in globals():

    possible_configs = [
        name
        for name, value in globals().items()
        if (
            "ZOO" in name.upper()
            and
            "CONFIG" in name.upper()
            and
            isinstance(value, dict)
        )
    ]

    raise RuntimeError(
        "ZOO_CONFIG_TRAIN introuvable.\n"
        f"Configurations ZOO trouvées : {possible_configs}"
    )

print("\nModèles V2 :")
for name in V2_MODELS:
    print(" •", name)

print("\nConfiguration ZOO réutilisée :")
for key, value in ZOO_CONFIG_TRAIN.items():
    print(f"  {key:25s} = {value}")

print("\n✅ Aucun paramètre ZOO n'est modifié.")
print("✅ Validation non utilisée pour l'entraînement.")
print("✅ Test toujours fermé.")

DÉFENSE V2 — PROTOCOLE

Modèles V2 :
 • Régression logistique
 • SVM linéaire

Configuration ZOO réutilisée :
  confidence                = 0.5
  targeted                  = True
  learning_rate             = 0.001
  max_iter                  = 200
  binary_search_steps       = 10
  initial_const             = 0.001
  abort_early               = True
  nb_parallel               = 30
  batch_size                = 1
  verbose                   = False

✅ Aucun paramètre ZOO n'est modifié.
✅ Validation non utilisée pour l'entraînement.
✅ Test toujours fermé.


In [77]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 3
# SIGNATURES EXACTES DES FONCTIONS À RÉUTILISER
# ============================================================

import inspect

functions_to_check = [
    run_zoo_train_attack,
    run_zoo_eval_attack,
    build_blackbox_classifier
]

for func in functions_to_check:

    print("=" * 80)
    print(func.__name__)
    print(inspect.signature(func))

run_zoo_train_attack
(model_name, model, source_position)
run_zoo_eval_attack
(model_name, model, source_position, X_source)
build_blackbox_classifier
(model)


In [76]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 2
# INVENTAIRE DES FONCTIONS / CONFIGS À RÉUTILISER
# ============================================================

import inspect

print("=" * 80)
print("FONCTIONS D'ATTAQUE DISPONIBLES")
print("=" * 80)

keywords = [
    "zoo",
    "hsj",
    "attack",
    "blackbox",
    "classifier",
    "score",
    "predictor"
]

found_functions = []

for name, obj in list(globals().items()):

    if callable(obj):

        lname = name.lower()

        if any(
            keyword in lname
            for keyword in keywords
        ):

            found_functions.append(name)

for name in sorted(found_functions):
    print(name)


print("\n" + "=" * 80)
print("CONFIGURATIONS IMPORTANTES")
print("=" * 80)

for name in sorted(globals()):

    upper = name.upper()

    if any(
        key in upper
        for key in [
            "ZOO",
            "HSJ",
            "EPS",
            "CONFIG"
        ]
    ):

        value = globals()[name]

        if isinstance(
            value,
            (
                int,
                float,
                str,
                bool,
                dict,
                list,
                tuple
            )
        ):

            text = repr(value)

            if len(text) < 1500:
                print(
                    f"\n{name} = {text}"
                )

FONCTIONS D'ATTAQUE DISPONIBLES
BlackBoxClassifier
QueryCountingPredictor
ZooAttack
accuracy_score
audit_clean_scores
build_blackbox_classifier
f1_score
find_hsj_eval_seed
find_hsj_train_seed
get_fraud_score
get_score_for_auc
positive_score
precision_score
recall_score
roc_auc_score
run_hsj_eval_attack
run_hsj_train_attack
run_zoo_eval_attack
run_zoo_train_attack
summarize_validation_attack

CONFIGURATIONS IMPORTANTES

HSJ_CONFIG = {'targeted': True, 'norm': inf, 'max_iter': 50, 'max_eval': 10000, 'init_eval': 100, 'init_size': 100, 'batch_size': 64, 'verbose': False}

HSJ_CONFIG_TRAIN = {'targeted': True, 'norm': inf, 'max_iter': 50, 'max_eval': 10000, 'init_eval': 100, 'init_size': 100, 'batch_size': 64, 'verbose': False}

HSJ_IMMUTABLE_FEATURES = ['Time', 'Amount']

HSJ_LOCAL_EPS = 0.05

HSJ_MUTABLE_FEATURES = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', '

In [75]:
# ============================================================
# DÉFENSE V2 — ÉTAPE 1
# FIGER V1 ET CRÉER UN ESPACE V2 SÉPARÉ
# ============================================================

from pathlib import Path
import pandas as pd
import json

V2_DIR = NB11_DIR / "defence_v2"
V2_DIR.mkdir(parents=True, exist_ok=True)

V2_ATTACK_DIR = V2_DIR / "train_attacks"
V2_BANK_DIR = V2_DIR / "adversarial_banks"
V2_MODEL_DIR = V2_DIR / "models"
V2_VALIDATION_DIR = V2_DIR / "validation"

for folder in [
    V2_ATTACK_DIR,
    V2_BANK_DIR,
    V2_MODEL_DIR,
    V2_VALIDATION_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# Sauvegarde officielle des résultats V1
validation_decision_df.to_csv(
    V2_DIR /
    "V1_validation_decision_frozen.csv",
    index=False
)

val_hsj_summary.to_csv(
    V2_DIR /
    "V1_HSJ_validation_frozen.csv",
    index=False
)

val_zoo_summary.to_csv(
    V2_DIR /
    "V1_ZOO_validation_frozen.csv",
    index=False
)

print("=" * 75)
print("DÉFENSE V2 — INITIALISATION")
print("=" * 75)

print("✅ Résultats V1 figés.")
print("✅ Dossier V2 créé.")
print("✅ Aucun résultat V1 ne sera écrasé.")
print()
print("V2_DIR :", V2_DIR)

DÉFENSE V2 — INITIALISATION
✅ Résultats V1 figés.
✅ Dossier V2 créé.
✅ Aucun résultat V1 ne sera écrasé.

V2_DIR : c:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training\defence_v2


In [73]:
# ============================================================
# ÉTAPE 41I — AUDIT DÉCISIF DU MAPPING
# Original → models_original
# Defended → models_defended
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("AUDIT DU MAPPING RÉEL DES MODÈLES DANS LA VALIDATION")
print("=" * 90)

# ------------------------------------------------------------
# Sécuriser l'indexation positionnelle
# ------------------------------------------------------------

X_val_audit = X_val.reset_index(drop=True)


def audit_model_mapping(detail_df, attack_name):

    rows = []

    for _, row in detail_df.iterrows():

        condition = row["Condition"]
        model_name = row["Model"]

        position = int(
            row["Source_Position"]
        )

        # ----------------------------------------------------
        # Choisir EXPLICITEMENT la bonne famille
        # ----------------------------------------------------

        if condition == "Original":

            model = models_original[
                model_name
            ]

        elif condition == "Defended":

            model = models_defended[
                model_name
            ]

        else:

            raise ValueError(
                f"Condition inconnue : {condition}"
            )

        # ----------------------------------------------------
        # Transaction Validation correspondante
        # ----------------------------------------------------

        x = X_val_audit.iloc[
            [position]
        ]

        # ----------------------------------------------------
        # Recalcul prédiction clean
        # ----------------------------------------------------

        predicted = int(
            np.asarray(
                model.predict(x)
            ).ravel()[0]
        )

        stored = int(
            row["Clean_Pred"]
        )

        rows.append({
            "Attack": attack_name,
            "Condition": condition,
            "Model": model_name,
            "Source_Position": position,
            "Stored_Clean_Pred": stored,
            "Recomputed_Clean_Pred": predicted,
            "Match": stored == predicted
        })

    audit_df = pd.DataFrame(
        rows
    )

    print(
        f"\n{attack_name}"
    )

    summary = (
        audit_df
        .groupby(
            ["Condition", "Model"]
        )
        .agg(
            N=("Match", "size"),
            Matches=("Match", "sum")
        )
        .reset_index()
    )

    summary[
        "Match_%"
    ] = (
        summary["Matches"]
        /
        summary["N"]
        *
        100
    )

    display(summary)

    mismatches = audit_df[
        ~audit_df["Match"]
    ]

    print(
        "Nombre total de désaccords :",
        len(mismatches)
    )

    if len(mismatches) > 0:

        print(
            "\n❌ Des incohérences ont été détectées :"
        )

        display(
            mismatches.head(20)
        )

    else:

        print(
            f"✅ {attack_name} : toutes les "
            "prédictions enregistrées correspondent "
            "à la bonne famille de modèles."
        )

    return audit_df


audit_mapping_hsj = audit_model_mapping(
    validation_hsj_df,
    "HSJ"
)

print("\n" + "-" * 90)

audit_mapping_zoo = audit_model_mapping(
    validation_zoo_df,
    "ZOO"
)

AUDIT DU MAPPING RÉEL DES MODÈLES DANS LA VALIDATION

HSJ


,Condition,Model,N,Matches,Match_%
0,Defended,LightGBM,20,20,100.0
1,Defended,Random Forest,20,20,100.0
2,Defended,Régression logistique,20,20,100.0
3,Defended,SVM linéaire,20,20,100.0
4,Defended,XGBoost,20,20,100.0
5,Original,LightGBM,20,20,100.0
6,Original,Random Forest,20,20,100.0
7,Original,Régression logistique,20,20,100.0
8,Original,SVM linéaire,20,20,100.0
9,Original,XGBoost,20,20,100.0


Nombre total de désaccords : 0
✅ HSJ : toutes les prédictions enregistrées correspondent à la bonne famille de modèles.

------------------------------------------------------------------------------------------

ZOO


,Condition,Model,N,Matches,Match_%
0,Defended,Régression logistique,20,20,100.0
1,Defended,SVM linéaire,20,20,100.0
2,Original,Régression logistique,20,20,100.0
3,Original,SVM linéaire,20,20,100.0


Nombre total de désaccords : 0
✅ ZOO : toutes les prédictions enregistrées correspondent à la bonne famille de modèles.


In [72]:
# ============================================================
# ÉTAPE 41H — VÉRIFIER LE MAPPING ORIGINAL / DEFENDED
# ============================================================

print("=" * 85)
print("AUDIT DU MAPPING DES MODÈLES")
print("=" * 85)

for model_name in models_original.keys():

    original_model = models_original[model_name]
    defended_model = models_defended[model_name]

    print(
        f"\n{model_name}"
    )

    print(
        "Original id :",
        id(original_model)
    )

    print(
        "Defended id :",
        id(defended_model)
    )

    print(
        "Même objet ? :",
        original_model is defended_model
    )

    assert (
        original_model
        is not
        defended_model
    )

print(
    "\n✅ Tous les modèles Original et Defended "
    "sont bien distincts."
)

AUDIT DU MAPPING DES MODÈLES

Régression logistique
Original id : 1971630410784
Defended id : 1975045699728
Même objet ? : False

SVM linéaire
Original id : 1971629452192
Defended id : 1975045686288
Même objet ? : False

Random Forest
Original id : 1971629454880
Defended id : 1975045696208
Même objet ? : False

XGBoost
Original id : 1971630413808
Defended id : 1975045696848
Même objet ? : False

LightGBM
Original id : 1974668745024
Defended id : 1975045701328
Même objet ? : False

✅ Tous les modèles Original et Defended sont bien distincts.


In [71]:
# ============================================================
# 41G — AUDIT DES DICTIONNAIRES DE MODÈLES
# ============================================================

print("=" * 80)
print("DICTIONNAIRES DE MODÈLES DÉTECTÉS")
print("=" * 80)

model_dicts = {}

for name, obj in list(globals().items()):

    if not isinstance(obj, dict):
        continue

    if len(obj) == 0:
        continue

    model_like = [
        value
        for value in obj.values()
        if hasattr(value, "predict")
    ]

    if len(model_like) >= 2:

        model_dicts[name] = obj

        print(
            "\nVariable :",
            name
        )

        for model_name, model in obj.items():

            if hasattr(model, "predict"):

                print(
                    f"  {model_name:25s}",
                    "→",
                    type(model).__name__,
                    "| id =",
                    id(model)
                )

DICTIONNAIRES DE MODÈLES DÉTECTÉS

Variable : models_original
  Régression logistique     → LogisticRegression | id = 1971630410784
  SVM linéaire              → LinearSVC | id = 1971629452192
  Random Forest             → RandomForestClassifier | id = 1971629454880
  XGBoost                   → XGBClassifier | id = 1971630413808
  LightGBM                  → LGBMClassifier | id = 1974668745024

Variable : models_defended
  Régression logistique     → LogisticRegression | id = 1975045699728
  SVM linéaire              → LinearSVC | id = 1975045686288
  Random Forest             → RandomForestClassifier | id = 1975045696208
  XGBoost                   → XGBClassifier | id = 1975045696848
  LightGBM                  → LGBMClassifier | id = 1975045701328

Variable : model_dict
  Régression logistique     → LogisticRegression | id = 1975045699728
  SVM linéaire              → LinearSVC | id = 1975045686288
  Random Forest             → RandomForestClassifier | id = 1975045696208
  XGBoost 

In [70]:
# ============================================================
# 41F — AUDIT DES POSITIONS DE VALIDATION
# ============================================================

position_variable_candidates = [
    "HSJ_VAL20_POSITIONS",
    "ZOO_VAL20_POSITIONS",
    "VAL20_POSITIONS",
    "validation_positions",
    "selected_positions"
]

print("=" * 80)
print("AUDIT DES PANELS DE VALIDATION")
print("=" * 80)

for var_name in position_variable_candidates:

    if var_name not in globals():
        continue

    positions = np.asarray(
        globals()[var_name]
    ).astype(int)

    # uniquement si positions compatibles avec X_val
    if (
        len(positions) > 0
        and
        positions.min() >= 0
        and
        positions.max() < len(y_val)
    ):

        labels = (
            y_val.iloc[positions]
            if hasattr(y_val, "iloc")
            else y_val[positions]
        )

        print(
            "\n",
            var_name,
            "→",
            len(positions),
            "positions"
        )

        print(
            "Labels fraude :",
            int((np.asarray(labels) == 1).sum()),
            "/",
            len(labels)
        )

        assert np.all(
            np.asarray(labels) == 1
        )

print(
    "\n✅ Audit labels terminé."
)

AUDIT DES PANELS DE VALIDATION

✅ Audit labels terminé.


In [69]:
# ============================================================
# 41E — AUDIT DU PANEL FIXE ORIGINAL VS DEFENDED
# ============================================================

print("=" * 80)
print("AUDIT DES TRANSACTIONS UTILISÉES")
print("=" * 80)

detailed_found = False

transaction_column_candidates = [
    "Transaction",
    "Position",
    "Transaction_Position",
    "Source_Position"
]

for name, df in validation_df_candidates.items():

    if (
        "Condition" not in df.columns
        or
        "Model" not in df.columns
    ):
        continue

    transaction_col = next(
        (
            c
            for c in transaction_column_candidates
            if c in df.columns
        ),
        None
    )

    if transaction_col is None:
        continue

    detailed_found = True

    print(
        f"\nTABLEAU DÉTAILLÉ : {name}"
    )

    for model in df["Model"].dropna().unique():

        model_df = df[
            df["Model"] == model
        ]

        original_set = set(
            model_df.loc[
                model_df["Condition"] == "Original",
                transaction_col
            ].astype(int)
        )

        defended_set = set(
            model_df.loc[
                model_df["Condition"] == "Defended",
                transaction_col
            ].astype(int)
        )

        if (
            len(original_set) == 0
            or
            len(defended_set) == 0
        ):
            continue

        identical = (
            original_set
            ==
            defended_set
        )

        print(
            f"{model:25s}",
            "| Original:",
            len(original_set),
            "| Defended:",
            len(defended_set),
            "| Même panel:",
            identical
        )

        assert identical, (
            f"Panel différent pour {model} "
            f"dans {name}"
        )

if not detailed_found:

    print(
        "⚠️ Aucun tableau transactionnel détaillé "
        "détecté automatiquement."
    )

else:

    print(
        "\n✅ Original et Defended utilisent "
        "le même panel de transactions."
    )

AUDIT DES TRANSACTIONS UTILISÉES

TABLEAU DÉTAILLÉ : validation_hsj_df
Régression logistique     | Original: 20 | Defended: 20 | Même panel: True
SVM linéaire              | Original: 20 | Defended: 20 | Même panel: True
Random Forest             | Original: 20 | Defended: 20 | Même panel: True
XGBoost                   | Original: 20 | Defended: 20 | Même panel: True
LightGBM                  | Original: 20 | Defended: 20 | Même panel: True

TABLEAU DÉTAILLÉ : validation_zoo_df
Régression logistique     | Original: 20 | Defended: 20 | Même panel: True
SVM linéaire              | Original: 20 | Defended: 20 | Même panel: True

✅ Original et Defended utilisent le même panel de transactions.


In [68]:
# ============================================================
# 41D — AUDIT MATHÉMATIQUE DE L'ASR
# ============================================================

print("=" * 80)
print("AUDIT DES CALCULS ASR")
print("=" * 80)

found_summary = False

for name, df in validation_df_candidates.items():

    required = {
        "Eligible",
        "Successful",
        "ASR_%"
    }

    if required.issubset(df.columns):

        found_summary = True

        tmp = df.copy()

        tmp["ASR_Recalculated"] = (
            tmp["Successful"]
            /
            tmp["Eligible"]
            *
            100
        )

        tmp["ASR_Difference"] = (
            tmp["ASR_%"]
            -
            tmp["ASR_Recalculated"]
        ).abs()

        print(
            f"\n--- {name} ---"
        )

        display(
            tmp[
                [
                    c for c in [
                        "Condition",
                        "Model",
                        "Eligible",
                        "Successful",
                        "ASR_%",
                        "ASR_Recalculated",
                        "ASR_Difference"
                    ]
                    if c in tmp.columns
                ]
            ].round(6)
        )

        max_error = (
            tmp["ASR_Difference"]
            .max()
        )

        print(
            "Erreur maximale :",
            max_error
        )

        assert max_error < 1e-8

if not found_summary:

    print(
        "⚠️ Aucun tableau résumé automatiquement détecté."
    )

else:

    print(
        "\n✅ Calcul des ASR mathématiquement correct."
    )

AUDIT DES CALCULS ASR

--- val_hsj_summary ---


,Condition,Model,Eligible,Successful,ASR_%,ASR_Recalculated,ASR_Difference
0,Defended,LightGBM,20,3,15.0,15.0,0.0
1,Defended,Random Forest,20,3,15.0,15.0,0.0
2,Defended,Régression logistique,20,4,20.0,20.0,0.0
3,Defended,SVM linéaire,20,5,25.0,25.0,0.0
4,Defended,XGBoost,20,5,25.0,25.0,0.0
5,Original,LightGBM,20,8,40.0,40.0,0.0
6,Original,Random Forest,20,5,25.0,25.0,0.0
7,Original,Régression logistique,20,5,25.0,25.0,0.0
8,Original,SVM linéaire,20,6,30.0,30.0,0.0
9,Original,XGBoost,20,7,35.0,35.0,0.0


Erreur maximale : 0.0

--- val_zoo_summary ---


,Condition,Model,Eligible,Successful,ASR_%,ASR_Recalculated,ASR_Difference
0,Defended,Régression logistique,20,12,60.0,60.0,0.0
1,Defended,SVM linéaire,20,20,100.0,100.0,0.0
2,Original,Régression logistique,20,11,55.0,55.0,0.0
3,Original,SVM linéaire,20,20,100.0,100.0,0.0


Erreur maximale : 7.105427357601002e-15

✅ Calcul des ASR mathématiquement correct.


In [67]:
# ============================================================
# 41C — INVENTAIRE DES DATAFRAMES DE VALIDATION
# ============================================================

print("=" * 80)
print("DATAFRAMES SUSCEPTIBLES DE CONTENIR LA VALIDATION")
print("=" * 80)

validation_df_candidates = {}

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        cols = set(
            str(c)
            for c in obj.columns
        )

        interesting = (
            "Model" in cols
            or
            "Condition" in cols
            or
            "ASR_%" in cols
            or
            "Transaction" in cols
            or
            "Local_Success" in cols
            or
            "Attack_Success" in cols
        )

        if interesting:

            validation_df_candidates[name] = obj

            print(
                "\nVariable :", name
            )

            print(
                "Shape    :", obj.shape
            )

            print(
                "Colonnes :",
                obj.columns.tolist()
            )

DATAFRAMES SUSCEPTIBLES DE CONTENIR LA VALIDATION

Variable : df_tmp
Shape    : (5, 8)
Colonnes : ['Model', 'ASR_HSJ', 'ASR_ZOO', 'Recall_Clean', 'Recall_HSJ', 'Recall_ZOO', 'Worst_Case_ASR', 'Worst_Case_Recall']

Variable : NB10_ROBUSTNESS_BASELINE
Shape    : (5, 3)
Colonnes : ['Model', 'ZOO_ASR_Before', 'HSJ_Local_ASR_Before']

Variable : asr_before
Shape    : (10, 7)
Colonnes : ['Attack', 'Model', 'Attacks', 'Successful_Attacks', 'ASR', 'CI95_Low', 'CI95_High']

Variable : metrics_before
Shape    : (15, 11)
Colonnes : ['Model', 'Condition', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'TN', 'FP', 'FN', 'TP']

Variable : perturb_before
Shape    : (10, 8)
Colonnes : ['Attack', 'Model', 'Mean_L2_All', 'Mean_Linf_All', 'Max_Linf_All', 'Mean_L2_Success', 'Mean_Linf_Success', 'Max_Linf_Success']

Variable : cost_before
Shape    : (10, 8)
Colonnes : ['Attack', 'Model', 'Mean_Queries', 'Median_Queries', 'Mean_Time_s', 'Median_Time_s', 'Total_Time_s', 'Total_Time_min']

Variable : rob

In [66]:
# ============================================================
# 41B — AUDIT DES TROIS ENSEMBLES
# ============================================================

required_objects = [
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_test",
    "y_test"
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Objets absents : {missing}"
    )

print("=" * 80)
print("AUDIT DES SPLITS")
print("=" * 80)

for split_name, X, y in [
    ("TRAIN", X_train, y_train),
    ("VALIDATION", X_val, y_val),
    ("TEST", X_test, y_test)
]:

    print("\n", split_name)

    print(
        "X :", X.shape,
        "| y :", y.shape
    )

    print(
        "Normales :",
        int((y == 0).sum())
    )

    print(
        "Fraudes  :",
        int((y == 1).sum())
    )

    assert len(X) == len(y)

# Colonnes identiques
assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

# Objets réellement distincts
assert X_train is not X_val
assert X_train is not X_test
assert X_val is not X_test

print(
    "\n✅ TRAIN / VALIDATION / TEST "
    "sont structurellement cohérents."
)

AUDIT DES SPLITS

 TRAIN
X : (198608, 30) | y : (198608,)
Normales : 198277
Fraudes  : 331

 VALIDATION
X : (42559, 30) | y : (42559,)
Normales : 42488
Fraudes  : 71

 TEST
X : (42559, 30) | y : (42559,)
Normales : 42488
Fraudes  : 71

✅ TRAIN / VALIDATION / TEST sont structurellement cohérents.


In [65]:
# ============================================================
# ÉTAPE 41 — AUDIT V1
# 41A — SNAPSHOT OFFICIEL AVANT TOUTE MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import sys
import platform
import importlib.metadata as metadata

# ------------------------------------------------------------
# Dossier audit
# ------------------------------------------------------------

if "NB11_DIR" in globals():
    AUDIT_DIR = NB11_DIR / "audit_v1"
else:
    AUDIT_DIR = Path.cwd() / "notebook11_adversarial_training" / "audit_v1"

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 80)
print("AUDIT OFFICIEL — DÉFENSE V1")
print("=" * 80)

print("\nPython :", sys.version.split()[0])
print("OS     :", platform.platform())

# Versions importantes
for package in [
    "numpy",
    "pandas",
    "scikit-learn",
    "adversarial-robustness-toolbox"
]:
    try:
        print(
            f"{package:35s} :",
            metadata.version(package)
        )
    except Exception:
        print(
            f"{package:35s} : non détecté"
        )

# ------------------------------------------------------------
# Sauvegarde de la décision actuelle
# ------------------------------------------------------------

if "validation_decision_df" not in globals():
    raise RuntimeError(
        "validation_decision_df introuvable."
    )

validation_decision_df.to_csv(
    AUDIT_DIR /
    "validation_decision_V1_before_audit.csv",
    index=False
)

print(
    "\n✅ Tableau V1 sauvegardé sans modification."
)
print("Dossier :", AUDIT_DIR)

AUDIT OFFICIEL — DÉFENSE V1

Python : 3.13.14
OS     : Windows-11-10.0.26200-SP0
numpy                               : 2.5.2
pandas                              : 3.0.5
scikit-learn                        : 1.9.0
adversarial-robustness-toolbox      : 1.20.1

✅ Tableau V1 sauvegardé sans modification.
Dossier : c:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training\audit_v1


In [64]:
# ============================================================
# DIAGNOSTIC — QUEL MODÈLE NE PASSE PAS LA VALIDATION ?
# ============================================================

print("Colonnes disponibles :")
print(validation_decision_df.columns.tolist())

print("\nDécisions complètes :")
display(validation_decision_df)

print("\n❌ Défenses qui ne passent pas :")

failed_defences = validation_decision_df[
    ~validation_decision_df["PASS"].astype(bool)
].copy()

display(failed_defences)

print(
    "\nNombre de défenses refusées :",
    len(failed_defences)
)

Colonnes disponibles :
['Model', 'Clean_Gate', 'HSJ_Original', 'HSJ_Defended', 'ZOO_Original', 'ZOO_Defended', 'HSJ_Not_Worse', 'ZOO_Not_Worse', 'Robust_Gain', 'PASS']

Décisions complètes :


,Model,Clean_Gate,HSJ_Original,HSJ_Defended,ZOO_Original,ZOO_Defended,HSJ_Not_Worse,ZOO_Not_Worse,Robust_Gain,PASS
0,Régression logistique,True,25.0,20.0,55.0,60.0,True,False,True,False
1,SVM linéaire,True,30.0,25.0,100.0,100.0,True,True,True,True
2,Random Forest,True,25.0,15.0,NaN,NaN,True,True,True,True
3,XGBoost,True,35.0,25.0,NaN,NaN,True,True,True,True
4,LightGBM,True,40.0,15.0,NaN,NaN,True,True,True,True



❌ Défenses qui ne passent pas :


,Model,Clean_Gate,HSJ_Original,HSJ_Defended,ZOO_Original,ZOO_Defended,HSJ_Not_Worse,ZOO_Not_Worse,Robust_Gain,PASS
0,Régression logistique,True,25.0,20.0,55.0,60.0,True,False,True,False



Nombre de défenses refusées : 1


In [62]:
# ============================================================
# ÉTAPE 41B — GATE DE VALIDATION
# ============================================================

validation_decision_rows = []


for model_name in MODEL_ORDER:

    clean_row = (
        validation_clean_delta_df[
            validation_clean_delta_df[
                "Model"
            ]
            ==
            model_name
        ]
        .iloc[0]
    )

    hsj_original = (
        val_hsj_summary[
            (
                val_hsj_summary["Model"]
                ==
                model_name
            )
            &
            (
                val_hsj_summary["Condition"]
                ==
                "Original"
            )
        ]["ASR_%"]
        .iloc[0]
    )

    hsj_defended = (
        val_hsj_summary[
            (
                val_hsj_summary["Model"]
                ==
                model_name
            )
            &
            (
                val_hsj_summary["Condition"]
                ==
                "Defended"
            )
        ]["ASR_%"]
        .iloc[0]
    )

    hsj_not_worse = (
        hsj_defended
        <=
        hsj_original
        +
        1e-12
    )

    zoo_not_worse = True

    zoo_original = np.nan
    zoo_defended = np.nan


    if model_name in ZOO_VALIDATION_MODELS:

        zoo_original = (
            val_zoo_summary[
                (
                    val_zoo_summary["Model"]
                    ==
                    model_name
                )
                &
                (
                    val_zoo_summary["Condition"]
                    ==
                    "Original"
                )
            ]["ASR_%"]
            .iloc[0]
        )

        zoo_defended = (
            val_zoo_summary[
                (
                    val_zoo_summary["Model"]
                    ==
                    model_name
                )
                &
                (
                    val_zoo_summary["Condition"]
                    ==
                    "Defended"
                )
            ]["ASR_%"]
            .iloc[0]
        )

        zoo_not_worse = (
            zoo_defended
            <=
            zoo_original
            +
            1e-12
        )


    robust_gain = (

        hsj_defended
        <
        hsj_original

        or

        (
            model_name
            in
            ZOO_VALIDATION_MODELS

            and

            zoo_defended
            <
            zoo_original
        )
    )


    validation_decision_rows.append({

        "Model":
            model_name,

        "Clean_Gate":
            bool(
                clean_row[
                    "Clean_Gate"
                ]
            ),

        "HSJ_Original":
            hsj_original,

        "HSJ_Defended":
            hsj_defended,

        "ZOO_Original":
            zoo_original,

        "ZOO_Defended":
            zoo_defended,

        "HSJ_Not_Worse":
            hsj_not_worse,

        "ZOO_Not_Worse":
            zoo_not_worse,

        "Robust_Gain":
            robust_gain,

        "PASS":
            bool(
                clean_row[
                    "Clean_Gate"
                ]
            )
            and
            hsj_not_worse
            and
            zoo_not_worse
            and
            robust_gain
    })


validation_decision_df = pd.DataFrame(
    validation_decision_rows
)

display(
    validation_decision_df.round(2)
)

,Model,Clean_Gate,HSJ_Original,HSJ_Defended,ZOO_Original,ZOO_Defended,HSJ_Not_Worse,ZOO_Not_Worse,Robust_Gain,PASS
0,Régression logistique,True,25.0,20.0,55.0,60.0,True,False,True,False
1,SVM linéaire,True,30.0,25.0,100.0,100.0,True,True,True,True
2,Random Forest,True,25.0,15.0,NaN,NaN,True,True,True,True
3,XGBoost,True,35.0,25.0,NaN,NaN,True,True,True,True
4,LightGBM,True,40.0,15.0,NaN,NaN,True,True,True,True


In [61]:
# ============================================================
# ÉTAPE 41 — SYNTHÈSE VALIDATION ET GEL DE LA DÉFENSE
# ============================================================

def summarize_validation_attack(
    df,
    success_column
):

    rows = []

    for (
        condition,
        model_name
    ), group in df.groupby(
        [
            "Condition",
            "Model"
        ]
    ):

        eligible = int(
            group[
                "Eligible"
            ].sum()
        )

        success = int(
            group[
                success_column
            ].sum()
        )

        asr = (
            100
            *
            success
            /
            eligible
            if eligible > 0
            else np.nan
        )

        clean_retention = (
            100
            *
            eligible
            /
            len(
                VAL_ATTACK_POSITIONS
            )
        )

        fixed_panel_failure = (
            100
            *
            (
                (
                    len(
                        VAL_ATTACK_POSITIONS
                    )
                    -
                    eligible
                )
                +
                success
            )
            /
            len(
                VAL_ATTACK_POSITIONS
            )
        )

        rows.append({

            "Condition":
                condition,

            "Model":
                model_name,

            "Eligible":
                eligible,

            "Successful":
                success,

            "ASR_%":
                asr,

            "Clean_Retention_%":
                clean_retention,

            "Fixed_Panel_Failure_%":
                fixed_panel_failure
        })

    return pd.DataFrame(
        rows
    )


val_hsj_summary = (
    summarize_validation_attack(
        validation_hsj_df,
        "Local_Success"
    )
)

val_zoo_summary = (
    summarize_validation_attack(
        validation_zoo_df,
        "Attack_Success"
    )
)


print("HSJ VALIDATION")
display(
    val_hsj_summary.round(2)
)

print("\nZOO VALIDATION")
display(
    val_zoo_summary.round(2)
)

HSJ VALIDATION


,Condition,Model,Eligible,Successful,ASR_%,Clean_Retention_%,Fixed_Panel_Failure_%
0,Defended,LightGBM,20,3,15.0,100.0,15.0
1,Defended,Random Forest,20,3,15.0,100.0,15.0
2,Defended,Régression logistique,20,4,20.0,100.0,20.0
3,Defended,SVM linéaire,20,5,25.0,100.0,25.0
4,Defended,XGBoost,20,5,25.0,100.0,25.0
5,Original,LightGBM,20,8,40.0,100.0,40.0
6,Original,Random Forest,20,5,25.0,100.0,25.0
7,Original,Régression logistique,20,5,25.0,100.0,25.0
8,Original,SVM linéaire,20,6,30.0,100.0,30.0
9,Original,XGBoost,20,7,35.0,100.0,35.0



ZOO VALIDATION


,Condition,Model,Eligible,Successful,ASR_%,Clean_Retention_%,Fixed_Panel_Failure_%
0,Defended,Régression logistique,20,12,60.0,100.0,60.0
1,Defended,SVM linéaire,20,20,100.0,100.0,100.0
2,Original,Régression logistique,20,11,55.0,100.0,55.0
3,Original,SVM linéaire,20,20,100.0,100.0,100.0


In [60]:
# ============================================================
# ÉTAPE 40B — ZOO VALIDATION AVANT / APRÈS
# ============================================================

validation_zoo_rows = []

ZOO_VALIDATION_MODELS = [
    "Régression logistique",
    "SVM linéaire"
]


for condition, model_dict in [

    (
        "Original",
        models_original
    ),

    (
        "Defended",
        models_defended
    )

]:

    for model_name in ZOO_VALIDATION_MODELS:

        model = model_dict[
            model_name
        ]

        print(
            "\n",
            condition,
            "→",
            model_name
        )

        for pos in VAL_ATTACK_POSITIONS:

            result, _ = (
                run_zoo_eval_attack(
                    model_name,
                    model,
                    pos,
                    X_val
                )
            )

            result[
                "Condition"
            ] = condition

            validation_zoo_rows.append(
                result
            )


validation_zoo_df = pd.DataFrame(
    validation_zoo_rows
)

validation_zoo_df.to_csv(
    VALIDATION_DIR
    /
    "zoo_validation_before_after.csv",
    index=False
)

print(
    "\n✅ ZOO Validation terminé."
)


 Original → Régression logistique

 Original → SVM linéaire

 Defended → Régression logistique

 Defended → SVM linéaire

✅ ZOO Validation terminé.


In [59]:
# ============================================================
# ÉTAPE 40A — HSJ VALIDATION AVANT / APRÈS
# ============================================================

validation_hsj_rows = []


for condition, model_dict in [

    (
        "Original",
        models_original
    ),

    (
        "Defended",
        models_defended
    )

]:

    for model_name in MODEL_ORDER:

        model = model_dict[
            model_name
        ]

        print(
            "\n",
            condition,
            "→",
            model_name
        )

        for pos in VAL_ATTACK_POSITIONS:

            result, _ = (
                run_hsj_eval_attack(
                    model_name,
                    model,
                    pos,
                    X_val,

                    # Validation sert ici
                    # uniquement à l'évaluation
                    X_val,
                    y_val,

                    salt=500000
                )
            )

            result[
                "Condition"
            ] = condition

            validation_hsj_rows.append(
                result
            )


validation_hsj_df = pd.DataFrame(
    validation_hsj_rows
)

validation_hsj_df.to_csv(
    VALIDATION_DIR
    /
    "hsj_validation_before_after.csv",
    index=False
)

print(
    "\n✅ HSJ Validation terminé."
)


 Original → Régression logistique

 Original → SVM linéaire

 Original → Random Forest

 Original → XGBoost

 Original → LightGBM

 Defended → Régression logistique

 Defended → SVM linéaire

 Defended → Random Forest

 Defended → XGBoost

 Defended → LightGBM

✅ HSJ Validation terminé.


In [58]:
# ============================================================
# ÉTAPE 39B — ZOO GÉNÉRIQUE POUR ÉVALUATION
# ============================================================

def run_zoo_eval_attack(
    model_name,
    model,
    source_position,
    X_source
):

    source_position = int(
        source_position
    )

    x_clean = (
        X_source
        .iloc[source_position]
        .to_numpy(
            dtype=np.float32
        )
    )

    clean_df = pd.DataFrame(
        [x_clean],
        columns=FEATURE_COLUMNS
    )

    clean_pred = int(
        model.predict(clean_df)[0]
    )

    clean_score = float(
        positive_score(
            model,
            clean_df
        )[0]
    )

    if clean_pred != 1:

        return {
            "Model": model_name,
            "Source_Position": source_position,
            "Eligible": 0,
            "Clean_Pred": clean_pred,
            "Clean_Score": clean_score,
            "Attack_Success": 0
        }, None

    classifier, counter = (
        build_blackbox_classifier(
            model
        )
    )

    attack = ZooAttack(
        classifier=classifier,
        **ZOO_CONFIG_TRAIN
    )

    counter.reset()

    start = time.perf_counter()

    x_adv = attack.generate(

        x=x_clean.reshape(
            1,
            -1
        ),

        y=np.array(
            [0],
            dtype=np.int64
        )

    )[0]

    elapsed = (
        time.perf_counter()
        -
        start
    )

    adv_df = pd.DataFrame(
        [x_adv],
        columns=FEATURE_COLUMNS
    )

    adv_pred = int(
        model.predict(
            adv_df
        )[0]
    )

    adv_score = float(
        positive_score(
            model,
            adv_df
        )[0]
    )

    delta = (
        x_adv.astype(np.float64)
        -
        x_clean.astype(np.float64)
    )

    return {

        "Model":
            model_name,

        "Source_Position":
            source_position,

        "Eligible":
            1,

        "Clean_Pred":
            clean_pred,

        "Adv_Pred":
            adv_pred,

        "Clean_Score":
            clean_score,

        "Adv_Score":
            adv_score,

        "Attack_Success":
            int(
                adv_pred == 0
            ),

        "L2":
            float(
                np.linalg.norm(
                    delta
                )
            ),

        "Linf":
            float(
                np.max(
                    np.abs(delta)
                )
            ),

        "Queries":
            int(counter.queries),

        "Time_s":
            elapsed

    }, x_adv

In [57]:
def run_hsj_eval_attack(
    model_name,
    model,
    source_position,
    X_source,
    seed_X,
    seed_y,
    salt=0
):

    source_position = int(
        source_position
    )

    x_clean = (
        X_source
        .iloc[source_position]
        .to_numpy(
            dtype=np.float32
        )
    )

    clean_df = pd.DataFrame(
        [x_clean],
        columns=FEATURE_COLUMNS
    )

    clean_pred = int(
        model.predict(clean_df)[0]
    )

    clean_score = float(
        positive_score(
            model,
            clean_df
        )[0]
    )

    if clean_pred != 1:

        return {
            "Model": model_name,
            "Source_Position": source_position,
            "Eligible": 0,
            "Clean_Pred": clean_pred,
            "Clean_Score": clean_score,
            "Raw_Success": 0,
            "Local_Success": 0
        }, None

    x_init = find_hsj_eval_seed(
        model_name,
        model,
        x_clean,
        seed_X,
        seed_y,
        source_position,
        salt=salt
    )

    if x_init is None:

        return {
            "Model": model_name,
            "Source_Position": source_position,
            "Eligible": 1,
            "Seed_Found": 0,
            "Clean_Pred": clean_pred,
            "Clean_Score": clean_score,
            "Raw_Success": 0,
            "Local_Success": 0
        }, None

    classifier, counter = (
        build_blackbox_classifier(
            model
        )
    )

    attack = HopSkipJump(
        classifier=classifier,
        **HSJ_CONFIG_TRAIN
    )

    counter.reset()

    start = time.perf_counter()

    x_adv = attack.generate(

        x=x_clean.reshape(
            1,
            -1
        ),

        y=np.array(
            [0],
            dtype=np.int64
        ),

        mask=HSJ_MASK,

        x_adv_init=x_init.reshape(
            1,
            -1
        )

    )[0]

    elapsed = (
        time.perf_counter()
        -
        start
    )

    adv_df = pd.DataFrame(
        [x_adv],
        columns=FEATURE_COLUMNS
    )

    adv_pred = int(
        model.predict(
            adv_df
        )[0]
    )

    adv_score = float(
        positive_score(
            model,
            adv_df
        )[0]
    )

    delta = (
        x_adv.astype(np.float64)
        -
        x_clean.astype(np.float64)
    )

    l2 = float(
        np.linalg.norm(
            delta,
            ord=2
        )
    )

    linf = float(
        np.max(
            np.abs(delta)
        )
    )

    immutable_change = float(
        np.max(
            np.abs(
                delta[
                    IMMUTABLE_IDX
                ]
            )
        )
    )

    raw_success = int(
        adv_pred == 0
    )

    local_success = int(

        raw_success == 1

        and

        linf
        <=
        HSJ_LOCAL_EPS
        +
        1e-8

        and

        immutable_change
        <=
        1e-8
    )

    result = {

        "Model":
            model_name,

        "Source_Position":
            source_position,

        "Eligible":
            1,

        "Seed_Found":
            1,

        "Clean_Pred":
            clean_pred,

        "Adv_Pred":
            adv_pred,

        "Clean_Score":
            clean_score,

        "Adv_Score":
            adv_score,

        "Raw_Success":
            raw_success,

        "Local_Success":
            local_success,

        "L2":
            l2,

        "Linf":
            linf,

        "Max_Immutable_Change":
            immutable_change,

        "Queries":
            int(counter.queries),

        "Time_s":
            elapsed
    }

    return result, x_adv

In [56]:
# ============================================================
# ÉTAPE 39 — HSJ GÉNÉRIQUE POUR ÉVALUATION
# ============================================================

def find_hsj_eval_seed(
    model_name,
    model,
    x_clean,
    seed_X,
    seed_y,
    source_position,
    salt=0,
    max_trials=5000
):

    normal_positions = np.where(
        seed_y.to_numpy() == 0
    )[0]

    rng = np.random.default_rng(
        2026
        +
        int(source_position)
        +
        MODEL_INDEX[model_name]
        * 100000
        +
        salt
    )

    n_trials = min(
        max_trials,
        len(normal_positions)
    )

    candidates_pos = rng.choice(
        normal_positions,
        size=n_trials,
        replace=False
    )

    candidates = np.repeat(
        x_clean.reshape(1, -1),
        n_trials,
        axis=0
    )

    candidates[
        :,
        MUTABLE_IDX
    ] = (
        seed_X
        .iloc[candidates_pos][
            HSJ_MUTABLE_FEATURES
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    pred = model.predict(
        pd.DataFrame(
            candidates,
            columns=FEATURE_COLUMNS
        )
    )

    valid = np.where(
        pred == 0
    )[0]

    if len(valid) == 0:
        return None

    return candidates[
        int(valid[0])
    ].copy()

In [55]:
# ============================================================
# ÉTAPE 38 — PANEL VALIDATION FIXE
# ============================================================

val_tp_by_model = {}


for model_name, model in models_original.items():

    predictions = model.predict(
        X_val
    )

    positions = np.where(
        (y_val.to_numpy() == 1)
        &
        (predictions == 1)
    )[0]

    val_tp_by_model[
        model_name
    ] = set(
        int(x)
        for x in positions
    )


common_val_positions = sorted(
    set.intersection(
        *val_tp_by_model.values()
    )
)


print(
    "Fraudes Validation communes :",
    len(common_val_positions)
)


N_VAL_ATTACK = min(
    20,
    len(common_val_positions)
)


VAL_ATTACK_POSITIONS = (
    common_val_positions[
        :N_VAL_ATTACK
    ]
)


print(
    "Panel Validation utilisé :",
    len(VAL_ATTACK_POSITIONS)
)

print(
    VAL_ATTACK_POSITIONS
)


np.save(
    VALIDATION_DIR
    /
    "validation_attack_positions.npy",

    np.asarray(
        VAL_ATTACK_POSITIONS,
        dtype=int
    )
)

Fraudes Validation communes : 52
Panel Validation utilisé : 20
[544, 666, 1899, 2897, 4127, 4775, 8418, 9179, 9888, 11066, 11324, 11695, 12246, 12521, 14382, 15814, 15912, 16048, 16641, 17312]


In [54]:
# ============================================================
# ÉTAPE 37B — DELTAS CLEAN VALIDATION
# ============================================================

validation_clean_pivot = (
    validation_clean_df
    .pivot(
        index="Model",
        columns="Condition"
    )
)

clean_delta_rows = []


for model_name in MODEL_ORDER:

    original = (
        validation_clean_df[
            (
                validation_clean_df["Model"]
                ==
                model_name
            )
            &
            (
                validation_clean_df["Condition"]
                ==
                "Original"
            )
        ]
        .iloc[0]
    )

    defended = (
        validation_clean_df[
            (
                validation_clean_df["Model"]
                ==
                model_name
            )
            &
            (
                validation_clean_df["Condition"]
                ==
                "Defended"
            )
        ]
        .iloc[0]
    )

    clean_delta_rows.append({

        "Model":
            model_name,

        "Recall_Original":
            original["Recall"],

        "Recall_Defended":
            defended["Recall"],

        "Delta_Recall":
            defended["Recall"]
            -
            original["Recall"],

        "F1_Original":
            original["F1"],

        "F1_Defended":
            defended["F1"],

        "Delta_F1":
            defended["F1"]
            -
            original["F1"],

        "FN_Original":
            int(original["FN"]),

        "FN_Defended":
            int(defended["FN"])
    })


validation_clean_delta_df = pd.DataFrame(
    clean_delta_rows
)


validation_clean_delta_df[
    "Clean_Gate"
] = (

    (
        validation_clean_delta_df[
            "Delta_Recall"
        ]
        >=
        -0.05
    )

    &

    (
        validation_clean_delta_df[
            "Delta_F1"
        ]
        >=
        -0.05
    )
)


display(
    validation_clean_delta_df.round(4)
)

,Model,Recall_Original,Recall_Defended,Delta_Recall,F1_Original,F1_Defended,Delta_F1,FN_Original,FN_Defended,Clean_Gate
0,Régression logistique,0.8310,0.8310,0.0000,0.6114,0.5990,-0.0124,12,12,True
1,SVM linéaire,0.8169,0.8169,0.0000,0.7342,0.7436,0.0094,13,13,True
2,Random Forest,0.7465,0.7746,0.0282,0.8346,0.8397,0.0050,18,16,True
3,XGBoost,0.7887,0.7887,0.0000,0.8485,0.8175,-0.0310,15,15,True
4,LightGBM,0.7887,0.7887,0.0000,0.8550,0.8421,-0.0129,15,15,True


In [53]:
# ============================================================
# ÉTAPE 37 — CLEAN VALIDATION AVANT / APRÈS
# ============================================================

validation_clean_rows = []


for model_name in MODEL_ORDER:

    for condition, model in [
        (
            "Original",
            models_original[
                model_name
            ]
        ),
        (
            "Defended",
            models_defended[
                model_name
            ]
        )
    ]:

        metrics = evaluate_model(
            model,
            X_val,
            y_val
        )

        validation_clean_rows.append({
            "Model":
                model_name,

            "Condition":
                condition,

            **metrics
        })


validation_clean_df = pd.DataFrame(
    validation_clean_rows
)

display(
    validation_clean_df.round(4)
)

,Model,Condition,Accuracy,Precision,Recall,F1,ROC_AUC,TN,FP,FN,TP
0,Régression logistique,Original,0.9982,0.4836,0.8310,0.6114,0.9672,42425,63,12,59
1,Régression logistique,Defended,0.9981,0.4683,0.8310,0.5990,0.9676,42421,67,12,59
2,SVM linéaire,Original,0.9990,0.6667,0.8169,0.7342,0.9714,42459,29,13,58
3,SVM linéaire,Defended,0.9991,0.6824,0.8169,0.7436,0.9731,42461,27,13,58
4,Random Forest,Original,0.9995,0.9464,0.7465,0.8346,0.9696,42485,3,18,53
5,Random Forest,Defended,0.9995,0.9167,0.7746,0.8397,0.9617,42483,5,16,55
6,XGBoost,Original,0.9995,0.9180,0.7887,0.8485,0.9772,42483,5,15,56
7,XGBoost,Defended,0.9994,0.8485,0.7887,0.8175,0.9801,42478,10,15,56
8,LightGBM,Original,0.9996,0.9333,0.7887,0.8550,0.9753,42484,4,15,56
9,LightGBM,Defended,0.9995,0.9032,0.7887,0.8421,0.9746,42482,6,15,56


In [52]:
# ============================================================
# ÉTAPE 36 — FONCTIONS D'ÉVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


def positive_score(model, X):

    if hasattr(model, "predict_proba"):
        return (
            model.predict_proba(X)[:, 1]
        )

    if hasattr(model, "decision_function"):
        return (
            model.decision_function(X)
        )

    return (
        model.predict(X)
        .astype(float)
    )


def evaluate_model(
    model,
    X,
    y
):

    y_pred = model.predict(X)

    y_score = positive_score(
        model,
        X
    )

    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            y_pred
        )
        .ravel()
    )

    return {

        "Accuracy":
            accuracy_score(
                y,
                y_pred
            ),

        "Precision":
            precision_score(
                y,
                y_pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y,
                y_pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y,
                y_pred,
                zero_division=0
            ),

        "ROC_AUC":
            roc_auc_score(
                y,
                y_score
            ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

In [51]:
# ============================================================
# ÉTAPE 35 — ADVERSARIAL TRAINING
# ============================================================

from sklearn.base import clone
import time
import joblib

models_defended = {}

adversarial_training_rows = []


for model_name in MODEL_ORDER:

    print("\n" + "=" * 75)
    print("DÉFENSE :", model_name)
    print("=" * 75)

    X_def, y_def = (
        defended_train_by_model[
            model_name
        ]
    )

    # Modèle neuf, mêmes hyperparamètres
    defended_model = clone(
        models_original[
            model_name
        ]
    )

    start = time.perf_counter()

    defended_model.fit(
        X_def,
        y_def
    )

    training_time = (
        time.perf_counter()
        -
        start
    )

    models_defended[
        model_name
    ] = defended_model

    adversarial_training_rows.append({
        "Model":
            model_name,

        "Train_Size":
            len(X_def),

        "Adversarial_Examples":
            len(
                adversarial_bank_by_model[
                    model_name
                ]
            ),

        "Training_Time_s":
            training_time
    })

    model_file = (
        MODELS_DIR
        /
        (
            MODEL_SLUG[
                model_name
            ]
            +
            "_defended.joblib"
        )
    )

    joblib.dump(
        defended_model,
        model_file
    )

    print(
        "✅ entraîné en",
        round(training_time, 2),
        "s"
    )


adversarial_training_df = pd.DataFrame(
    adversarial_training_rows
)

display(
    adversarial_training_df.round(3)
)

print(
    "\n✅ Les 5 modèles défendus sont disponibles."
)


DÉFENSE : Régression logistique
✅ entraîné en 0.55 s

DÉFENSE : SVM linéaire
✅ entraîné en 1.51 s

DÉFENSE : Random Forest
✅ entraîné en 28.93 s

DÉFENSE : XGBoost
✅ entraîné en 1.78 s

DÉFENSE : LightGBM
✅ entraîné en 1.17 s


,Model,Train_Size,Adversarial_Examples,Training_Time_s
0,Régression logistique,218342,237,0.546
1,SVM linéaire,218481,376,1.511
2,Random Forest,218194,89,28.925
3,XGBoost,218202,97,1.778
4,LightGBM,218234,129,1.165



✅ Les 5 modèles défendus sont disponibles.


In [50]:
# ============================================================
# ÉTAPE 34B — CONTRÔLE ANTI-FUITE
# ============================================================

print("=" * 75)
print("CONTRÔLE MÉTHODOLOGIQUE")
print("=" * 75)

print(
    "Sources adversariales : TRAIN uniquement"
)

print(
    "Validation utilisée pour entraînement : NON"
)

print(
    "Test utilisé pour entraînement : NON"
)

print(
    "Anciens exemples adversariaux Test utilisés : NON"
)

assert len(common_test_positions) == 53

print(
    "\n✅ Séparation Train / Validation / Test respectée."
)

CONTRÔLE MÉTHODOLOGIQUE
Sources adversariales : TRAIN uniquement
Validation utilisée pour entraînement : NON
Test utilisé pour entraînement : NON
Anciens exemples adversariaux Test utilisés : NON

✅ Séparation Train / Validation / Test respectée.


In [49]:
# ============================================================
# ÉTAPE 34 — CONSTRUCTION DES TRAIN DÉFENDUS
# ============================================================

from sklearn.utils import shuffle

defended_train_by_model = {}

training_composition_rows = []

for model_name in MODEL_ORDER:

    X_adv = (
        adversarial_bank_by_model[
            model_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    y_adv = pd.Series(
        np.ones(
            len(X_adv),
            dtype=int
        ),
        name="Class"
    )

    # --------------------------------------------------------
    # Ajouter les exemples adversariaux au Train augmenté
    # --------------------------------------------------------

    X_def = pd.concat(
        [
            X_train_aug.reset_index(drop=True),
            X_adv
        ],
        ignore_index=True
    )

    y_def = pd.concat(
        [
            y_train_aug.reset_index(drop=True),
            y_adv
        ],
        ignore_index=True
    )

    # --------------------------------------------------------
    # Mélange reproductible
    # --------------------------------------------------------

    X_def, y_def = shuffle(
        X_def,
        y_def,
        random_state=42
    )

    X_def = X_def.reset_index(drop=True)
    y_def = y_def.reset_index(drop=True)

    defended_train_by_model[
        model_name
    ] = (
        X_def,
        y_def
    )

    training_composition_rows.append({
        "Model": model_name,

        "Base_Train":
            len(X_train_aug),

        "Adversarial_Added":
            len(X_adv),

        "Final_Train":
            len(X_def),

        "Final_Normal":
            int((y_def == 0).sum()),

        "Final_Fraud":
            int((y_def == 1).sum()),

        "Fraud_Normal_Ratio":
            float(
                (y_def == 1).sum()
                /
                (y_def == 0).sum()
            )
    })


training_composition_df = pd.DataFrame(
    training_composition_rows
)

display(
    training_composition_df.round(5)
)

print(
    "\n✅ Train défendu construit "
    "séparément pour chaque modèle."
)

,Model,Base_Train,Adversarial_Added,Final_Train,Final_Normal,Final_Fraud,Fraud_Normal_Ratio
0,Régression logistique,218105,237,218342,198277,20065,0.10120
1,SVM linéaire,218105,376,218481,198277,20204,0.10190
2,Random Forest,218105,89,218194,198277,19917,0.10045
3,XGBoost,218105,97,218202,198277,19925,0.10049
4,LightGBM,218105,129,218234,198277,19957,0.10065



✅ Train défendu construit séparément pour chaque modèle.


In [48]:
# ============================================================
# ÉTAPE 33C — SAUVEGARDE DES BANQUES ADVERSARIALES PROPRES
# ============================================================

CLEAN_BANK_DIR = (
    BANK_DIR /
    "clean_adversarial_banks"
)

CLEAN_BANK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_SLUG = {
    "Régression logistique": "regression_logistique",
    "SVM linéaire": "svm_lineaire",
    "Random Forest": "random_forest",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm"
}

saved_bank_rows = []

for model_name in MODEL_ORDER:

    X_adv = adversarial_bank_by_model[
        model_name
    ].copy()

    # Label réel
    y_adv = pd.Series(
        np.ones(
            len(X_adv),
            dtype=int
        ),
        name="Class"
    )

    slug = MODEL_SLUG[
        model_name
    ]

    X_file = (
        CLEAN_BANK_DIR /
        f"{slug}_X_adv.parquet"
    )

    y_file = (
        CLEAN_BANK_DIR /
        f"{slug}_y_adv.parquet"
    )

    X_adv.to_parquet(
        X_file,
        index=False
    )

    y_adv.to_frame().to_parquet(
        y_file,
        index=False
    )

    saved_bank_rows.append({
        "Model": model_name,
        "N_Adversarial": len(X_adv),
        "X_File": str(X_file),
        "y_File": str(y_file)
    })


# ------------------------------------------------------------
# Sauvegarder aussi les métadonnées
# ------------------------------------------------------------

METADATA_FILE = (
    CLEAN_BANK_DIR /
    "adversarial_metadata.csv"
)

adversarial_metadata.to_csv(
    METADATA_FILE,
    index=False
)

saved_banks_df = pd.DataFrame(
    saved_bank_rows
)

display(saved_banks_df)

print(
    "\nMétadonnées :",
    METADATA_FILE
)

print(
    "\n✅ Banques adversariales propres sauvegardées."
)

,Model,N_Adversarial,X_File,y_File
0,Régression logistique,237,c:\Users\MSI\Desktop\fraud_detection_memoire\n...,c:\Users\MSI\Desktop\fraud_detection_memoire\n...
1,SVM linéaire,376,c:\Users\MSI\Desktop\fraud_detection_memoire\n...,c:\Users\MSI\Desktop\fraud_detection_memoire\n...
2,Random Forest,89,c:\Users\MSI\Desktop\fraud_detection_memoire\n...,c:\Users\MSI\Desktop\fraud_detection_memoire\n...
3,XGBoost,97,c:\Users\MSI\Desktop\fraud_detection_memoire\n...,c:\Users\MSI\Desktop\fraud_detection_memoire\n...
4,LightGBM,129,c:\Users\MSI\Desktop\fraud_detection_memoire\n...,c:\Users\MSI\Desktop\fraud_detection_memoire\n...



Métadonnées : c:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training\adversarial_bank\clean_adversarial_banks\adversarial_metadata.csv

✅ Banques adversariales propres sauvegardées.


In [47]:
# ============================================================
# ÉTAPE 33B — CONTRÔLE D'INTÉGRITÉ DES BANQUES ADVERSARIALES
# ============================================================

integrity_rows = []

for model_name in MODEL_ORDER:

    X_adv = adversarial_bank_by_model[model_name]

    # --------------------------------------------------------
    # Contrôles de base
    # --------------------------------------------------------

    n_samples = len(X_adv)

    n_nan = int(
        X_adv.isna().sum().sum()
    )

    n_inf = int(
        np.isinf(
            X_adv.to_numpy(dtype=float)
        ).sum()
    )

    duplicate_rows = int(
        X_adv.duplicated().sum()
    )

    correct_shape = (
        X_adv.shape[1]
        ==
        len(FEATURE_COLUMNS)
    )

    correct_columns = (
        list(X_adv.columns)
        ==
        FEATURE_COLUMNS
    )

    # --------------------------------------------------------
    # Vérifier que le modèle original est bien trompé
    # --------------------------------------------------------

    original_model = models_original[
        model_name
    ]

    adv_predictions = (
        original_model.predict(
            X_adv
        )
    )

    predicted_normal = int(
        (adv_predictions == 0).sum()
    )

    predicted_fraud = int(
        (adv_predictions == 1).sum()
    )

    # --------------------------------------------------------
    # Label réel pour l'Adversarial Training
    # --------------------------------------------------------

    y_adv = pd.Series(
        np.ones(
            n_samples,
            dtype=int
        ),
        name="Class"
    )

    assert (
        y_adv == 1
    ).all()

    integrity_rows.append({

        "Model": model_name,

        "N_Examples":
            n_samples,

        "Features":
            X_adv.shape[1],

        "NaN":
            n_nan,

        "Inf":
            n_inf,

        "Duplicates":
            duplicate_rows,

        "Predicted_Normal":
            predicted_normal,

        "Predicted_Fraud":
            predicted_fraud,

        "Columns_OK":
            correct_columns,

        "Shape_OK":
            correct_shape,

        "True_Label":
            1
    })


integrity_df = pd.DataFrame(
    integrity_rows
)

display(integrity_df)

# ------------------------------------------------------------
# Vérifications strictes
# ------------------------------------------------------------

assert (
    integrity_df["NaN"] == 0
).all()

assert (
    integrity_df["Inf"] == 0
).all()

assert (
    integrity_df["Columns_OK"]
).all()

assert (
    integrity_df["Shape_OK"]
).all()

assert (
    integrity_df["Predicted_Fraud"] == 0
).all()

print(
    "\n✅ Toutes les banques adversariales "
    "sont cohérentes et exploitables."
)

,Model,N_Examples,Features,NaN,Inf,Duplicates,Predicted_Normal,Predicted_Fraud,Columns_OK,Shape_OK,True_Label
0,Régression logistique,237,30,0,0,0,237,0,True,True,1
1,SVM linéaire,376,30,0,0,0,376,0,True,True,1
2,Random Forest,89,30,0,0,0,89,0,True,True,1
3,XGBoost,97,30,0,0,0,97,0,True,True,1
4,LightGBM,129,30,0,0,0,129,0,True,True,1



✅ Toutes les banques adversariales sont cohérentes et exploitables.


In [46]:
# ============================================================
# ÉTAPE 33 — CONSTRUCTION DES BANQUES PAR MODÈLE
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

MODEL_ORDER = [
    "Régression logistique",
    "SVM linéaire",
    "Random Forest",
    "XGBoost",
    "LightGBM"
]

MODEL_SLUG = {
    "Régression logistique": "regression_logistique",
    "SVM linéaire": "svm_lineaire",
    "Random Forest": "random_forest",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm"
}

# ------------------------------------------------------------
# Plan de défense
# ------------------------------------------------------------

DEFENCE_ATTACK_PLAN = {
    "Régression logistique": ["HSJ", "ZOO"],
    "SVM linéaire": ["HSJ", "ZOO"],
    "Random Forest": ["HSJ"],
    "XGBoost": ["HSJ"],
    "LightGBM": ["HSJ"]
}

# ------------------------------------------------------------
# Construire une banque propre à chaque modèle
# ------------------------------------------------------------

adversarial_bank_by_model = {}
adversarial_metadata_rows = []

for model_name in MODEL_ORDER:

    samples = []

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    # ========================================================
    # HSJ
    # On garde uniquement Local_Success = 1
    # ========================================================

    if "HSJ" in DEFENCE_ATTACK_PLAN[model_name]:

        valid_hsj = hsj_train_df[
            (hsj_train_df["Model"] == model_name)
            &
            (hsj_train_df["Local_Success"] == 1)
        ].copy()

        print(
            "HSJ locaux valides :",
            len(valid_hsj)
        )

        for _, row in valid_hsj.iterrows():

            source_position = int(
                row["Source_Position"]
            )

            sample_file = (
                HSJ_TRAIN_ADV_DIR
                /
                (
                    f"{MODEL_SLUG[model_name]}"
                    f"_train_{source_position}.npy"
                )
            )

            if not sample_file.exists():
                raise FileNotFoundError(
                    f"HSJ manquant : {sample_file}"
                )

            x_adv = np.load(
                sample_file
            ).astype(np.float64)

            samples.append(x_adv)

            adversarial_metadata_rows.append({
                "Model": model_name,
                "Attack": "HSJ",
                "Source_Position": source_position,
                "Linf": float(row["Linf"]),
                "L2": float(row["L2"])
            })

    # ========================================================
    # ZOO
    # On garde uniquement Attack_Success = 1
    # ========================================================

    if "ZOO" in DEFENCE_ATTACK_PLAN[model_name]:

        valid_zoo = zoo_train_df[
            (zoo_train_df["Model"] == model_name)
            &
            (zoo_train_df["Attack_Success"] == 1)
        ].copy()

        print(
            "ZOO valides :",
            len(valid_zoo)
        )

        for _, row in valid_zoo.iterrows():

            source_position = int(
                row["Source_Position"]
            )

            sample_file = (
                ZOO_TRAIN_ADV_DIR
                /
                (
                    f"{MODEL_SLUG[model_name]}"
                    f"_train_{source_position}.npy"
                )
            )

            if not sample_file.exists():
                raise FileNotFoundError(
                    f"ZOO manquant : {sample_file}"
                )

            x_adv = np.load(
                sample_file
            ).astype(np.float64)

            samples.append(x_adv)

            adversarial_metadata_rows.append({
                "Model": model_name,
                "Attack": "ZOO",
                "Source_Position": source_position,
                "Linf": float(row["Linf"]),
                "L2": float(row["L2"])
            })

    # --------------------------------------------------------
    # Transformer la banque du modèle en DataFrame
    # --------------------------------------------------------

    if len(samples) == 0:
        raise RuntimeError(
            f"Aucun exemple adversarial pour {model_name}"
        )

    X_adv_model = pd.DataFrame(
        np.vstack(samples),
        columns=FEATURE_COLUMNS
    )

    adversarial_bank_by_model[
        model_name
    ] = X_adv_model

    print(
        "Total candidats adversariaux :",
        len(X_adv_model)
    )

# ------------------------------------------------------------
# Métadonnées globales
# ------------------------------------------------------------

adversarial_metadata = pd.DataFrame(
    adversarial_metadata_rows
)

print("\n" + "=" * 75)
print("BANQUES ADVERSARIALES PAR MODÈLE")
print("=" * 75)

summary_bank = pd.DataFrame({
    "Model": MODEL_ORDER,

    "N_Adversarial": [
        len(
            adversarial_bank_by_model[
                model
            ]
        )
        for model in MODEL_ORDER
    ]
})

display(summary_bank)

print(
    "\nTotal :",
    summary_bank["N_Adversarial"].sum()
)

assert (
    summary_bank["N_Adversarial"].sum()
    ==
    928
)

print(
    "\n✅ Banques adversariales par modèle construites."
)


Régression logistique
HSJ locaux valides : 72
ZOO valides : 165
Total candidats adversariaux : 237

SVM linéaire
HSJ locaux valides : 95
ZOO valides : 281
Total candidats adversariaux : 376

Random Forest
HSJ locaux valides : 89
Total candidats adversariaux : 89

XGBoost
HSJ locaux valides : 97
Total candidats adversariaux : 97

LightGBM
HSJ locaux valides : 129
Total candidats adversariaux : 129

BANQUES ADVERSARIALES PAR MODÈLE


,Model,N_Adversarial
0,Régression logistique,237
1,SVM linéaire,376
2,Random Forest,89
3,XGBoost,97
4,LightGBM,129



Total : 928

✅ Banques adversariales par modèle construites.


In [45]:
# ============================================================
# ÉTAPE 32 — CONTRÔLE COMBINÉ HSJ + ZOO
# ============================================================

hsj_train_df = pd.read_csv(
    HSJ_TRAIN_RESULTS_FILE
)

zoo_train_df = pd.read_csv(
    ZOO_TRAIN_RESULTS_FILE
)

print("=" * 75)
print("BANQUE ADVERSARIALE — CONTRÔLE COMBINÉ")
print("=" * 75)

# ------------------------------------------------------------
# HSJ
# ------------------------------------------------------------

hsj_errors = (
    hsj_train_df["Error"]
    .fillna("")
    .ne("")
    .sum()
)

hsj_local_total = int(
    hsj_train_df["Local_Success"].sum()
)

print("\nHSJ")
print("-" * 40)

print(
    "Résultats :",
    len(hsj_train_df),
    "/ 1405"
)

print(
    "Erreurs :",
    hsj_errors
)

print(
    "Succès locaux retenables :",
    hsj_local_total
)

hsj_summary = (
    hsj_train_df
    .groupby("Model")
    .agg(
        Attaques=("Source_Position", "count"),
        Succes_bruts=("Raw_Success", "sum"),
        Succes_locaux=("Local_Success", "sum"),
        Linf_moyen=("Linf", "mean"),
        Requetes_moyennes=("Queries", "mean"),
        Temps_moyen_s=("Time_s", "mean")
    )
)

hsj_summary["ASR_local_%"] = (
    hsj_summary["Succes_locaux"]
    /
    hsj_summary["Attaques"]
    * 100
)

display(
    hsj_summary.round(4)
)

# ------------------------------------------------------------
# ZOO
# ------------------------------------------------------------

zoo_errors = (
    zoo_train_df["Error"]
    .fillna("")
    .ne("")
    .sum()
)

zoo_success_total = int(
    zoo_train_df[
        "Attack_Success"
    ].sum()
)

print("\nZOO")
print("-" * 40)

print(
    "Résultats :",
    len(zoo_train_df),
    "/ 562"
)

print(
    "Erreurs :",
    zoo_errors
)

print(
    "Succès retenables :",
    zoo_success_total
)

zoo_summary = (
    zoo_train_df
    .groupby("Model")
    .agg(
        Attaques=("Source_Position", "count"),
        Succes=("Attack_Success", "sum"),
        Linf_moyen=("Linf", "mean"),
        Requetes_moyennes=("Queries", "mean"),
        Temps_moyen_s=("Time_s", "mean")
    )
)

zoo_summary["ASR_%"] = (
    zoo_summary["Succes"]
    /
    zoo_summary["Attaques"]
    * 100
)

display(
    zoo_summary.round(4)
)

# ------------------------------------------------------------
# Contrôles
# ------------------------------------------------------------

assert len(hsj_train_df) == 1405
assert len(zoo_train_df) == 562

assert hsj_errors == 0
assert zoo_errors == 0

print("\n" + "=" * 75)
print("RÉSUMÉ GLOBAL")
print("=" * 75)

print(
    "HSJ exploitables :",
    hsj_local_total
)

print(
    "ZOO exploitables :",
    zoo_success_total
)

print(
    "Total candidats adversariaux :",
    hsj_local_total
    +
    zoo_success_total
)

print(
    "\n✅ Banques HSJ et ZOO contrôlées."
)

BANQUE ADVERSARIALE — CONTRÔLE COMBINÉ

HSJ
----------------------------------------
Résultats : 1405 / 1405
Erreurs : 0
Succès locaux retenables : 482


,Attaques,Succes_bruts,Succes_locaux,Linf_moyen,Requetes_moyennes,Temps_moyen_s,ASR_local_%
Model,,,,,,,
LightGBM,281,281,129,0.0829,24595.5125,1.2632,45.9075
Random Forest,281,281,89,0.0886,24587.4235,31.1611,31.6726
Régression logistique,281,281,72,0.1059,24536.0000,0.8341,25.6228
SVM linéaire,281,281,95,0.0902,24536.0000,0.8085,33.8078
XGBoost,281,281,97,0.0909,24597.4947,2.5034,34.5196



ZOO
----------------------------------------
Résultats : 562 / 562
Erreurs : 0
Succès retenables : 446


,Attaques,Succes,Linf_moyen,Requetes_moyennes,Temps_moyen_s,ASR_%
Model,,,,,,
Régression logistique,281,165,0.0530,16557.8327,12.0755,58.7189
SVM linéaire,281,281,0.1158,14378.3310,10.3912,100.0000



RÉSUMÉ GLOBAL
HSJ exploitables : 482
ZOO exploitables : 446
Total candidats adversariaux : 928

✅ Banques HSJ et ZOO contrôlées.


In [44]:
# ============================================================
# ÉTAPE 31B — CONTRÔLE RAPIDE DE LA BANQUE ZOO
# ============================================================

zoo_train_df = pd.read_csv(
    ZOO_TRAIN_RESULTS_FILE
)

print("=" * 70)
print("CONTRÔLE FINAL ZOO TRAIN")
print("=" * 70)

print("Résultats enregistrés :", len(zoo_train_df))

errors = (
    zoo_train_df["Error"]
    .fillna("")
    .ne("")
    .sum()
)

print("Erreurs :", errors)

print("\nRésumé par modèle :")

zoo_summary_31b = (
    zoo_train_df
    .groupby("Model")
    .agg(
        Attaques=("Source_Position", "count"),
        Succes=("Attack_Success", "sum"),
        Linf_moyen=("Linf", "mean"),
        L2_moyen=("L2", "mean"),
        Requetes_moyennes=("Queries", "mean"),
        Temps_moyen_s=("Time_s", "mean")
    )
)

zoo_summary_31b["ASR_%"] = (
    zoo_summary_31b["Succes"]
    /
    zoo_summary_31b["Attaques"]
    * 100
)

display(
    zoo_summary_31b.round(4)
)

print(
    "\nSuccès ZOO totaux :",
    int(zoo_train_df["Attack_Success"].sum())
)

assert len(zoo_train_df) == 562
assert errors == 0

print("\n✅ Banque ZOO Train complète et exploitable.")

CONTRÔLE FINAL ZOO TRAIN
Résultats enregistrés : 562
Erreurs : 0

Résumé par modèle :


,Attaques,Succes,Linf_moyen,L2_moyen,Requetes_moyennes,Temps_moyen_s,ASR_%
Model,,,,,,,
Régression logistique,281,165,0.0530,0.1143,16557.8327,12.0755,58.7189
SVM linéaire,281,281,0.1158,0.2760,14378.3310,10.3912,100.0000



Succès ZOO totaux : 446

✅ Banque ZOO Train complète et exploitable.


In [43]:
# ============================================================
# ÉTAPE 31 — BANQUE ZOO TRAIN COMPLÈTE
# 281 FRAUDES × 2 MODÈLES
# ============================================================

ZOO_TRAIN_DIR = (
    BANK_DIR /
    "zoo_train"
)

ZOO_TRAIN_ADV_DIR = (
    ZOO_TRAIN_DIR /
    "adversarial_samples"
)

ZOO_TRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ZOO_TRAIN_ADV_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ZOO_TRAIN_RESULTS_FILE = (
    ZOO_TRAIN_DIR /
    "zoo_train_results.csv"
)


if ZOO_TRAIN_RESULTS_FILE.exists():

    zoo_train_df = pd.read_csv(
        ZOO_TRAIN_RESULTS_FILE
    )

else:

    zoo_train_df = pd.DataFrame()


if len(zoo_train_df) > 0:

    if "Error" not in zoo_train_df.columns:
        zoo_train_df["Error"] = ""

    completed_zoo = {

        (
            row["Model"],
            int(
                row[
                    "Source_Position"
                ]
            )
        )

        for _, row
        in zoo_train_df[
            zoo_train_df[
                "Error"
            ]
            .fillna("")
            .eq("")
        ].iterrows()
    }

else:

    completed_zoo = set()


TOTAL_ZOO = (
    len(
        common_train_fraud_positions
    )
    *
    len(ZOO_TRAIN_MODELS)
)

print(
    "Déjà terminées :",
    len(completed_zoo),
    "/",
    TOTAL_ZOO
)


done_counter = len(
    completed_zoo
)


for source_position in common_train_fraud_positions:

    for model_name in ZOO_TRAIN_MODELS:

        pair = (
            model_name,
            int(source_position)
        )

        if pair in completed_zoo:
            continue

        model = models_original[
            model_name
        ]

        try:

            result, x_adv = (
                run_zoo_train_attack(
                    model_name,
                    model,
                    source_position
                )
            )

            result["Error"] = ""

            sample_file = (
                ZOO_TRAIN_ADV_DIR
                /
                (
                    f"{MODEL_SLUG[model_name]}"
                    f"_train_{source_position}.npy"
                )
            )

            np.save(
                sample_file,
                x_adv
            )

        except Exception as e:

            result = {

                "Model":
                    model_name,

                "Source_Position":
                    int(source_position),

                "Error":
                    str(e)
            }

        if len(zoo_train_df) > 0:

            same_pair = (
                (
                    zoo_train_df["Model"]
                    ==
                    model_name
                )
                &
                (
                    zoo_train_df[
                        "Source_Position"
                    ].astype(int)
                    ==
                    int(source_position)
                )
            )

            zoo_train_df = (
                zoo_train_df[
                    ~same_pair
                ]
            )

        zoo_train_df = pd.concat(
            [
                zoo_train_df,
                pd.DataFrame(
                    [result]
                )
            ],
            ignore_index=True
        )

        zoo_train_df.to_csv(
            ZOO_TRAIN_RESULTS_FILE,
            index=False
        )

        if result.get(
            "Error",
            ""
        ) == "":

            completed_zoo.add(
                pair
            )

            done_counter += 1

        if (
            done_counter % 25 == 0
            or
            done_counter == TOTAL_ZOO
        ):

            print(
                f"Progression : "
                f"{done_counter}/{TOTAL_ZOO}"
            )


print(
    "\n✅ ZOO Train terminé."
)

Déjà terminées : 0 / 562
Progression : 25/562
Progression : 50/562
Progression : 75/562
Progression : 100/562
Progression : 125/562
Progression : 150/562
Progression : 175/562
Progression : 200/562
Progression : 225/562
Progression : 250/562
Progression : 275/562
Progression : 300/562
Progression : 325/562
Progression : 350/562
Progression : 375/562
Progression : 400/562
Progression : 425/562
Progression : 450/562
Progression : 475/562
Progression : 500/562
Progression : 525/562
Progression : 550/562
Progression : 562/562

✅ ZOO Train terminé.


In [42]:
# ============================================================
# ÉTAPE 30B — CONTRÔLE RAPIDE DE LA BANQUE HSJ
# ============================================================

hsj_train_df = pd.read_csv(
    HSJ_TRAIN_RESULTS_FILE
)

print("=" * 70)
print("CONTRÔLE FINAL HSJ TRAIN")
print("=" * 70)

print("Résultats enregistrés :", len(hsj_train_df))

errors = (
    hsj_train_df["Error"]
    .fillna("")
    .ne("")
    .sum()
)

print("Erreurs :", errors)

print("\nRésumé par modèle :")

display(
    hsj_train_df
    .groupby("Model")
    .agg(
        Attaques=("Source_Position", "count"),
        Succes_bruts=("Raw_Success", "sum"),
        Succes_locaux=("Local_Success", "sum"),
        Linf_moyen=("Linf", "mean"),
        Temps_moyen_s=("Time_s", "mean")
    )
)

print(
    "\nSuccès locaux totaux :",
    int(hsj_train_df["Local_Success"].sum())
)

assert len(hsj_train_df) == 1405
assert errors == 0

print("\n✅ Banque HSJ Train complète et exploitable.")

CONTRÔLE FINAL HSJ TRAIN
Résultats enregistrés : 1405
Erreurs : 0

Résumé par modèle :


,Attaques,Succes_bruts,Succes_locaux,Linf_moyen,Temps_moyen_s
Model,,,,,
LightGBM,281,281,129,0.082918,1.263153
Random Forest,281,281,89,0.088567,31.161097
Régression logistique,281,281,72,0.105868,0.834090
SVM linéaire,281,281,95,0.090195,0.808497
XGBoost,281,281,97,0.090884,2.503355



Succès locaux totaux : 482

✅ Banque HSJ Train complète et exploitable.


In [41]:
# ============================================================
# ÉTAPE 30 — BANQUE HSJ TRAIN COMPLÈTE
# 281 FRAUDES × 5 MODÈLES
# ============================================================

HSJ_TRAIN_DIR = (
    BANK_DIR /
    "hsj_train"
)

HSJ_TRAIN_ADV_DIR = (
    HSJ_TRAIN_DIR /
    "adversarial_samples"
)

HSJ_TRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HSJ_TRAIN_ADV_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HSJ_TRAIN_RESULTS_FILE = (
    HSJ_TRAIN_DIR /
    "hsj_train_results.csv"
)


MODEL_SLUG = {

    "XGBoost":
        "xgboost",

    "Random Forest":
        "random_forest",

    "LightGBM":
        "lightgbm",

    "SVM linéaire":
        "svm_lineaire",

    "Régression logistique":
        "regression_logistique"
}


if HSJ_TRAIN_RESULTS_FILE.exists():

    hsj_train_df = pd.read_csv(
        HSJ_TRAIN_RESULTS_FILE
    )

else:

    hsj_train_df = pd.DataFrame()


if len(hsj_train_df) > 0:

    if "Error" not in hsj_train_df.columns:
        hsj_train_df["Error"] = ""

    completed_hsj = {

        (
            row["Model"],
            int(
                row[
                    "Source_Position"
                ]
            )
        )

        for _, row
        in hsj_train_df[
            hsj_train_df[
                "Error"
            ]
            .fillna("")
            .eq("")
        ].iterrows()
    }

else:

    completed_hsj = set()


TOTAL_HSJ = (
    len(
        common_train_fraud_positions
    )
    *
    len(models_original)
)

print(
    "Déjà terminées :",
    len(completed_hsj),
    "/",
    TOTAL_HSJ
)


done_counter = len(
    completed_hsj
)


for source_position in common_train_fraud_positions:

    for model_name, model in models_original.items():

        pair = (
            model_name,
            int(source_position)
        )

        if pair in completed_hsj:
            continue

        try:

            result, x_adv = (
                run_hsj_train_attack(
                    model_name,
                    model,
                    source_position
                )
            )

            result["Error"] = ""

            if x_adv is not None:

                sample_file = (
                    HSJ_TRAIN_ADV_DIR
                    /
                    (
                        f"{MODEL_SLUG[model_name]}"
                        f"_train_{source_position}.npy"
                    )
                )

                np.save(
                    sample_file,
                    x_adv
                )

        except Exception as e:

            result = {

                "Model":
                    model_name,

                "Source_Position":
                    int(source_position),

                "Error":
                    str(e)
            }

        # supprimer ancienne tentative éventuelle
        if len(hsj_train_df) > 0:

            same_pair = (
                (
                    hsj_train_df["Model"]
                    ==
                    model_name
                )
                &
                (
                    hsj_train_df[
                        "Source_Position"
                    ].astype(int)
                    ==
                    int(source_position)
                )
            )

            hsj_train_df = (
                hsj_train_df[
                    ~same_pair
                ]
            )

        hsj_train_df = pd.concat(
            [
                hsj_train_df,
                pd.DataFrame(
                    [result]
                )
            ],
            ignore_index=True
        )

        hsj_train_df.to_csv(
            HSJ_TRAIN_RESULTS_FILE,
            index=False
        )

        if result.get(
            "Error",
            ""
        ) == "":

            completed_hsj.add(
                pair
            )

            done_counter += 1

        if (
            done_counter % 25 == 0
            or
            done_counter == TOTAL_HSJ
        ):

            print(
                f"Progression : "
                f"{done_counter}/{TOTAL_HSJ}"
            )


print(
    "\n✅ HSJ Train terminé."
)

Déjà terminées : 0 / 1405
Progression : 25/1405
Progression : 50/1405
Progression : 75/1405
Progression : 100/1405
Progression : 125/1405
Progression : 150/1405
Progression : 175/1405
Progression : 200/1405
Progression : 225/1405
Progression : 250/1405
Progression : 275/1405
Progression : 300/1405
Progression : 325/1405
Progression : 350/1405
Progression : 375/1405
Progression : 400/1405
Progression : 425/1405
Progression : 450/1405
Progression : 475/1405
Progression : 500/1405
Progression : 525/1405
Progression : 550/1405
Progression : 575/1405
Progression : 600/1405
Progression : 625/1405
Progression : 650/1405
Progression : 675/1405
Progression : 700/1405
Progression : 725/1405
Progression : 750/1405
Progression : 775/1405
Progression : 800/1405
Progression : 825/1405
Progression : 850/1405
Progression : 875/1405
Progression : 900/1405
Progression : 925/1405
Progression : 950/1405
Progression : 975/1405
Progression : 1000/1405
Progression : 1025/1405
Progression : 1050/1405
Progress

In [40]:
# ============================================================
# ÉTAPE 29 — SAUVEGARDE PILOTES
# ============================================================

PILOT_DIR = (
    NB11_DIR /
    "pilots"
)

PILOT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

hsj_pilot_df.to_csv(
    PILOT_DIR /
    "hsj_train_pilot.csv",
    index=False
)

zoo_pilot_df.to_csv(
    PILOT_DIR /
    "zoo_train_pilot.csv",
    index=False
)

print(
    "✅ Pilotes sauvegardés."
)

✅ Pilotes sauvegardés.


In [39]:
# ============================================================
# ÉTAPE 28B — PILOTE ZOO TRAIN
# ============================================================

ZOO_TRAIN_MODELS = [
    "Régression logistique",
    "SVM linéaire"
]

zoo_pilot_results = []


for source_position in HSJ_PILOT_POSITIONS:

    print(
        "\nTransaction :",
        source_position
    )

    for model_name in ZOO_TRAIN_MODELS:

        model = models_original[
            model_name
        ]

        print(
            "   →",
            model_name,
            end=" ... "
        )

        result, _ = (
            run_zoo_train_attack(
                model_name,
                model,
                source_position
            )
        )

        zoo_pilot_results.append(
            result
        )

        print(
            "Success:",
            result[
                "Attack_Success"
            ],
            "| Linf:",
            round(
                result["Linf"],
                5
            )
        )


zoo_pilot_df = pd.DataFrame(
    zoo_pilot_results
)

display(
    zoo_pilot_df
)

assert len(zoo_pilot_df) == 10

print(
    "\n✅ Pilote ZOO terminé."
)


Transaction : 727
   → Régression logistique ... Success: 0 | Linf: 0.0
   → SVM linéaire ... Success: 1 | Linf: 0.15521

Transaction : 1317
   → Régression logistique ... Success: 0 | Linf: 0.0
   → SVM linéaire ... Success: 1 | Linf: 0.15391

Transaction : 1870
   → Régression logistique ... Success: 1 | Linf: 0.02005
   → SVM linéaire ... Success: 1 | Linf: 0.00489

Transaction : 3214
   → Régression logistique ... Success: 1 | Linf: 0.06014
   → SVM linéaire ... Success: 1 | Linf: 0.04012

Transaction : 3485
   → Régression logistique ... Success: 1 | Linf: 0.07337
   → SVM linéaire ... Success: 1 | Linf: 0.03929


,Model,Source_Position,Clean_Pred,Adv_Pred,Attack_Success,L2,Linf,Queries,Time_s
0,Régression logistique,727,1,1,0,0.000000,0.000000,11591,10.820483
1,SVM linéaire,727,1,0,1,0.360591,0.155210,12811,10.391362
2,Régression logistique,1317,1,1,0,0.000000,0.000000,11591,8.899199
3,SVM linéaire,1317,1,0,1,0.357322,0.153907,12811,10.040761
4,Régression logistique,1870,1,0,1,0.031332,0.020046,12811,9.443644
5,SVM linéaire,1870,1,0,1,0.009720,0.004894,12811,9.243321
6,Régression logistique,3214,1,0,1,0.095506,0.060140,12811,8.888706
7,SVM linéaire,3214,1,0,1,0.081645,0.040124,12811,9.071345
8,Régression logistique,3485,1,0,1,0.130906,0.073373,12811,9.286534
9,SVM linéaire,3485,1,0,1,0.079897,0.039289,12811,9.207139



✅ Pilote ZOO terminé.


In [38]:
# ============================================================
# ÉTAPE 28 — ZOO TRAIN : UNE TRANSACTION
# ============================================================

def run_zoo_train_attack(
    model_name,
    model,
    source_position
):

    source_position = int(
        source_position
    )

    x_clean = (
        X_train
        .iloc[source_position]
        .to_numpy(
            dtype=np.float32
        )
    )

    clean_df = pd.DataFrame(
        [x_clean],
        columns=FEATURE_COLUMNS
    )

    clean_pred = int(
        model.predict(clean_df)[0]
    )

    if clean_pred != 1:
        raise ValueError(
            "Fraude source non détectée "
            "avant ZOO."
        )

    classifier, counter = (
        build_blackbox_classifier(
            model
        )
    )

    attack = ZooAttack(
        classifier=classifier,
        **ZOO_CONFIG_TRAIN
    )

    target = np.array(
        [0],
        dtype=np.int64
    )

    counter.reset()

    start = time.perf_counter()

    x_adv = attack.generate(
        x=x_clean.reshape(
            1,
            -1
        ),
        y=target
    )[0]

    elapsed = (
        time.perf_counter()
        -
        start
    )

    queries = int(
        counter.queries
    )

    adv_df = pd.DataFrame(
        [x_adv],
        columns=FEATURE_COLUMNS
    )

    adv_pred = int(
        model.predict(
            adv_df
        )[0]
    )

    delta = (
        x_adv.astype(np.float64)
        -
        x_clean.astype(np.float64)
    )

    l2 = float(
        np.linalg.norm(delta)
    )

    linf = float(
        np.max(
            np.abs(delta)
        )
    )

    success = int(
        clean_pred == 1
        and
        adv_pred == 0
    )

    result = {

        "Model":
            model_name,

        "Source_Position":
            source_position,

        "Clean_Pred":
            clean_pred,

        "Adv_Pred":
            adv_pred,

        "Attack_Success":
            success,

        "L2":
            l2,

        "Linf":
            linf,

        "Queries":
            queries,

        "Time_s":
            elapsed
    }

    return result, x_adv

In [37]:
# ============================================================
# ÉTAPE 27B — CONTRÔLE PILOTE HSJ
# ============================================================

print("=" * 75)
print("CONTRÔLE PILOTE HSJ")
print("=" * 75)

print(
    "Attaques :",
    len(hsj_pilot_df),
    "/ 25"
)

print(
    "Seeds trouvés :",
    int(
        hsj_pilot_df[
            "Seed_Found"
        ].sum()
    ),
    "/ 25"
)

print(
    "Modification max Time/Amount :",
    hsj_pilot_df[
        "Max_Immutable_Change"
    ].max()
)

print(
    "\nSuccès locaux par modèle :"
)

display(
    hsj_pilot_df
    .groupby("Model")
    .agg(
        Attacks=(
            "Source_Position",
            "count"
        ),
        Raw_Success=(
            "Raw_Success",
            "sum"
        ),
        Local_Success=(
            "Local_Success",
            "sum"
        ),
        Mean_Linf=(
            "Linf",
            "mean"
        )
    )
)


assert len(hsj_pilot_df) == 25

assert (
    hsj_pilot_df[
        "Seed_Found"
    ].sum()
    ==
    25
)

assert (
    hsj_pilot_df[
        "Max_Immutable_Change"
    ].max()
    <=
    1e-8
)

print(
    "\n✅ Pilote HSJ techniquement valide."
)

CONTRÔLE PILOTE HSJ
Attaques : 25 / 25
Seeds trouvés : 25 / 25
Modification max Time/Amount : 0.0

Succès locaux par modèle :


,Attacks,Raw_Success,Local_Success,Mean_Linf
Model,,,,
LightGBM,5,5,3,0.064152
Random Forest,5,5,2,0.060768
Régression logistique,5,5,3,0.077126
SVM linéaire,5,5,3,0.060022
XGBoost,5,5,3,0.049940



✅ Pilote HSJ techniquement valide.


In [36]:
# ============================================================
# ÉTAPE 27 — PILOTE HSJ TRAIN
# 5 FRAUDES × 5 MODÈLES = 25 ATTAQUES
# ============================================================

HSJ_PILOT_POSITIONS = (
    common_train_fraud_positions[:5]
)

hsj_pilot_results = []


for source_position in HSJ_PILOT_POSITIONS:

    print(
        "\nTransaction :",
        source_position
    )

    for model_name, model in models_original.items():

        print(
            "   →",
            model_name,
            end=" ... "
        )

        result, _ = (
            run_hsj_train_attack(
                model_name,
                model,
                source_position
            )
        )

        hsj_pilot_results.append(
            result
        )

        print(
            "Raw:",
            result["Raw_Success"],
            "| Local:",
            result["Local_Success"],
            "| Linf:",
            round(
                result["Linf"],
                5
            )
            if not pd.isna(
                result["Linf"]
            )
            else "NA"
        )


hsj_pilot_df = pd.DataFrame(
    hsj_pilot_results
)

display(
    hsj_pilot_df
)


Transaction : 727
   → Régression logistique ... Raw: 1 | Local: 0 | Linf: 0.15553
   → SVM linéaire ... Raw: 1 | Local: 0 | Linf: 0.1319
   → Random Forest ... Raw: 1 | Local: 0 | Linf: 0.14962
   → XGBoost ... Raw: 1 | Local: 0 | Linf: 0.12817
   → LightGBM ... Raw: 1 | Local: 0 | Linf: 0.17958

Transaction : 1317
   → Régression logistique ... Raw: 1 | Local: 0 | Linf: 0.14836
   → SVM linéaire ... Raw: 1 | Local: 0 | Linf: 0.11341
   → Random Forest ... Raw: 1 | Local: 0 | Linf: 0.07808
   → XGBoost ... Raw: 1 | Local: 0 | Linf: 0.06686
   → LightGBM ... Raw: 1 | Local: 0 | Linf: 0.10258

Transaction : 1870
   → Régression logistique ... Raw: 1 | Local: 1 | Linf: 0.00998
   → SVM linéaire ... Raw: 1 | Local: 1 | Linf: 0.00314
   → Random Forest ... Raw: 1 | Local: 1 | Linf: 0.00664
   → XGBoost ... Raw: 1 | Local: 1 | Linf: 0.00599
   → LightGBM ... Raw: 1 | Local: 1 | Linf: 0.00427

Transaction : 3214
   → Régression logistique ... Raw: 1 | Local: 1 | Linf: 0.03021
   → SVM linéa

,Model,Source_Position,Seed_Found,Seed_Position,Clean_Pred,Adv_Pred,Raw_Success,Local_Success,L2,Linf,Max_Immutable_Change,Queries,Time_s
0,Régression logistique,727,1,163708,1,0,1,0,0.743037,0.155530,0.0,24536,0.797072
1,SVM linéaire,727,1,51754,1,0,1,0,0.605782,0.131903,0.0,24536,0.806011
2,Random Forest,727,1,164976,1,0,1,0,0.520442,0.149615,0.0,24605,30.285055
3,XGBoost,727,1,113845,1,0,1,0,0.428342,0.128166,0.0,24601,2.535935
4,LightGBM,727,1,74877,1,0,1,0,0.568691,0.179585,0.0,24640,1.385966
5,Régression logistique,1317,1,64943,1,0,1,0,0.726041,0.148357,0.0,24536,0.807326
6,SVM linéaire,1317,1,124536,1,0,1,0,0.547934,0.113411,0.0,24536,0.774211
7,Random Forest,1317,1,72208,1,0,1,0,0.274611,0.078082,0.0,24645,31.976119
8,XGBoost,1317,1,18722,1,0,1,0,0.253200,0.066860,0.0,24630,2.651085
9,LightGBM,1317,1,55486,1,0,1,0,0.273863,0.102577,0.0,24573,1.199619


In [35]:
# ============================================================
# ÉTAPE 26 — HSJ TRAIN : UNE TRANSACTION
# ============================================================

import time


def run_hsj_train_attack(
    model_name,
    model,
    source_position
):

    source_position = int(
        source_position
    )

    x_clean = (
        X_train
        .iloc[source_position]
        .to_numpy(
            dtype=np.float32
        )
    )

    clean_df = pd.DataFrame(
        [x_clean],
        columns=FEATURE_COLUMNS
    )

    clean_pred = int(
        model.predict(clean_df)[0]
    )

    if clean_pred != 1:
        raise ValueError(
            "La fraude source n'est pas "
            "correctement détectée."
        )

    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    x_init, seed_position = (
        find_hsj_train_seed(
            model_name,
            model,
            source_position
        )
    )

    if x_init is None:

        return {
            "Model": model_name,
            "Source_Position":
                source_position,
            "Seed_Found": 0,
            "Seed_Position":
                np.nan,
            "Clean_Pred":
                clean_pred,
            "Adv_Pred":
                np.nan,
            "Raw_Success":
                0,
            "Local_Success":
                0,
            "L2":
                np.nan,
            "Linf":
                np.nan,
            "Max_Immutable_Change":
                np.nan,
            "Queries":
                np.nan,
            "Time_s":
                np.nan
        }, None

    # --------------------------------------------------------
    # Black-box
    # --------------------------------------------------------

    classifier, counter = (
        build_blackbox_classifier(
            model
        )
    )

    attack = HopSkipJump(
        classifier=classifier,
        **HSJ_CONFIG_TRAIN
    )

    target = np.array(
        [0],
        dtype=np.int64
    )

    counter.reset()

    start = time.perf_counter()

    x_adv = attack.generate(
        x=x_clean.reshape(1, -1),
        y=target,
        mask=HSJ_MASK,
        x_adv_init=x_init.reshape(
            1,
            -1
        )
    )[0]

    elapsed = (
        time.perf_counter()
        -
        start
    )

    queries = int(
        counter.queries
    )

    # --------------------------------------------------------
    # Évaluation
    # --------------------------------------------------------

    adv_df = pd.DataFrame(
        [x_adv],
        columns=FEATURE_COLUMNS
    )

    adv_pred = int(
        model.predict(adv_df)[0]
    )

    delta = (
        x_adv.astype(np.float64)
        -
        x_clean.astype(np.float64)
    )

    l2 = float(
        np.linalg.norm(
            delta,
            ord=2
        )
    )

    linf = float(
        np.max(
            np.abs(delta)
        )
    )

    immutable_change = float(
        np.max(
            np.abs(
                delta[
                    IMMUTABLE_IDX
                ]
            )
        )
    )

    raw_success = int(
        clean_pred == 1
        and
        adv_pred == 0
    )

    local_success = int(

        raw_success == 1

        and

        linf
        <=
        HSJ_LOCAL_EPS
        + 1e-8

        and

        immutable_change
        <=
        1e-8
    )

    result = {

        "Model":
            model_name,

        "Source_Position":
            source_position,

        "Seed_Found":
            1,

        "Seed_Position":
            seed_position,

        "Clean_Pred":
            clean_pred,

        "Adv_Pred":
            adv_pred,

        "Raw_Success":
            raw_success,

        "Local_Success":
            local_success,

        "L2":
            l2,

        "Linf":
            linf,

        "Max_Immutable_Change":
            immutable_change,

        "Queries":
            queries,

        "Time_s":
            elapsed
    }

    return result, x_adv

In [34]:
# ============================================================
# ÉTAPE 25B — TEST DES SEEDS HSJ
# ============================================================

seed_test_position = (
    common_train_fraud_positions[0]
)

for model_name, model in models_original.items():

    seed, normal_pos = (
        find_hsj_train_seed(
            model_name,
            model,
            seed_test_position
        )
    )

    assert seed is not None

    seed_df = pd.DataFrame(
        [seed],
        columns=FEATURE_COLUMNS
    )

    seed_prediction = int(
        model.predict(seed_df)[0]
    )

    print(
        f"{model_name:25s} "
        f"→ seed normal={normal_pos}, "
        f"prediction={seed_prediction}"
    )

    assert seed_prediction == 0


print(
    "\n✅ Seeds HSJ Train valides."
)

Régression logistique     → seed normal=163708, prediction=0
SVM linéaire              → seed normal=51754, prediction=0
Random Forest             → seed normal=164976, prediction=0
XGBoost                   → seed normal=113845, prediction=0
LightGBM                  → seed normal=74877, prediction=0

✅ Seeds HSJ Train valides.


In [33]:
# ============================================================
# ÉTAPE 25 — GÉNÉRATION D'UN SEED HSJ TRAIN
# ============================================================

TRAIN_NORMAL_POSITIONS = np.where(
    y_train.to_numpy() == 0
)[0]


MODEL_ORDER = list(
    models_original.keys()
)

MODEL_INDEX = {
    name: i
    for i, name
    in enumerate(MODEL_ORDER)
}


def find_hsj_train_seed(
    model_name,
    model,
    source_position,
    max_trials=5000
):

    source_position = int(
        source_position
    )

    x_clean = (
        X_train
        .iloc[source_position]
        .to_numpy(
            dtype=np.float32
        )
    )

    # Seed déterministe
    local_seed = (
        2026
        +
        source_position
        +
        MODEL_INDEX[model_name]
        * 100000
    )

    rng = np.random.default_rng(
        local_seed
    )

    n_trials = min(
        max_trials,
        len(TRAIN_NORMAL_POSITIONS)
    )

    candidates = rng.choice(
        TRAIN_NORMAL_POSITIONS,
        size=n_trials,
        replace=False
    )

    # Répéter la fraude
    candidate_seeds = np.repeat(
        x_clean.reshape(1, -1),
        n_trials,
        axis=0
    )

    # Transplanter uniquement V1...V28
    normal_mutable_values = (
        X_train
        .iloc[candidates][
            HSJ_MUTABLE_FEATURES
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    candidate_seeds[
        :,
        MUTABLE_IDX
    ] = normal_mutable_values

    # Time / Amount restent ceux de la fraude
    seed_df = pd.DataFrame(
        candidate_seeds,
        columns=FEATURE_COLUMNS
    )

    predictions = (
        model.predict(seed_df)
    )

    valid_indices = np.where(
        predictions == 0
    )[0]

    if len(valid_indices) == 0:
        return None, None

    first_valid = int(
        valid_indices[0]
    )

    seed = (
        candidate_seeds[
            first_valid
        ]
        .copy()
    )

    seed_position = int(
        candidates[first_valid]
    )

    # Contrôle immutables
    immutable_change = np.max(
        np.abs(
            seed[IMMUTABLE_IDX]
            -
            x_clean[IMMUTABLE_IDX]
        )
    )

    assert immutable_change <= 1e-8

    return seed, seed_position

In [32]:
# ============================================================
# ÉTAPE 24 — CONFIGURATIONS FIGÉES ZOO / HSJ
# ============================================================

from art.attacks.evasion import (
    ZooAttack,
    HopSkipJump
)

# ------------------------------------------------------------
# ZOO — protocole Notebook 10
# ------------------------------------------------------------

ZOO_CONFIG_TRAIN = {

    "confidence": 0.5,
    "targeted": True,

    "learning_rate": 1e-3,

    "max_iter": 200,

    "binary_search_steps": 10,

    "initial_const": 0.001,

    "abort_early": True,

    "nb_parallel":
        len(FEATURE_COLUMNS),

    "batch_size": 1,

    "verbose": False
}


# ------------------------------------------------------------
# HSJ — protocole local Notebook 10
# ------------------------------------------------------------

HSJ_LOCAL_EPS = 0.05

HSJ_CONFIG_TRAIN = {

    "targeted": True,

    "norm": np.inf,

    "max_iter": 50,

    "max_eval": 10000,

    "init_eval": 100,

    "init_size": 100,

    "batch_size": 64,

    "verbose": False
}


HSJ_MUTABLE_FEATURES = [
    col
    for col in FEATURE_COLUMNS
    if col.startswith("V")
]

HSJ_IMMUTABLE_FEATURES = [
    "Time",
    "Amount"
]


assert len(HSJ_MUTABLE_FEATURES) == 28
assert len(HSJ_IMMUTABLE_FEATURES) == 2


MUTABLE_IDX = [
    FEATURE_COLUMNS.index(col)
    for col in HSJ_MUTABLE_FEATURES
]

IMMUTABLE_IDX = [
    FEATURE_COLUMNS.index(col)
    for col in HSJ_IMMUTABLE_FEATURES
]


HSJ_MASK = np.array(
    [
        1.0
        if col in HSJ_MUTABLE_FEATURES
        else 0.0

        for col
        in FEATURE_COLUMNS
    ],
    dtype=np.float32
).reshape(1, -1)


print("ZOO : configuration figée.")
print(
    "HSJ : L∞ local <=",
    HSJ_LOCAL_EPS
)

print(
    "Variables HSJ modifiables :",
    len(HSJ_MUTABLE_FEATURES)
)

print(
    "Variables HSJ figées :",
    HSJ_IMMUTABLE_FEATURES
)

ZOO : configuration figée.
HSJ : L∞ local <= 0.05
Variables HSJ modifiables : 28
Variables HSJ figées : ['Time', 'Amount']


In [31]:
# ============================================================
# ÉTAPE 23B — CONTRÔLE WRAPPER / MODÈLE ORIGINAL
# ============================================================

wrapper_check_positions = (
    np.arange(
        min(
            100,
            len(X_test)
        )
    )
)

X_wrapper_check = (
    X_test
    .iloc[wrapper_check_positions]
)

for model_name, model in models_original.items():

    classifier, counter = (
        build_blackbox_classifier(
            model
        )
    )

    original_pred = (
        model.predict(
            X_wrapper_check
        )
    )

    blackbox_scores = (
        classifier.predict(
            X_wrapper_check
            .to_numpy(
                dtype=np.float32
            )
        )
    )

    blackbox_pred = (
        np.argmax(
            blackbox_scores,
            axis=1
        )
    )

    identical = np.array_equal(
        original_pred,
        blackbox_pred
    )

    print(
        f"{model_name:25s} → "
        f"{'✅ identique' if identical else '❌ différent'}"
    )

    assert identical


print(
    "\n✅ Wrapper black-box compatible "
    "avec les 5 modèles."
)

Régression logistique     → ✅ identique
SVM linéaire              → ✅ identique
Random Forest             → ✅ identique
XGBoost                   → ✅ identique
LightGBM                  → ✅ identique

✅ Wrapper black-box compatible avec les 5 modèles.


In [30]:
# ============================================================
# ÉTAPE 23 — WRAPPER BLACK-BOX + COMPTEUR DE REQUÊTES
# ============================================================

import numpy as np
import pandas as pd

from art.estimators.classification import BlackBoxClassifier


class QueryCountingPredictor:

    def __init__(
        self,
        model,
        feature_columns
    ):
        self.model = model
        self.feature_columns = list(feature_columns)
        self.queries = 0

    def reset(self):
        self.queries = 0

    def __call__(self, x):

        x = np.asarray(x)

        if x.ndim == 1:
            x = x.reshape(1, -1)

        self.queries += len(x)

        X_df = pd.DataFrame(
            x,
            columns=self.feature_columns
        )

        # ---------------------------------------
        # Modèles avec predict_proba
        # LR / RF / XGB / LGBM
        # ---------------------------------------

        if hasattr(
            self.model,
            "predict_proba"
        ):

            proba = np.asarray(
                self.model.predict_proba(X_df)
            )

            classes = list(
                self.model.classes_
            )

            idx_0 = classes.index(0)
            idx_1 = classes.index(1)

            scores = np.column_stack([
                proba[:, idx_0],
                proba[:, idx_1]
            ])

            return scores.astype(
                np.float32
            )

        # ---------------------------------------
        # LinearSVC
        # ---------------------------------------

        if hasattr(
            self.model,
            "decision_function"
        ):

            decision = np.asarray(
                self.model.decision_function(
                    X_df
                )
            ).reshape(-1)

            # Conversion stable en scores [P0, P1]
            decision = np.clip(
                decision,
                -50,
                50
            )

            p1 = (
                1.0
                /
                (
                    1.0
                    +
                    np.exp(-decision)
                )
            )

            p0 = 1.0 - p1

            return np.column_stack([
                p0,
                p1
            ]).astype(np.float32)

        # ---------------------------------------
        # Fallback
        # ---------------------------------------

        pred = np.asarray(
            self.model.predict(X_df),
            dtype=int
        )

        scores = np.zeros(
            (len(pred), 2),
            dtype=np.float32
        )

        scores[
            np.arange(len(pred)),
            pred
        ] = 1.0

        return scores


def build_blackbox_classifier(model):

    counter = QueryCountingPredictor(
        model=model,
        feature_columns=FEATURE_COLUMNS
    )

    classifier = BlackBoxClassifier(
        predict_fn=counter,
        input_shape=(
            len(FEATURE_COLUMNS),
        ),
        nb_classes=2
    )

    return classifier, counter

In [29]:
# ============================================================
# ÉTAPE 22 — GEL DU PANEL TRAIN COMMUN
# ============================================================

common_train_fraud_positions = [
    int(pos)
    for pos in common_train_fraud_positions
]

print("=" * 75)
print("PANEL TRAIN COMMUN — DÉFENSE")
print("=" * 75)

print(
    "Fraudes réelles Train :",
    int((y_train == 1).sum())
)

print(
    "Fraudes détectées par les 5 modèles :",
    len(common_train_fraud_positions)
)

print(
    "Premières positions :",
    common_train_fraud_positions[:10]
)

assert len(common_train_fraud_positions) == 281

assert all(
    int(y_train.iloc[pos]) == 1
    for pos in common_train_fraud_positions
)

COMMON_TRAIN_FILE = (
    NB11_DIR /
    "common_train_fraud_positions.npy"
)

np.save(
    COMMON_TRAIN_FILE,
    np.asarray(
        common_train_fraud_positions,
        dtype=np.int64
    )
)

print(
    "\n✅ Panel de 281 fraudes Train gelé."
)
print(COMMON_TRAIN_FILE.resolve())

PANEL TRAIN COMMUN — DÉFENSE
Fraudes réelles Train : 331
Fraudes détectées par les 5 modèles : 281
Premières positions : [727, 1317, 1870, 3214, 3485, 5498, 5653, 6296, 6460, 6592]

✅ Panel de 281 fraudes Train gelé.
C:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training\common_train_fraud_positions.npy


In [28]:
# Fraudes correctement détectées par les 5 modèles
common_train_fraud_positions = sorted(
    set.intersection(
        *[
            set(v)
            for v in train_tp_by_model.values()
        ]
    )
)

print(
    "\nFraudes Train correctement détectées "
    "par les 5 modèles :",
    len(common_train_fraud_positions)
)

print(
    "Premières positions :",
    common_train_fraud_positions[:10]
)


Fraudes Train correctement détectées par les 5 modèles : 281
Premières positions : [np.int64(727), np.int64(1317), np.int64(1870), np.int64(3214), np.int64(3485), np.int64(5498), np.int64(5653), np.int64(6296), np.int64(6460), np.int64(6592)]


In [27]:
# ============================================================
# ÉTAPE 21 — TRUE POSITIVES TRAIN PAR MODÈLE
# ============================================================

train_tp_by_model = {}

for model_name, model in models_original.items():

    predictions = model.predict(
        X_train
    )

    tp_positions = np.where(
        (y_train.to_numpy() == 1)
        &
        (predictions == 1)
    )[0]

    train_tp_by_model[
        model_name
    ] = tp_positions

    print(
        f"{model_name:25s} : "
        f"{len(tp_positions)} fraudes Train correctement détectées"
    )

Régression logistique     : 283 fraudes Train correctement détectées
SVM linéaire              : 283 fraudes Train correctement détectées
Random Forest             : 331 fraudes Train correctement détectées
XGBoost                   : 331 fraudes Train correctement détectées
LightGBM                  : 331 fraudes Train correctement détectées


In [26]:
# ============================================================
# ÉTAPE 20 — FRAUDES RÉELLES DU TRAIN
# ============================================================

train_fraud_positions = np.where(
    y_train.to_numpy() == 1
)[0]

X_train_real_fraud = (
    X_train.iloc[
        train_fraud_positions
    ]
    .copy()
)

print("=" * 70)
print("FRAUDES RÉELLES DU TRAIN")
print("=" * 70)

print(
    "Nombre :",
    len(X_train_real_fraud)
)

print(
    "Shape :",
    X_train_real_fraud.shape
)

assert (
    len(X_train_real_fraud)
    ==
    int((y_train == 1).sum())
)

print("\n✅ Sources adversariales = fraudes réelles Train uniquement.")

FRAUDES RÉELLES DU TRAIN
Nombre : 331
Shape : (331, 30)

✅ Sources adversariales = fraudes réelles Train uniquement.


In [25]:
# ============================================================
# ÉTAPE 19B — COMPARAISON AVEC LA BASELINE NOTEBOOK 10
# VERSION CORRIGÉE
# ============================================================

print("Colonnes disponibles :")
print(metrics_before.columns.tolist())

print("\nConditions disponibles :")
print(metrics_before["Condition"].unique())

# ------------------------------------------------------------
# Récupérer uniquement les performances CLEAN de Notebook 10
# ------------------------------------------------------------

clean_nb10 = (
    metrics_before[
        metrics_before["Condition"] == "Clean"
    ]
    .copy()
)

print(
    "\nNombre de lignes Clean trouvées :",
    len(clean_nb10)
)

# ------------------------------------------------------------
# Comparaison avec les métriques recalculées
# ------------------------------------------------------------

comparison_clean = (
    clean_recomputed
    .merge(
        clean_nb10[
            [
                "Model",
                "Recall",
                "F1",
                "FN",
                "TP"
            ]
        ],
        on="Model",
        suffixes=(
            "_Recomputed",
            "_Notebook10"
        )
    )
)

comparison_clean["Recall_Diff"] = (
    comparison_clean["Recall_Recomputed"]
    -
    comparison_clean["Recall_Notebook10"]
)

comparison_clean["F1_Diff"] = (
    comparison_clean["F1_Recomputed"]
    -
    comparison_clean["F1_Notebook10"]
)

comparison_clean["FN_Diff"] = (
    comparison_clean["FN_Recomputed"]
    -
    comparison_clean["FN_Notebook10"]
)

comparison_clean["TP_Diff"] = (
    comparison_clean["TP_Recomputed"]
    -
    comparison_clean["TP_Notebook10"]
)

display(
    comparison_clean.round(8)
)

# ------------------------------------------------------------
# Vérification stricte
# ------------------------------------------------------------

max_recall_diff = comparison_clean[
    "Recall_Diff"
].abs().max()

max_f1_diff = comparison_clean[
    "F1_Diff"
].abs().max()

max_fn_diff = comparison_clean[
    "FN_Diff"
].abs().max()

max_tp_diff = comparison_clean[
    "TP_Diff"
].abs().max()

print("\nÉcarts maximums :")
print("Recall :", max_recall_diff)
print("F1     :", max_f1_diff)
print("FN     :", max_fn_diff)
print("TP     :", max_tp_diff)

assert max_recall_diff < 1e-8
assert max_f1_diff < 1e-8
assert max_fn_diff == 0
assert max_tp_diff == 0

print(
    "\n✅ Continuité parfaite avec Notebook 10."
)

Colonnes disponibles :
['Model', 'Condition', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'TN', 'FP', 'FN', 'TP']

Conditions disponibles :
<ArrowStringArray>
['Clean', 'ZOO', 'HSJ']
Length: 3, dtype: str

Nombre de lignes Clean trouvées : 5


,Model,Accuracy,Precision,Recall_Recomputed,F1_Recomputed,ROC_AUC,FN_Recomputed,TP_Recomputed,Recall_Notebook10,F1_Notebook10,FN_Notebook10,TP_Notebook10,Recall_Diff,F1_Diff,FN_Diff,TP_Diff
0,Régression logistique,0.998167,0.472000,0.830986,0.602041,0.947638,12,59,0.830986,0.602041,12,59,0.0,0.0,0,0
1,SVM linéaire,0.998849,0.619565,0.802817,0.699387,0.956374,14,57,0.802817,0.699387,14,57,0.0,0.0,0,0
2,Random Forest,0.999483,0.901639,0.774648,0.833333,0.984725,16,55,0.774648,0.833333,16,55,0.0,0.0,0,0
3,XGBoost,0.999342,0.820896,0.774648,0.797101,0.974825,16,55,0.774648,0.797101,16,55,0.0,0.0,0,0
4,LightGBM,0.999413,0.870968,0.760563,0.812030,0.969766,17,54,0.760563,0.812030,17,54,0.0,0.0,0,0



Écarts maximums :
Recall : 0.0
F1     : 0.0
FN     : 0
TP     : 0

✅ Continuité parfaite avec Notebook 10.


In [23]:
# ============================================================
# ÉTAPE 19 — CONTRÔLE DE CONTINUITÉ NOTEBOOK 10
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def get_score_for_auc(model, X):

    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


rows = []

for model_name, model in models_original.items():

    y_pred = model.predict(X_test)

    y_score = get_score_for_auc(
        model,
        X_test
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_test,
            y_pred
        ),
        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test,
            y_score
        ),
        "FN": int(fn),
        "TP": int(tp)
    })

clean_recomputed = pd.DataFrame(rows)

display(
    clean_recomputed.round(4)
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,FN,TP
0,Régression logistique,0.9982,0.4720,0.8310,0.6020,0.9476,12,59
1,SVM linéaire,0.9988,0.6196,0.8028,0.6994,0.9564,14,57
2,Random Forest,0.9995,0.9016,0.7746,0.8333,0.9847,16,55
3,XGBoost,0.9993,0.8209,0.7746,0.7971,0.9748,16,55
4,LightGBM,0.9994,0.8710,0.7606,0.8120,0.9698,17,54


In [22]:
# ============================================================
# ÉTAPE 18 — CHARGEMENT DES MODÈLES ORIGINAUX
# ============================================================

model_paths = {

    "Régression logistique":
        ORIGINAL_MODELS_DIR
        / "logistic_regression.joblib",

    "SVM linéaire":
        ORIGINAL_MODELS_DIR
        / "linear_svm.joblib",

    "Random Forest":
        ORIGINAL_MODELS_DIR
        / "random_forest.joblib",

    "XGBoost":
        ORIGINAL_MODELS_DIR
        / "xgboost.joblib",

    "LightGBM":
        ORIGINAL_MODELS_DIR
        / "lightgbm.joblib",
}

models_original = {}

print("=" * 70)
print("MODÈLES NOTEBOOK 10")
print("=" * 70)

for model_name, path in model_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{model_name} introuvable : {path}"
        )

    models_original[model_name] = joblib.load(
        path
    )

    print(
        f"✅ {model_name:25s} → chargé"
    )

assert len(models_original) == 5

print("\n✅ 5 modèles originaux chargés.")

MODÈLES NOTEBOOK 10
✅ Régression logistique     → chargé
✅ SVM linéaire              → chargé
✅ Random Forest             → chargé
✅ XGBoost                   → chargé
✅ LightGBM                  → chargé

✅ 5 modèles originaux chargés.


In [21]:
# ============================================================
# ÉTAPE 17 — RECONSTRUCTION EXACTE DU TRAIN AUGMENTÉ
# ============================================================

# Retrouver la cellule contenant la fonction originale
augmentation_function_source = None

for cell in nb9["cells"]:

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if "def build_augmented_train" in source:
        augmentation_function_source = source
        break

if augmentation_function_source is None:
    raise RuntimeError(
        "Fonction build_augmented_train introuvable "
        "dans Notebook 9."
    )

# Exécuter EXACTEMENT la fonction originale
exec(
    augmentation_function_source,
    globals()
)

print("✅ Fonction originale build_augmented_train chargée.")

# ------------------------------------------------------------
# Reconstruction
# ------------------------------------------------------------

X_train_aug, y_train_aug, n_added = (
    build_augmented_train(
        X_train=X_train,
        y_train=y_train,
        synthetic_data=smote_synthetic,
        target_ratio=BEST_RATIO,
        random_state=SEED
    )
)

print("\n" + "=" * 70)
print("TRAIN AUGMENTÉ RECONSTRUIT")
print("=" * 70)

print("Shape :", X_train_aug.shape)

print("\nRépartition :")
print(
    y_train_aug
    .value_counts()
    .sort_index()
)

print(
    "\nFraudes synthétiques ajoutées :",
    n_added
)

ratio_observed = (
    (y_train_aug == 1).sum()
    /
    (y_train_aug == 0).sum()
)

print(
    "Ratio fraude/normale obtenu :",
    round(float(ratio_observed), 4)
)

assert np.isclose(
    ratio_observed,
    BEST_RATIO,
    atol=1e-4
)

print("\n✅ X_train_aug reconstruit correctement.")

✅ Fonction originale build_augmented_train chargée.

TRAIN AUGMENTÉ RECONSTRUIT
Shape : (218105, 30)

Répartition :
Class
0    198277
1     19828
Name: count, dtype: int64

Fraudes synthétiques ajoutées : 19497
Ratio fraude/normale obtenu : 0.1

✅ X_train_aug reconstruit correctement.


In [20]:
# ============================================================
# ÉTAPE 16 — CHARGEMENT DES FRAUDES SYNTHÉTIQUES SMOTE
# ============================================================

SMOTE_FILE = (
    SYNTHETIC_DIR
    / "smote_fraud_synthetic.parquet"
)

if not SMOTE_FILE.exists():
    raise FileNotFoundError(
        f"Fichier SMOTE introuvable : {SMOTE_FILE}"
    )

smote_synthetic = pd.read_parquet(
    SMOTE_FILE
)

if "Class" in smote_synthetic.columns:
    smote_synthetic = smote_synthetic.drop(
        columns=["Class"]
    )

if list(smote_synthetic.columns) != FEATURE_COLUMNS:
    raise ValueError(
        "Les colonnes SMOTE ne correspondent pas "
        "aux colonnes du Train."
    )

BEST_METHOD = "SMOTE"
BEST_RATIO = 0.10
SEED = 42

print("Méthode :", BEST_METHOD)
print("Ratio cible fraude/normale :", BEST_RATIO)
print("Fraudes SMOTE disponibles :", smote_synthetic.shape)

print("\n✅ Dataset SMOTE chargé.")

Méthode : SMOTE
Ratio cible fraude/normale : 0.1
Fraudes SMOTE disponibles : (197946, 30)

✅ Dataset SMOTE chargé.


In [19]:
# ============================================================
# ÉTAPE 15 — CHARGEMENT OFFICIEL DES DONNÉES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SYNTHETIC_DIR = PROJECT_ROOT / "results" / "synthetic"
ORIGINAL_MODELS_DIR = PROJECT_ROOT / "models" / "augmented"

# ------------------------------------------------------------
# Données réelles
# ------------------------------------------------------------

X_train = pd.read_parquet(
    PROCESSED_DIR / "X_train.parquet"
)

y_train = pd.read_parquet(
    PROCESSED_DIR / "y_train.parquet"
).squeeze()

X_val = pd.read_parquet(
    PROCESSED_DIR / "X_val.parquet"
)

y_val = pd.read_parquet(
    PROCESSED_DIR / "y_val.parquet"
).squeeze()

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test.parquet"
)

y_test = pd.read_parquet(
    PROCESSED_DIR / "y_test.parquet"
).squeeze()

FEATURE_COLUMNS = list(X_train.columns)

print("=" * 70)
print("DONNÉES RÉELLES")
print("=" * 70)

print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

print("\nFraudes :")
print("Train      :", int((y_train == 1).sum()))
print("Validation :", int((y_val == 1).sum()))
print("Test       :", int((y_test == 1).sum()))

print("\nNombre de features :", len(FEATURE_COLUMNS))

assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

print("\n✅ Train / Validation / Test compatibles.")

DONNÉES RÉELLES
Train      : (198608, 30)
Validation : (42559, 30)
Test       : (42559, 30)

Fraudes :
Train      : 331
Validation : 71
Test       : 71

Nombre de features : 30

✅ Train / Validation / Test compatibles.


In [18]:
# ============================================================
# ÉTAPE 14B — EXTRAIRE LES CELLULES CLÉS DU NOTEBOOK 9
# ============================================================

targets = [
    "smote_synthetic = pd.read_parquet",
    "model.fit(",
    "MODELS_DIR =",
]

for i, cell in enumerate(nb9["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(target in source for target in targets):

        print("\n" + "=" * 85)
        print(f"CELLULE {i}")
        print("=" * 85)
        print(source)


CELLULE 18
baseline_test_rows = []

baseline_models = make_models()

for model_name, model in baseline_models.items():

    print(
        "Baseline →",
        model_name
    )

    start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.perf_counter()
        -
        start
    )

    metrics, _, _ = (
        evaluate_classifier(
            model,
            X_test,
            y_test
        )
    )

    baseline_test_rows.append({

        "Modèle":
            model_name,

        "Temps_s":
            training_time,

        **metrics
    })

CELLULE 23
augmented_models = {}

augmented_test_rows = []

test_predictions = pd.DataFrame({
    "y_true":
        y_test
        .reset_index(drop=True)
})

model_file_names = {

    "Régression logistique":
        "logistic_regression",

    "SVM linéaire":
        "linear_svm",

    "Random Forest":
        "random_forest",

    "XGBoost":
        "xgboost",

    "Ligh

In [17]:
# ============================================================
# ÉTAPE 14A — RETROUVER EXACTEMENT LA CONFIGURATION
#              D'AUGMENTATION DU NOTEBOOK 9
# ============================================================

import json
from pathlib import Path

nb9_candidates = list(ROOT.rglob("09_augmented_models.ipynb"))

if len(nb9_candidates) == 0:
    raise FileNotFoundError(
        "09_augmented_models.ipynb introuvable."
    )

nb9_path = nb9_candidates[0]

print("Notebook trouvé :")
print(nb9_path.resolve())

with open(nb9_path, "r", encoding="utf-8") as f:
    nb9 = json.load(f)

keywords = [
    "BEST_RATIO",
    "BEST_METHOD",
    "smote_synthetic",
    "SYNTHETIC_DIR",
    "MODELS_DIR",
    "X_train_aug",
    "y_train_aug",
]

print("\n" + "=" * 85)
print("CONFIGURATION D'AUGMENTATION NOTEBOOK 9")
print("=" * 85)

for i, cell in enumerate(nb9["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(keyword in source for keyword in keywords):

        print(f"\n----- CELLULE {i} -----")

        for line in source.splitlines():
            if any(keyword in line for keyword in keywords):
                print(line)

Notebook trouvé :
C:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\09_augmented_models.ipynb

CONFIGURATION D'AUGMENTATION NOTEBOOK 9

----- CELLULE 1 -----
        MODELS_DIR /

----- CELLULE 3 -----
        BEST_METHOD,
        BEST_RATIO,
        int(len(X_train_aug)),

----- CELLULE 22 -----
    MODELS_DIR /

----- CELLULE 23 -----
        X_train_aug,
        y_train_aug
        MODELS_DIR /

----- CELLULE 27 -----
        X_train_aug
            X_train_aug.to_numpy()
    len(X_train_aug)
    len(y_train_aug)
            y_train_aug == 1
            y_train_aug == 0

----- CELLULE 28 -----
X_train_aug, y_train_aug, n_added = (
        synthetic_data=smote_synthetic,
        target_ratio=BEST_RATIO,
    X_train_aug.shape
    y_train_aug.value_counts()

----- CELLULE 30 -----
    SYNTHETIC_DIR /
smote_synthetic = pd.read_parquet(
if "Class" in smote_synthetic.columns:
    smote_synthetic = (
        smote_synthetic
if list(smote_synthetic.columns) != list(
    smote_synthetic

In [16]:
# ============================================================
# CONTRÔLE CIBLÉ — MODÈLES UTILISÉS PAR NOTEBOOK 10
# ============================================================

import json
from pathlib import Path

nb10_path = Path("10_zoo_attack.ipynb")

with open(nb10_path, "r", encoding="utf-8") as f:
    nb10 = json.load(f)

for i, cell in enumerate(nb10["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if "model_paths" in source or "MODELS_DIR" in source:

        print("\n" + "=" * 80)
        print("CELLULE", i)
        print("=" * 80)
        print(source)


CELLULE 82
models = {}

for model_name, path in model_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Modèle introuvable : {path}"
        )

    models[model_name] = joblib.load(
        path
    )

    print(
        model_name,
        "→ chargé"
    )

CELLULE 83
model_paths = {

    "Régression logistique":
        MODELS_DIR /
        "logistic_regression.joblib",

    "SVM linéaire":
        MODELS_DIR /
        "linear_svm.joblib",

    "Random Forest":
        MODELS_DIR /
        "random_forest.joblib",

    "XGBoost":
        MODELS_DIR /
        "xgboost.joblib",

    "LightGBM":
        MODELS_DIR /
        "lightgbm.joblib"
}

CELLULE 85
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)

MODELS_DIR = (
    PROJECT_ROOT /
    "models" /
    "augmented"
)

METRICS_DIR = (
    PRO

In [15]:
# ============================================================
# ÉTAPE 12C — RETROUVER LA CRÉATION DES DONNÉES ET MODÈLES
# DANS LES NOTEBOOKS PRÉCÉDENTS
# ============================================================

import json
import re
from pathlib import Path

# Chercher récursivement tous les notebooks du projet
notebook_files = sorted(ROOT.rglob("*.ipynb"))

# Exclure Notebook 11 actuel
notebook_files = [
    p for p in notebook_files
    if "11_Adversarial_Training_Defense" not in p.name
    and "notebook11_adversarial_training" not in str(p).lower()
]

# Expressions réellement importantes
patterns = [
    r"\bX_train\s*=",
    r"\by_train\s*=",
    r"\bX_val\s*=",
    r"\by_val\s*=",
    r"\bX_test\s*=",
    r"\by_test\s*=",

    r"\bFEATURE_COLUMNS\s*=",
    r"\bmodels\s*=",

    r"train_test_split\(",
    r"pd\.read_csv\(",

    r"joblib\.load\(",
    r"pickle\.load",
    r"\.fit\(",

    r"XGBClassifier\(",
    r"RandomForestClassifier\(",
    r"LGBMClassifier\(",
    r"LogisticRegression\(",
    r"LinearSVC\(",
    r"SVC\(",
]

regex = re.compile(
    "|".join(patterns),
    re.IGNORECASE
)

print("=" * 90)
print("RECHERCHE CIBLÉE DANS LES NOTEBOOKS")
print("=" * 90)

found_any = False

for notebook_path in notebook_files:

    try:
        with open(
            notebook_path,
            "r",
            encoding="utf-8"
        ) as f:
            nb = json.load(f)

    except Exception:
        continue

    notebook_matches = []

    for cell_index, cell in enumerate(
        nb.get("cells", [])
    ):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(
            cell.get("source", [])
        )

        if regex.search(source):
            notebook_matches.append(
                (cell_index, source)
            )

    if notebook_matches:

        found_any = True

        print("\n" + "=" * 90)
        print("NOTEBOOK :", notebook_path.name)
        print("=" * 90)

        for cell_index, source in notebook_matches:

            print(
                f"\n----- CELLULE {cell_index} -----"
            )

            # afficher maximum 40 lignes par cellule
            lines = source.splitlines()

            for line in lines[:40]:
                print(line)

            if len(lines) > 40:
                print("... [cellule tronquée] ...")

if not found_any:
    print(
        "\n⚠️ Aucune définition importante trouvée."
    )

RECHERCHE CIBLÉE DANS LES NOTEBOOKS

NOTEBOOK : 01_data_audit.ipynb

----- CELLULE 21 -----
df = pd.read_csv(DATA_PATH)

NOTEBOOK : 02_preprocessing.ipynb

----- CELLULE 15 -----
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

----- CELLULE 18 -----
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

----- CELLULE 27 -----
df = pd.read_csv(DATA_PATH)

print("Dimensions du dataset brut :", df.shape)

NOTEBOOK : 03_baseline.ipynb

----- CELLULE 8 -----
xgb_curve_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_curve_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),


In [14]:
# ============================================================
# ÉTAPE 12B — RECHERCHE DES ARTEFACTS ML DU PROJET
# ============================================================

ARTIFACT_EXTENSIONS = {
    ".pkl",
    ".joblib",
    ".npy",
    ".npz",
    ".csv",
    ".parquet",
    ".json",
}

KEYWORDS = [
    "train",
    "val",
    "validation",
    "test",
    "model",
    "xgb",
    "xgboost",
    "forest",
    "random",
    "lightgbm",
    "lgbm",
    "svm",
    "logistic",
    "regression",
    "scaler",
    "feature",
    "augment",
]

candidate_files = []

for path in ROOT.rglob("*"):

    if not path.is_file():
        continue

    # Éviter les dépendances et notre nouveau notebook
    lowered_parts = [
        p.lower()
        for p in path.parts
    ]

    if ".venv" in lowered_parts:
        continue

    if "node_modules" in lowered_parts:
        continue

    if NB11_DIR in path.parents:
        continue

    if path.suffix.lower() not in ARTIFACT_EXTENSIONS:
        continue

    name_lower = path.name.lower()

    if any(
        keyword in name_lower
        for keyword in KEYWORDS
    ):
        candidate_files.append(path)


print("=" * 80)
print("ARTEFACTS ML POTENTIELS")
print("=" * 80)

for path in sorted(candidate_files):

    try:
        relative = path.relative_to(ROOT)
    except Exception:
        relative = path

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{str(relative):70s}"
        f" | {size_mb:8.2f} MB"
    )

print(
    "\nNombre de candidats :",
    len(candidate_files)
)

ARTEFACTS ML POTENTIELS
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_10100.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_10198.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_10358.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_11130.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_11883.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_1252.npy   |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_12641.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_13051.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_14569.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_15205.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversarial_samples\lightgbm_15225.npy  |     0.00 MB
hsj_experiment_b_v2\test_final\adversaria

In [13]:
# ============================================================
# ÉTAPE 12A — AUDIT DES VARIABLES ACTUELLES
# ============================================================

interesting_terms = [
    "x_train",
    "y_train",
    "x_val",
    "y_val",
    "x_test",
    "y_test",
    "model",
    "models",
    "feature",
    "scaler",
    "augment",
    "real",
]

found_variables = []

for name, obj in globals().copy().items():
    
    name_lower = name.lower()

    if any(term in name_lower for term in interesting_terms):
        
        try:
            if isinstance(obj, pd.DataFrame):
                description = (
                    f"DataFrame {obj.shape}"
                )

            elif isinstance(obj, pd.Series):
                description = (
                    f"Series {obj.shape}"
                )

            elif isinstance(obj, np.ndarray):
                description = (
                    f"ndarray {obj.shape}"
                )

            elif isinstance(obj, dict):
                description = (
                    f"dict ({len(obj)} clés)"
                )

            else:
                description = type(obj).__name__

            found_variables.append(
                (name, description)
            )

        except Exception:
            pass

print("=" * 75)
print("VARIABLES POTENTIELLEMENT UTILES")
print("=" * 75)

for name, description in sorted(found_variables):
    print(
        f"{name:35s} → {description}"
    )

print(
    "\nNombre trouvé :",
    len(found_variables)
)

VARIABLES POTENTIELLEMENT UTILES
HSJ_IMMUTABLE_FEATURES              → list
HSJ_MUTABLE_FEATURES                → list
MODELS_DIR                          → WindowsPath
model                               → str

Nombre trouvé : 4


In [12]:
# ============================================================
# ÉTAPE 11 — PLAN DE DÉFENSE PRÉ-ENREGISTRÉ
# ============================================================

DEFENCE_ATTACK_PLAN = {
    "XGBoost": ["HSJ"],
    "Random Forest": ["HSJ"],
    "LightGBM": ["HSJ"],
    "SVM linéaire": ["ZOO", "HSJ"],
    "Régression logistique": ["ZOO", "HSJ"],
}

DEFENCE_SELECTION_RULE = {
    "primary_objective": "reduce_validation_ASR",
    "max_clean_recall_drop": 0.05,
    "max_clean_f1_drop": 0.05,
    "test_used_for_selection": False,
}

DEFENCE_PLAN = {
    "attack_plan": DEFENCE_ATTACK_PLAN,
    "selection_rule": DEFENCE_SELECTION_RULE,
    "adversarial_label": 1,
    "training_source": "TRAIN_ONLY",
    "validation_role": "DEFENCE_SELECTION_ONLY",
    "test_role": "FINAL_EVALUATION_ONLY",
    "old_test_adversarial_examples_used_for_training": False,
}

DEFENCE_PLAN_FILE = (
    NB11_DIR
    / "defence_plan_pretest.json"
)

with open(
    DEFENCE_PLAN_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        DEFENCE_PLAN,
        f,
        indent=4,
        ensure_ascii=False
    )

print("✅ Plan de défense pré-enregistré.")
print(DEFENCE_PLAN_FILE.resolve())

print("\nPlan :")
for model, attacks in DEFENCE_ATTACK_PLAN.items():
    print(f"{model:25s} → {' + '.join(attacks)}")

✅ Plan de défense pré-enregistré.
C:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training\defence_plan_pretest.json

Plan :
XGBoost                   → HSJ
Random Forest             → HSJ
LightGBM                  → HSJ
SVM linéaire              → ZOO + HSJ
Régression logistique     → ZOO + HSJ


In [11]:
# ============================================================
# ÉTAPE 10 — SYNTHÈSE AVANT DÉFENSE
# ============================================================

baseline_view = (
    asr_before[
        [
            "Attack",
            "Model",
            "Attacks",
            "Successful_Attacks",
            "ASR"
        ]
    ]
    .copy()
)

baseline_view["ASR"] = baseline_view["ASR"].round(2)

display(
    baseline_view
    .sort_values(["Attack", "ASR"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nCette table constitue la référence AVANT défense.")

,Attack,Model,Attacks,Successful_Attacks,ASR
0,HSJ,XGBoost,53,19,35.85
1,HSJ,LightGBM,53,18,33.96
2,HSJ,Random Forest,53,16,30.19
3,HSJ,SVM linéaire,53,15,28.30
4,HSJ,Régression logistique,53,12,22.64
5,ZOO,SVM linéaire,53,43,81.13
6,ZOO,Régression logistique,53,33,62.26
7,ZOO,LightGBM,53,0,0.00
8,ZOO,Random Forest,53,0,0.00
9,ZOO,XGBoost,53,0,0.00



Cette table constitue la référence AVANT défense.


In [10]:
# ============================================================
# ÉTAPE 9 — CHARGEMENT OFFICIEL DE LA BASELINE NOTEBOOK 10
# ============================================================

import pandas as pd
import numpy as np

ASR_BEFORE_FILE = NB10_EXPORT_DIR / "asr_summary.csv"
METRICS_BEFORE_FILE = NB10_EXPORT_DIR / "clean_vs_attacks_metrics.csv"
PERTURB_BEFORE_FILE = NB10_EXPORT_DIR / "perturbation_summary.csv"
COST_BEFORE_FILE = NB10_EXPORT_DIR / "computational_cost.csv"
ROBUSTNESS_BEFORE_FILE = NB10_EXPORT_DIR / "robustness_summary.csv"
COMMON_TEST_FILE = NB10_EXPORT_DIR / "common_test_fraud_positions.npy"

# ------------------------------------------------------------
# Chargement
# ------------------------------------------------------------

asr_before = pd.read_csv(ASR_BEFORE_FILE)
metrics_before = pd.read_csv(METRICS_BEFORE_FILE)
perturb_before = pd.read_csv(PERTURB_BEFORE_FILE)
cost_before = pd.read_csv(COST_BEFORE_FILE)
robustness_before = pd.read_csv(ROBUSTNESS_BEFORE_FILE)

common_test_positions = np.load(
    COMMON_TEST_FILE,
    allow_pickle=False
).astype(int)

# ------------------------------------------------------------
# Contrôles
# ------------------------------------------------------------

print("=" * 75)
print("BASELINE NOTEBOOK 10")
print("=" * 75)

print("ASR :", asr_before.shape)
print("Métriques :", metrics_before.shape)
print("Perturbations :", perturb_before.shape)
print("Coût :", cost_before.shape)
print("Robustesse :", robustness_before.shape)

print(
    "\nFraudes Test communes :",
    len(common_test_positions)
)

print(
    "Positions uniques :",
    len(np.unique(common_test_positions))
)

assert len(common_test_positions) == 53
assert len(np.unique(common_test_positions)) == 53

assert set(asr_before["Attack"].unique()) == {"ZOO", "HSJ"}

print("\n✅ Baseline Notebook 10 chargée et verrouillée.")

BASELINE NOTEBOOK 10
ASR : (10, 7)
Métriques : (15, 11)
Perturbations : (10, 8)
Coût : (10, 8)
Robustesse : (5, 8)

Fraudes Test communes : 53
Positions uniques : 53

✅ Baseline Notebook 10 chargée et verrouillée.


In [9]:
# ============================================================
# ÉTAPE 8 — SAUVEGARDE DU PROTOCOLE
# ============================================================

PROTOCOL = {
    "random_state": RANDOM_STATE,

    "data_rule": {
        "train": "generation + adversarial training",
        "validation": "defence selection",
        "test": "final evaluation only",
        "test_used_for_training": False,
    },

    "hsj": {
        "local_eps": HSJ_LOCAL_EPS,
        "mutable_features": HSJ_MUTABLE_FEATURES,
        "immutable_features": HSJ_IMMUTABLE_FEATURES,
        "norm": "Linf",
        "max_iter": HSJ_CONFIG["max_iter"],
        "max_eval": HSJ_CONFIG["max_eval"],
        "init_eval": HSJ_CONFIG["init_eval"],
        "init_size": HSJ_CONFIG["init_size"],
    },

    "zoo": {
        "confidence": ZOO_CONFIG["confidence"],
        "learning_rate": ZOO_CONFIG["learning_rate"],
        "max_iter": ZOO_CONFIG["max_iter"],
        "binary_search_steps": ZOO_CONFIG["binary_search_steps"],
        "initial_const": ZOO_CONFIG["initial_const"],
    }
}

PROTOCOL_FILE = NB11_DIR / "protocol_initial.json"

with open(PROTOCOL_FILE, "w", encoding="utf-8") as f:
    json.dump(PROTOCOL, f, indent=4, ensure_ascii=False)

print("✅ Protocole enregistré :")
print(PROTOCOL_FILE.resolve())

✅ Protocole enregistré :
C:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training\protocol_initial.json


In [8]:
# ============================================================
# ÉTAPE 7 — CONFIGURATION HSJ GELÉE
# ============================================================

HSJ_LOCAL_EPS = 0.05

HSJ_CONFIG = {
    "targeted": True,
    "norm": np.inf,
    "max_iter": 50,
    "max_eval": 10000,
    "init_eval": 100,
    "init_size": 100,
    "batch_size": 64,
    "verbose": False,
}

HSJ_MUTABLE_FEATURES = [
    f"V{i}" for i in range(1, 29)
]

HSJ_IMMUTABLE_FEATURES = [
    "Time",
    "Amount"
]

print("HSJ_LOCAL_EPS =", HSJ_LOCAL_EPS)

print("\nVariables modifiables :")
print(HSJ_MUTABLE_FEATURES)

print("\nVariables immuables :")
print(HSJ_IMMUTABLE_FEATURES)

HSJ_LOCAL_EPS = 0.05

Variables modifiables :
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28']

Variables immuables :
['Time', 'Amount']


In [7]:
# ============================================================
# ÉTAPE 6 — CONFIGURATION ZOO GELÉE
# ============================================================

ZOO_CONFIG = {
    "confidence": 0.5,
    "targeted": True,
    "learning_rate": 1e-3,
    "max_iter": 200,
    "binary_search_steps": 10,
    "initial_const": 0.001,
    "abort_early": True,
    "batch_size": 1,
    "verbose": False,
}

print("Configuration ZOO :")
for key, value in ZOO_CONFIG.items():
    print(f"{key:25s} = {value}")

Configuration ZOO :
confidence                = 0.5
targeted                  = True
learning_rate             = 0.001
max_iter                  = 200
binary_search_steps       = 10
initial_const             = 0.001
abort_early               = True
batch_size                = 1
verbose                   = False


In [6]:
# ============================================================
# ÉTAPE 5 — BASELINE DE ROBUSTESSE NOTEBOOK 10
# ============================================================

NB10_ROBUSTNESS_BASELINE = pd.DataFrame([
    
    ["XGBoost",                0.00, 35.85],
    ["Random Forest",          0.00, 30.19],
    ["LightGBM",               0.00, 33.96],
    ["SVM linéaire",          81.13, 28.30],
    ["Régression logistique", 62.26, 22.64],

], columns=[
    "Model",
    "ZOO_ASR_Before",
    "HSJ_Local_ASR_Before"
])

display(NB10_ROBUSTNESS_BASELINE)

,Model,ZOO_ASR_Before,HSJ_Local_ASR_Before
0,XGBoost,0.00,35.85
1,Random Forest,0.00,30.19
2,LightGBM,0.00,33.96
3,SVM linéaire,81.13,28.30
4,Régression logistique,62.26,22.64


In [5]:
# ============================================================
# ÉTAPE 4B — RÉSUMÉ COMPACT DES EXPORTS NOTEBOOK 10
# ============================================================

csv_files = sorted(NB10_EXPORT_DIR.rglob("*.csv"))

for path in csv_files:
    df_tmp = pd.read_csv(path)

    print("\n" + "=" * 70)
    print("Fichier :", path.name)
    print("Shape   :", df_tmp.shape)
    print("Colonnes:", df_tmp.columns.tolist())


Fichier : asr_summary.csv
Shape   : (10, 7)
Colonnes: ['Attack', 'Model', 'Attacks', 'Successful_Attacks', 'ASR', 'CI95_Low', 'CI95_High']

Fichier : clean_vs_attacks_metrics.csv
Shape   : (15, 11)
Colonnes: ['Model', 'Condition', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'TN', 'FP', 'FN', 'TP']

Fichier : computational_cost.csv
Shape   : (10, 8)
Colonnes: ['Attack', 'Model', 'Mean_Queries', 'Median_Queries', 'Mean_Time_s', 'Median_Time_s', 'Total_Time_s', 'Total_Time_min']

Fichier : perturbation_summary.csv
Shape   : (10, 8)
Colonnes: ['Attack', 'Model', 'Mean_L2_All', 'Mean_Linf_All', 'Max_Linf_All', 'Mean_L2_Success', 'Mean_Linf_Success', 'Max_Linf_Success']

Fichier : robustness_summary.csv
Shape   : (5, 8)
Colonnes: ['Model', 'ASR_HSJ', 'ASR_ZOO', 'Recall_Clean', 'Recall_HSJ', 'Recall_ZOO', 'Worst_Case_ASR', 'Worst_Case_Recall']


In [4]:
# ============================================================
# ÉTAPE 4 — INSPECTION DES TABLEAUX EXPORTÉS
# ============================================================

csv_files = sorted(NB10_EXPORT_DIR.rglob("*.csv"))

print(f"Nombre de CSV trouvés : {len(csv_files)}")

for path in csv_files:
    
    print("\n" + "=" * 75)
    print(path.name)
    print("=" * 75)

    try:
        df_tmp = pd.read_csv(path)

        print("Shape :", df_tmp.shape)
        print("Colonnes :")
        print(df_tmp.columns.tolist())

        print("\nAperçu :")
        display(df_tmp.head(3))

    except Exception as e:
        print("⚠️ Impossible de lire :", e)

Nombre de CSV trouvés : 5

asr_summary.csv
Shape : (10, 7)
Colonnes :
['Attack', 'Model', 'Attacks', 'Successful_Attacks', 'ASR', 'CI95_Low', 'CI95_High']

Aperçu :


,Attack,Model,Attacks,Successful_Attacks,ASR,CI95_Low,CI95_High
0,HSJ,LightGBM,53,18,33.962264,22.686863,47.405464
1,HSJ,Random Forest,53,16,30.188679,19.518140,43.537087
2,HSJ,Régression logistique,53,12,22.641509,13.454500,35.526528



clean_vs_attacks_metrics.csv
Shape : (15, 11)
Colonnes :
['Model', 'Condition', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'TN', 'FP', 'FN', 'TP']

Aperçu :


,Model,Condition,Accuracy,Precision,Recall,F1,ROC_AUC,TN,FP,FN,TP
0,XGBoost,Clean,0.999342,0.820896,0.774648,0.797101,0.974825,42476,12,16,55
1,XGBoost,ZOO,0.999342,0.820896,0.774648,0.797101,0.974825,42476,12,16,55
2,XGBoost,HSJ,0.998896,0.750000,0.507042,0.605042,0.974666,42476,12,35,36



computational_cost.csv
Shape : (10, 8)
Colonnes :
['Attack', 'Model', 'Mean_Queries', 'Median_Queries', 'Mean_Time_s', 'Median_Time_s', 'Total_Time_s', 'Total_Time_min']

Aperçu :


,Attack,Model,Mean_Queries,Median_Queries,Mean_Time_s,Median_Time_s,Total_Time_s,Total_Time_min
0,HSJ,LightGBM,24848.754717,24845.0,1.241188,1.205132,65.782946,1.096382
1,HSJ,Random Forest,24841.226415,24836.0,31.106388,30.912789,1648.638539,27.477309
2,HSJ,Régression logistique,24792.000000,24792.0,0.813765,0.790898,43.129558,0.718826



perturbation_summary.csv
Shape : (10, 8)
Colonnes :
['Attack', 'Model', 'Mean_L2_All', 'Mean_Linf_All', 'Max_Linf_All', 'Mean_L2_Success', 'Mean_Linf_Success', 'Max_Linf_Success']

Aperçu :


,Attack,Model,Mean_L2_All,Mean_Linf_All,Max_Linf_All,Mean_L2_Success,Mean_Linf_Success,Max_Linf_Success
0,HSJ,LightGBM,0.266247,0.081435,0.372506,0.109596,0.032009,0.048352
1,HSJ,Random Forest,0.317787,0.095180,0.309907,0.116566,0.032987,0.048869
2,HSJ,Régression logistique,0.520044,0.104693,0.277470,0.175621,0.035226,0.049286



robustness_summary.csv
Shape : (5, 8)
Colonnes :
['Model', 'ASR_HSJ', 'ASR_ZOO', 'Recall_Clean', 'Recall_HSJ', 'Recall_ZOO', 'Worst_Case_ASR', 'Worst_Case_Recall']

Aperçu :


,Model,ASR_HSJ,ASR_ZOO,Recall_Clean,Recall_HSJ,Recall_ZOO,Worst_Case_ASR,Worst_Case_Recall
0,Random Forest,30.188679,0.0,0.774648,0.549296,0.774648,30.188679,0.549296
1,LightGBM,33.962264,0.0,0.760563,0.507042,0.760563,33.962264,0.507042
2,XGBoost,35.849057,0.0,0.774648,0.507042,0.774648,35.849057,0.507042


In [3]:
# ============================================================
# ÉTAPE 3 — AUDIT DES EXPORTS NOTEBOOK 10
# ============================================================

NB10_EXPORT_DIR = ROOT / "notebook10_final" / "exports_for_notebook11"

if not NB10_EXPORT_DIR.exists():
    print("⚠️ Répertoire attendu non trouvé directement.")
    
    candidates = list(ROOT.rglob("exports_for_notebook11"))
    
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Impossible de trouver exports_for_notebook11."
        )
    
    print("\nCandidats trouvés :")
    for candidate in candidates:
        print(" -", candidate)
    
    NB10_EXPORT_DIR = candidates[0]

print("\n✅ Exports Notebook 10 trouvés :")
print(NB10_EXPORT_DIR.resolve())

print("\nContenu :")
print("-" * 70)

files_nb10 = [
    p for p in NB10_EXPORT_DIR.rglob("*")
    if p.is_file()
]

for p in sorted(files_nb10):
    print(
        f"{p.relative_to(NB10_EXPORT_DIR)}"
        f"  |  {p.stat().st_size / 1024:.1f} KB"
    )

print("-" * 70)
print("Nombre de fichiers :", len(files_nb10))


✅ Exports Notebook 10 trouvés :
C:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook10_final\exports_for_notebook11

Contenu :
----------------------------------------------------------------------
asr_summary.csv  |  0.8 KB
clean_vs_attacks_metrics.csv  |  1.9 KB
common_test_fraud_positions.npy  |  0.5 KB
computational_cost.csv  |  1.2 KB
manifest.json  |  1.0 KB
perturbation_summary.csv  |  1.3 KB
robustness_summary.csv  |  0.8 KB
----------------------------------------------------------------------
Nombre de fichiers : 7


In [2]:
# ============================================================
# ÉTAPE 2 — STRUCTURE DU NOTEBOOK 11
# ============================================================

ROOT = Path.cwd()

NB11_DIR = ROOT / "notebook11_adversarial_training"
BANK_DIR = NB11_DIR / "adversarial_bank"
MODELS_DIR = NB11_DIR / "defended_models"
VALIDATION_DIR = NB11_DIR / "validation"
TEST_DIR = NB11_DIR / "test_final"
EXPORT_DIR = NB11_DIR / "exports_for_notebook12"

for directory in [
    NB11_DIR,
    BANK_DIR,
    MODELS_DIR,
    VALIDATION_DIR,
    TEST_DIR,
    EXPORT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Répertoire courant :", ROOT)
print("Notebook 11 :", NB11_DIR.resolve())

print("\n✅ Structure Notebook 11 créée.")

Répertoire courant : c:\Users\MSI\Desktop\fraud_detection_memoire\notebooks
Notebook 11 : C:\Users\MSI\Desktop\fraud_detection_memoire\notebooks\notebook11_adversarial_training

✅ Structure Notebook 11 créée.


In [1]:
# ============================================================
# ÉTAPE 1 — IMPORTS ET REPRODUCTIBILITÉ
# ============================================================

from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from art.attacks.evasion import ZooAttack, HopSkipJump
from art.estimators.classification import BlackBoxClassifier

# ------------------------------------------------------------
# Reproductibilité
# ------------------------------------------------------------

RANDOM_STATE = 2026

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("✅ Imports terminés")
print("✅ RANDOM_STATE =", RANDOM_STATE)

c:\Users\MSI\Desktop\fraud_detection_memoire\.venv\Lib\site-packages\art\estimators\certification\__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


✅ Imports terminés
✅ RANDOM_STATE = 2026


# Notebook 11 — Défense par entraînement adversarial

## Objectif scientifique

Le Notebook 10 a montré que les cinq classificateurs étudiés présentent
des profils de vulnérabilité différents face aux attaques ZOO et
HopSkipJump (HSJ).

L'objectif du présent notebook est d'étudier si l'entraînement
contradictoire (Adversarial Training) permet de réduire ces
vulnérabilités sans dégrader excessivement les performances de
détection sur les transactions propres.

La défense consiste à :

1. générer des transactions frauduleuses adversariales à partir
   des données d'apprentissage uniquement ;
2. vérifier leur validité selon le protocole expérimental ;
3. conserver leur véritable étiquette frauduleuse (Class = 1) ;
4. les intégrer au jeu d'apprentissage ;
5. réentraîner les classificateurs ;
6. vérifier les performances propres ;
7. générer de nouvelles attaques contre les modèles défendus ;
8. comparer la robustesse avant et après défense.

## Règle fondamentale

Aucun exemple provenant de l'ensemble Test, y compris les exemples
adversariaux générés dans le Notebook 10, ne sera utilisé pour
l'entraînement.

Train  → apprentissage et génération de la défense  
Validation → choix de la configuration de défense  
Test → évaluation finale uniquement